In [1]:
get_ipython().run_cell_magic('bash', '', '\ncd /content\n\nrm -rf kc-slice\nmkdir -p kc-slice\n\ncd kc-slice\n\nmkdir -p \\\n    data/raw \\\n    data/smoke \\\n    data/processed \\\n    methods/mondrian \\\n    methods/arx \\\n    methods/kc_slice \\\n    hierarchies \\\n    configs \\\n    scripts \\\n    results \\\n    notebooks\n\ntouch README.md\ntouch requirements.txt\n\necho "KC-SLICE PROJECT"\necho "================"\nfind . -maxdepth 2 -type d | sort\n')

In [2]:
#Cell 2
get_ipython().run_line_magic('pip', 'install -q pandas numpy scikit-learn pyyaml')

In [3]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/methods/mondrian\n\nrm -rf upstream\n\ngit clone --depth 1 \\\n    https://github.com/qiyuangong/Mondrian.git \\\n    upstream\n\necho\necho "Mondrian repository:"\nls -la upstream\n')

In [4]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/methods/mondrian/upstream\n\necho "Running Mondrian core tests..."\necho\n\npython -m unittest -v \\\n    mondrian_test.functionTest.test1_mondrian_strict \\\n    mondrian_test.functionTest.test1_mondrian_relax \\\n    mondrian_test.functionTest.test2_mondrian_strict \\\n    mondrian_test.functionTest.test2_mondrian_relax \\\n    mondrian_test.functionTest.test_mondrian_datetime\n')

In [5]:
# #Cell 5
# %%bash

# set -e

# cd /content/kc-slice/methods/mondrian/upstream

# python -m unittest discover -v

In [6]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\napt-get update -qq\napt-get install -y -qq openjdk-21-jdk-headless\n\necho\necho "Java:"\njava -version\n\necho\necho "Javac:"\njavac -version\n')

In [7]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/methods/arx\n\nrm -f libarx-3.9.2.jar\n\nwget -q \\\n    https://github.com/arx-deidentifier/arx/releases/download/v3.9.2/libarx-3.9.2.jar \\\n    -O libarx-3.9.2.jar\n\necho "Downloaded ARX:"\nls -lh libarx-3.9.2.jar\n')

In [8]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/methods/arx\n\necho "Checking ARX JAR..."\n\njar tf libarx-3.9.2.jar | grep \\\n    \'org/deidentifier/arx/ARXAnonymizer.class\'\n\necho\necho "ARX class found."\n')

In [9]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/methods/arx\n\ncat > SmokeTest.java <<\'EOF\'\nimport org.deidentifier.arx.ARXAnonymizer;\n\npublic class SmokeTest {\n    public static void main(String[] args) {\n        System.out.println("ARX library loaded successfully.");\n        System.out.println("Loaded class: " + ARXAnonymizer.class.getName());\n        Package pkg = ARXAnonymizer.class.getPackage();\n        System.out.println(\n            "Implementation version: " + pkg.getImplementationVersion()\n        );\n    }\n}\nEOF\n\njavac -cp libarx-3.9.2.jar SmokeTest.java\n\njava -cp "libarx-3.9.2.jar:." SmokeTest\n')

In [10]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/data/raw\n\nrm -f adult.csv\n\nwget -q \\\n    https://huggingface.co/datasets/scikit-learn/adult-census-income/resolve/main/adult.csv \\\n    -O adult.csv\n\necho "Adult dataset:"\nls -lh adult.csv\n')

In [11]:
#Cell 11
import pandas as pd

adult_path = "/content/kc-slice/data/raw/adult.csv"

adult = pd.read_csv(adult_path)

print("Adult shape:", adult.shape)
print("Rows:", len(adult))
print("Columns:", len(adult.columns))
print()
print("Column names:")
print(adult.columns.tolist())

assert len(adult) == 32561, (
    f"Expected 32561 rows, got {len(adult)}"
)

assert len(adult.columns) == 15, (
    f"Expected 15 columns, got {len(adult.columns)}"
)

print()
print("Adult verification: PASSED")

In [12]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice/data/raw\n\nrm -f diabetes.zip\n\nwget -q \\\n    "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip" \\\n    -O diabetes.zip\n\necho "Downloaded:"\nls -lh diabetes.zip\n\necho\necho "Extracting..."\n\nrm -f diabetic_data.csv IDS_mapping.csv\n\nunzip -o diabetes.zip\n\necho\necho "Files:"\nls -lh\n')

In [13]:
#Cell 13
import pandas as pd

diabetes_path = "/content/kc-slice/data/raw/diabetic_data.csv"

diabetes = pd.read_csv(diabetes_path)

print("Diabetes shape:", diabetes.shape)
print("Rows:", len(diabetes))
print("Columns:", len(diabetes.columns))

assert len(diabetes) == 101766, (
    f"Expected 101766 rows, got {len(diabetes)}"
)

print()
print("Checking '?' sentinel values...")
print()

sentinel_columns = [
    "race",
    "weight",
    "payer_code",
    "medical_specialty",
]

for col in sentinel_columns:
    count = (diabetes[col].astype(str) == "?").sum()
    print(f"{col:20s}: {count}")

    assert count > 0, (
        f"No '?' sentinel found in {col}"
    )

print()
print("Diabetes verification: PASSED")

In [14]:
#Cell 14
print("DIABETES COLUMNS")
print("=" * 60)

for i, col in enumerate(diabetes.columns, start=1):
    print(f"{i:2d}. {col}")

In [15]:
#Cell 15
from pathlib import Path

ROOT = Path("/content/kc-slice")

adult_smoke = adult.head(100).copy()
diabetes_smoke = diabetes.head(100).copy()

adult_smoke.to_csv(
    ROOT / "data/smoke/adult_small.csv",
    index=False
)

diabetes_smoke.to_csv(
    ROOT / "data/smoke/diabetes_small.csv",
    index=False
)

print("Adult smoke dataset:", adult_smoke.shape)
print("Diabetes smoke dataset:", diabetes_smoke.shape)

In [16]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice\n\ncat > configs/datasets.yaml <<\'EOF\'\nadult:\n  path: data/raw/adult.csv\n  smoke_path: data/smoke/adult_small.csv\n  expected_rows: 32561\n\ndiabetes:\n  path: data/raw/diabetic_data.csv\n  smoke_path: data/smoke/diabetes_small.csv\n  expected_rows: 101766\nEOF\n\ncat > configs/experiment.yaml <<\'EOF\'\nk: 10\n\nseeds:\n  - 1\n  - 2\n  - 3\n\nsensitive_attribute_counts:\n  - 1\n  - 2\n  - 3\n\nmethods:\n  - mondrian\n  - arx\n  - kc_slice\nEOF\n\necho "Created configuration files:"\necho\ncat configs/datasets.yaml\necho\ncat configs/experiment.yaml\n')

In [17]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\ncd /content/kc-slice\n\ntouch methods/kc_slice/__init__.py\ntouch methods/kc_slice/algorithm.py\ntouch methods/kc_slice/config.py\ntouch methods/kc_slice/io.py\ntouch methods/kc_slice/evaluate.py\ntouch methods/kc_slice/smoke_test.py\n\ncat > methods/kc_slice/README.md <<\'EOF\'\n# KC-Slice Reimplementation\n\nThis directory contains the project reimplementation of the KC-Slice\nsensitivity-induced slicing algorithm.\n\nSource paper:\nKC-Slice: A dynamic privacy-preserving data publishing technique\nfor multisensitive attributes.\n\nImplementation rules:\n1. Follow the origin paper.\n2. Record every parameter taken from the paper.\n3. Do not silently invent unspecified parameters.\n4. Record every experimental assumption.\n5. Keep this implementation separate from Mondrian and ARX.\nEOF\n\necho "KC-Slice skeleton created."\nfind methods/kc_slice -maxdepth 1 -type f | sort\n')

In [18]:
#Cell 18
from pathlib import Path
import sys
import pandas as pd
import yaml

ROOT = Path("/content/kc-slice")

print("=" * 70)
print("KC-SLICE MILESTONE 2 — SETUP VERIFICATION")
print("=" * 70)

# --------------------------------------------------
# 1. Adult
# --------------------------------------------------

adult_path = ROOT / "data/raw/adult.csv"
adult = pd.read_csv(adult_path)

assert len(adult) == 32561
assert len(adult.columns) == 15

print("✅ Adult")
print(f"   Rows    : {len(adult)}")
print(f"   Columns : {len(adult.columns)}")

# --------------------------------------------------
# 2. Diabetes
# --------------------------------------------------

diabetes_path = ROOT / "data/raw/diabetic_data.csv"
diabetes = pd.read_csv(diabetes_path)

assert len(diabetes) == 101766

print()
print("✅ Diabetes")
print(f"   Rows    : {len(diabetes)}")
print(f"   Columns : {len(diabetes.columns)}")

for col in ["race", "weight", "payer_code", "medical_specialty"]:
    assert (diabetes[col].astype(str) == "?").any()

print("   '?' sentinel verified in:")
print("      race")
print("      weight")
print("      payer_code")
print("      medical_specialty")

# --------------------------------------------------
# 3. Smoke datasets
# --------------------------------------------------

adult_small = pd.read_csv(
    ROOT / "data/smoke/adult_small.csv"
)

diabetes_small = pd.read_csv(
    ROOT / "data/smoke/diabetes_small.csv"
)

assert len(adult_small) == 100
assert len(diabetes_small) == 100

print()
print("✅ Smoke datasets")
print(f"   Adult     : {adult_small.shape}")
print(f"   Diabetes  : {diabetes_small.shape}")

# --------------------------------------------------
# 4. Mondrian
# --------------------------------------------------

mondrian_path = str(
    ROOT / "methods/mondrian/upstream"
)

if mondrian_path not in sys.path:
    sys.path.insert(0, mondrian_path)

import mondrian

print()
print("✅ Mondrian")
print(f"   Module: {mondrian.__file__}")

# --------------------------------------------------
# 5. Config files
# --------------------------------------------------

with open(ROOT / "configs/datasets.yaml") as f:
    datasets_cfg = yaml.safe_load(f)

with open(ROOT / "configs/experiment.yaml") as f:
    experiment_cfg = yaml.safe_load(f)

assert datasets_cfg["adult"]["expected_rows"] == 32561
assert datasets_cfg["diabetes"]["expected_rows"] == 101766
assert experiment_cfg["k"] == 10
assert experiment_cfg["sensitive_attribute_counts"] == [1, 2, 3]
assert experiment_cfg["seeds"] == [1, 2, 3]

print()
print("✅ Configuration files")

# --------------------------------------------------
# Final
# --------------------------------------------------

print()
print("=" * 70)
print("🎉 MILESTONE 2 BASE SETUP VERIFIED")
print("=" * 70)

In [19]:
get_ipython().run_cell_magic('bash', '', '\nset -e\n\necho "======================================"\necho "ARX VERIFICATION"\necho "======================================"\n\necho\necho "Java version:"\njava -version\n\necho\necho "Checking ARX JAR:"\nls -lh /content/kc-slice/methods/arx/libarx-3.9.2.jar\n\necho\necho "Checking ARXAnonymizer class:"\njar tf /content/kc-slice/methods/arx/libarx-3.9.2.jar | \\\n    grep \'org/deidentifier/arx/ARXAnonymizer.class\'\n\necho\necho "Compiling ARX smoke test..."\n\ncd /content/kc-slice/methods/arx\n\ncat > SmokeTest.java <<\'EOF\'\nimport org.deidentifier.arx.ARXAnonymizer;\n\npublic class SmokeTest {\n    public static void main(String[] args) {\n        System.out.println("ARX library loaded successfully.");\n        System.out.println("Loaded class: " + ARXAnonymizer.class.getName());\n    }\n}\nEOF\n\njavac -cp libarx-3.9.2.jar SmokeTest.java\n\necho\necho "Running ARX smoke test:"\njava -cp "libarx-3.9.2.jar:." SmokeTest\n')

In [20]:
#cell 20
from pathlib import Path
from datetime import datetime

root = Path("/content/kc-slice")

# Make sure the results directory exists
(root / "results").mkdir(parents=True, exist_ok=True)

status = f"""
KC-SLICE MILESTONE 2 — SETUP COMPLETE
======================================

Verified at: {datetime.now().isoformat()}

Mondrian:
  Repository: qiyuangong/Mondrian
  Core tests: PASS

Java:
  Version: 21.0.12.1
  Status: PASS

ARX:
  Version: 3.9.2
  JAR: methods/arx/libarx-3.9.2.jar
  ARXAnonymizer class loading: PASS

Adult:
  Rows: 32561
  Columns: 15
  Status: PASS

Diabetes:
  Rows: 101766
  Raw CSV columns: 50
  '?' sentinel verified in:
    - race
    - weight
    - payer_code
    - medical_specialty
  Status: PASS

Smoke datasets:
  Adult: 100 rows
  Diabetes: 100 rows

KC-Slice:
  Project skeleton: CREATED

======================================
MILESTONE 2 COMPLETE
"""

print(status)

output_path = root / "results" / "milestone2_setup_verified.txt"
output_path.write_text(status.strip() + "\n", encoding="utf-8")

print("\nSaved successfully:")
print(output_path)

In [21]:
#cell 21
from pathlib import Path
import pandas as pd

# ============================================================
# MILESTONE 3 — ADULT BASELINE PREPARATION
# ============================================================

ROOT = Path("/content/kc-slice")

K = 10

QID_COLUMNS = [
    "age",
    "sex",
    "race"
]

SENSITIVE_COLUMN = "income"

print("=" * 70)
print("MILESTONE 3 — ADULT BASELINE")
print("=" * 70)

print(f"k                   : {K}")
print(f"QIDs                : {QID_COLUMNS}")
print(f"Sensitive attribute : {SENSITIVE_COLUMN}")

# ------------------------------------------------------------
# Load the Adult dataset already downloaded in Milestone 2
# ------------------------------------------------------------

adult_raw = pd.read_csv(
    ROOT / "data/raw/adult.csv"
)

print()
print("Raw Adult dataset")
print(f"Rows    : {len(adult_raw)}")
print(f"Columns : {len(adult_raw.columns)}")

# ------------------------------------------------------------
# Match the published Adult preprocessing:
# remove every record containing '?'
# ------------------------------------------------------------

adult_clean = adult_raw[
    ~(adult_raw.astype(str) == "?").any(axis=1)
].copy()

print()
print("After removing records containing '?'")
print(f"Rows    : {len(adult_clean)}")
print(f"Columns : {len(adult_clean.columns)}")

# Published comparison benchmark uses 30,162 complete records.
assert len(adult_clean) == 30162, (
    f"Expected 30162 records, got {len(adult_clean)}"
)

# ------------------------------------------------------------
# Keep exactly the baseline variables
# ------------------------------------------------------------

baseline_columns = QID_COLUMNS + [SENSITIVE_COLUMN]

adult_baseline = adult_clean[baseline_columns].copy()

# ------------------------------------------------------------
# Save baseline input
# ------------------------------------------------------------

output_dir = ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)

baseline_path = output_dir / "adult_baseline_k10_1sa.csv"

adult_baseline.to_csv(
    baseline_path,
    index=False
)

print()
print("Baseline dataset")
print(f"Shape   : {adult_baseline.shape}")
print(f"Saved   : {baseline_path}")

print()
print("First 5 rows:")
print(adult_baseline.head())

print()
print("=" * 70)
print("ADULT BASELINE PREPARATION PASSED")
print("=" * 70)

In [22]:
#cell 22
from pathlib import Path
from collections import Counter
import copy
import sys
import pandas as pd

# ============================================================
# MILESTONE 3 — MONDRIAN BASELINE
# ============================================================

ROOT = Path("/content/kc-slice")
MONDRIAN_PATH = ROOT / "methods/mondrian/upstream"

if str(MONDRIAN_PATH) not in sys.path:
    sys.path.insert(0, str(MONDRIAN_PATH))

from mondrian import mondrian

print("=" * 70)
print("MONDRIAN BASELINE — k=10, 1 SENSITIVE ATTRIBUTE")
print("=" * 70)

# ------------------------------------------------------------
# Load the fixed baseline created in the previous cell
# ------------------------------------------------------------

df = pd.read_csv(
    ROOT / "data/processed/adult_baseline_k10_1sa.csv"
)

print(f"Input rows : {len(df)}")
print(f"QIDs       : age, sex, race")
print(f"Sensitive  : income")
print(f"k          : 10")

# ------------------------------------------------------------
# Encode categorical QIDs.
#
# Use first-appearance order, matching the ordering strategy
# used by qiyuangong/Mondrian's Adult reader.
# ------------------------------------------------------------

sex_order = list(dict.fromkeys(df["sex"].tolist()))
race_order = list(dict.fromkeys(df["race"].tolist()))

sex_map = {
    value: index
    for index, value in enumerate(sex_order)
}

race_map = {
    value: index
    for index, value in enumerate(race_order)
}

print()
print("Categorical ordering:")
print("Sex :", sex_order)
print("Race:", race_order)

# ------------------------------------------------------------
# Convert rows to the format expected by Mondrian:
#
# [age, encoded_sex, encoded_race, income]
# ------------------------------------------------------------

data = []

for _, row in df.iterrows():
    data.append([
        int(row["age"]),
        sex_map[row["sex"]],
        race_map[row["race"]],
        row["income"]
    ])

# ------------------------------------------------------------
# Run STRICT Mondrian
# ------------------------------------------------------------

print()
print("Running strict Mondrian...")

result, eval_result = mondrian(
    copy.deepcopy(data),
    10,
    False
)

ncp = eval_result[0]
runtime = eval_result[1]

# ------------------------------------------------------------
# Calculate equivalence classes and Discernibility Metric
#
# Each unique anonymized QID tuple defines an equivalence class.
# DM = sum(|E|^2)
# ------------------------------------------------------------

equivalence_classes = Counter(
    (
        record[0],
        record[1],
        record[2]
    )
    for record in result
)

class_sizes = list(equivalence_classes.values())

discernibility = sum(
    size ** 2
    for size in class_sizes
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print()
print("=" * 70)
print("MONDRIAN RESULTS")
print("=" * 70)

print(f"NCP (%)                 : {ncp:.4f}")
print(f"Runtime (seconds)       : {runtime:.4f}")
print(f"Equivalence classes     : {len(class_sizes)}")
print(f"Minimum class size      : {min(class_sizes)}")
print(f"Maximum class size      : {max(class_sizes)}")
print(f"Mean class size         : {sum(class_sizes)/len(class_sizes):.2f}")
print(f"Discernibility Metric   : {discernibility:,}")

assert min(class_sizes) >= 10, (
    f"k-anonymity violation: smallest class = {min(class_sizes)}"
)

print()
print("✅ k=10 anonymity check PASSED")

# ------------------------------------------------------------
# Save baseline result summary
# ------------------------------------------------------------

results_dir = ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

summary_path = results_dir / "milestone3_mondrian_baseline.txt"

summary = f"""MILESTONE 3 — MONDRIAN BASELINE

Dataset: Adult Census Income
Input records: {len(df)}
QIDs: age, sex, race
Sensitive attribute: income
k: 10
Algorithm: qiyuangong/Mondrian
Model: strict

NCP (%): {ncp:.4f}
Runtime (seconds): {runtime:.4f}
Equivalence classes: {len(class_sizes)}
Minimum class size: {min(class_sizes)}
Maximum class size: {max(class_sizes)}
Mean class size: {sum(class_sizes)/len(class_sizes):.2f}
Discernibility Metric: {discernibility}

k-anonymity verification: PASS
"""

summary_path.write_text(summary, encoding="utf-8")

print(f"Saved summary: {summary_path}")

print()
print("=" * 70)
print("MONDRIAN BASELINE COMPLETE")
print("=" * 70)

In [23]:
#cell 23
# ============================================================
# MILESTONE 3 — MONDRIAN BASELINE DIAGNOSTIC
# ============================================================

from collections import Counter
import pandas as pd

print("=" * 70)
print("MONDRIAN BASELINE — PARTITION DIAGNOSTIC")
print("=" * 70)

# Reuse `result`, `equivalence_classes`, and `class_sizes`
# from the previous Mondrian cell.

print(f"Total records          : {len(result):,}")
print(f"Equivalence classes    : {len(class_sizes):,}")
print(f"Minimum class size     : {min(class_sizes):,}")
print(f"Maximum class size     : {max(class_sizes):,}")
print(f"Mean class size        : {sum(class_sizes) / len(class_sizes):.2f}")

# Class-size distribution
size_counts = Counter(class_sizes)

print()
print("Most common equivalence-class sizes:")
for size, count in sorted(
    size_counts.items(),
    key=lambda x: (-x[1], x[0])
)[:15]:
    print(f"  size {size:5d} : {count:4d} classes")

print()
print("Largest equivalence classes:")
for size in sorted(class_sizes, reverse=True)[:10]:
    print(f"  {size:,} records")

# Verify total number of records accounted for
assert sum(class_sizes) == len(result)

# Verify k-anonymity
assert min(class_sizes) >= 10

print()
print("Record accounting      : PASS")
print("k=10 anonymity         : PASS")

print("=" * 70)

In [24]:
#cell 24
# ============================================================
# MILESTONE 3 — STANDARD ADULT/MONDRIAN BASELINE CONFIG
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

K = 10

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

SENSITIVE_COLUMN = "income"

print("=" * 70)
print("MILESTONE 3 — STANDARD ADULT BASELINE")
print("=" * 70)

print(f"k                   : {K}")
print("QIDs                :")
for i, col in enumerate(QID_COLUMNS, 1):
    print(f"  {i}. {col}")

print(f"Sensitive attribute : {SENSITIVE_COLUMN}")

# Load Adult
adult_raw = pd.read_csv(
    ROOT / "data/raw/adult.csv"
)

print()
print("Raw Adult dataset")
print(f"Rows    : {len(adult_raw)}")
print(f"Columns : {len(adult_raw.columns)}")

# Remove rows containing '?'
adult_clean = adult_raw[
    ~(adult_raw.astype(str) == "?").any(axis=1)
].copy()

print()
print("After removing records containing '?'")
print(f"Rows    : {len(adult_clean)}")
print(f"Columns : {len(adult_clean.columns)}")

assert len(adult_clean) == 30162

# Select standard Mondrian configuration
baseline_columns = QID_COLUMNS + [SENSITIVE_COLUMN]

adult_baseline = adult_clean[baseline_columns].copy()

output_path = (
    ROOT / "data/processed/adult_standard_baseline_k10_1sa.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

adult_baseline.to_csv(
    output_path,
    index=False
)

print()
print("Baseline dataset")
print(f"Shape : {adult_baseline.shape}")
print(f"Saved : {output_path}")

print()
print(adult_baseline.head())

print()
print("=" * 70)
print("STANDARD ADULT BASELINE PREPARATION PASSED")
print("=" * 70)

In [25]:
# Cell 25
# ============================================================
# MILESTONE 3 — STANDARD ADULT MONDRIAN BASELINE
# 8 QIDs, 1 sensitive attribute, k=10
# ============================================================

from pathlib import Path
from collections import Counter
import copy
import sys
import pandas as pd

ROOT = Path("/content/kc-slice")
MONDRIAN_PATH = ROOT / "methods/mondrian/upstream"

if str(MONDRIAN_PATH) not in sys.path:
    sys.path.insert(0, str(MONDRIAN_PATH))

from mondrian import mondrian

K = 10

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

SENSITIVE_COLUMN = "income"

# ------------------------------------------------------------
# Load standard baseline
# ------------------------------------------------------------

input_path = (
    ROOT / "data/processed/adult_standard_baseline_k10_1sa.csv"
)

df = pd.read_csv(input_path)

assert len(df) == 30162
assert all(col in df.columns for col in QID_COLUMNS)
assert SENSITIVE_COLUMN in df.columns

print("=" * 70)
print("STANDARD ADULT — MONDRIAN BASELINE")
print("=" * 70)
print(f"Input rows : {len(df):,}")
print(f"QID count  : {len(QID_COLUMNS)}")
print(f"QIDs       : {QID_COLUMNS}")
print(f"Sensitive  : {SENSITIVE_COLUMN}")
print(f"k          : {K}")

# ------------------------------------------------------------
# Encode categorical QIDs using first-appearance ordering
# ------------------------------------------------------------

NUMERIC_QIDS = ["age", "education.num"]
CATEGORICAL_QIDS = [
    col for col in QID_COLUMNS
    if col not in NUMERIC_QIDS
]

category_orders = {}
category_maps = {}

for col in CATEGORICAL_QIDS:
    order = list(dict.fromkeys(df[col].tolist()))
    category_orders[col] = order
    category_maps[col] = {
        value: index for index, value in enumerate(order)
    }

print()
print("Categorical ordering:")
for col in CATEGORICAL_QIDS:
    print(f"{col}: {category_orders[col]}")

# ------------------------------------------------------------
# Convert records to Mondrian format:
# [qid1, qid2, ..., qid8, sensitive]
# ------------------------------------------------------------

data = []

for _, row in df.iterrows():
    record = []

    for col in QID_COLUMNS:
        if col in NUMERIC_QIDS:
            record.append(int(row[col]))
        else:
            record.append(category_maps[col][row[col]])

    record.append(row[SENSITIVE_COLUMN])
    data.append(record)

assert len(data) == len(df)
assert all(len(record) == 9 for record in data)

# ------------------------------------------------------------
# Run strict Mondrian
# ------------------------------------------------------------

print()
print("Running strict Mondrian...")

result, eval_result = mondrian(
    copy.deepcopy(data),
    K,
    False
)

ncp = eval_result[0]
runtime = eval_result[1]

# ------------------------------------------------------------
# Calculate equivalence classes and DM
# ------------------------------------------------------------

equivalence_classes = Counter(
    tuple(record[:len(QID_COLUMNS)])
    for record in result
)

class_sizes = list(equivalence_classes.values())

assert class_sizes, "Mondrian returned no equivalence classes"

discernibility = sum(size ** 2 for size in class_sizes)

# ------------------------------------------------------------
# Verify output
# ------------------------------------------------------------

print()
print("=" * 70)
print("STANDARD MONDRIAN RESULTS")
print("=" * 70)

print(f"Input rows              : {len(df):,}")
print(f"Output rows             : {len(result):,}")
print(f"QID count               : {len(QID_COLUMNS)}")
print(f"NCP (%)                 : {ncp:.4f}")
print(f"Runtime (seconds)       : {runtime:.4f}")
print(f"Equivalence classes     : {len(class_sizes):,}")
print(f"Minimum class size      : {min(class_sizes):,}")
print(f"Maximum class size      : {max(class_sizes):,}")
print(f"Mean class size         : {sum(class_sizes)/len(class_sizes):.2f}")
print(f"Discernibility Metric   : {discernibility:,}")

assert len(result) == len(df), (
    f"Record count changed: {len(df)} -> {len(result)}"
)

assert min(class_sizes) >= K, (
    f"k-anonymity violation: min class size = {min(class_sizes)}"
)

assert sum(class_sizes) == len(result)

print()
print("Record accounting       : PASS")
print("k=10 anonymity          : PASS")

# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------

results_dir = ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

summary_path = results_dir / "milestone3_mondrian_standard_8qid.txt"

summary = f"""MILESTONE 3 — STANDARD ADULT MONDRIAN BASELINE

Dataset: Adult Census Income
Input records: {len(df)}
Output records: {len(result)}
QIDs: {", ".join(QID_COLUMNS)}
Sensitive attribute: {SENSITIVE_COLUMN}
k: {K}
Algorithm: qiyuangong/Mondrian
Model: strict

NCP (%): {ncp:.4f}
Runtime (seconds): {runtime:.4f}
Equivalence classes: {len(class_sizes)}
Minimum class size: {min(class_sizes)}
Maximum class size: {max(class_sizes)}
Mean class size: {sum(class_sizes)/len(class_sizes):.2f}
Discernibility Metric: {discernibility}

Record accounting: PASS
k-anonymity verification: PASS
"""

summary_path.write_text(summary, encoding="utf-8")

print(f"Saved summary: {summary_path}")
print()
print("=" * 70)
print("STANDARD MONDRIAN BASELINE COMPLETE")
print("=" * 70)

In [26]:
# Cell 26
# ============================================================
# MILESTONE 3 — STANDARD ADULT ARX BASELINE
# 8 QIDs, k=10
#
# This baseline:
#   - uses the same 30,162-row Adult dataset as Mondrian
#   - uses explicit hand-built hierarchies
#   - uses k-anonymity with k=10
#   - uses zero tuple suppression
#   - keeps income unchanged as a non-QID attribute
#   - verifies the materialized output independently
# ============================================================

from pathlib import Path
from collections import Counter
import subprocess
import time
import pandas as pd

ROOT = Path("/content/kc-slice")

ARX_DIR = ROOT / "methods/arx"
JAR = ARX_DIR / "libarx-3.9.2.jar"

DATA = ROOT / "data/processed/adult_standard_baseline_k10_1sa.csv"

HIER_DIR = ROOT / "hierarchies/adult_arx_standard"
OUT_DIR = ROOT / "results/arx_standard_adult"

HIER_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

JAVA_FILE = ARX_DIR / "AdultArxBaseline.java"
OUTPUT_FILE = OUT_DIR / "adult_standard_arx_k10.csv"

K = 10

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

SENSITIVE_COLUMN = "income"


# ============================================================
# 1. Verify prerequisites
# ============================================================

assert DATA.exists(), f"Missing input dataset: {DATA}"
assert JAR.exists(), f"Missing ARX JAR: {JAR}"

df = pd.read_csv(DATA)

assert len(df) == 30162, (
    f"Expected 30,162 input rows, got {len(df)}"
)

expected_columns = QID_COLUMNS + [SENSITIVE_COLUMN]

assert list(df.columns) == expected_columns, (
    "Unexpected baseline columns:\n"
    f"{df.columns.tolist()}"
)

print("=" * 70)
print("MILESTONE 3 — STANDARD ADULT ARX BASELINE")
print("=" * 70)
print(f"Input rows : {len(df):,}")
print(f"QID count  : {len(QID_COLUMNS)}")
print(f"Sensitive  : {SENSITIVE_COLUMN}")
print(f"k          : {K}")
print(f"ARX JAR    : {JAR}")


# ============================================================
# 2. Create ARX hierarchies
# ============================================================

print()
print("Creating ARX hierarchies...")


# ------------------------------------------------------------
# AGE
#
# exact age -> decade -> *
# ------------------------------------------------------------

age_hierarchy = HIER_DIR / "age.csv"

with age_hierarchy.open("w", encoding="utf-8") as f:

    for age in sorted(df["age"].astype(int).unique()):

        decade_start = (age // 10) * 10
        decade_end = min(decade_start + 9, 99)

        decade = f"{decade_start}-{decade_end}"

        f.write(f"{age},{decade},*\n")


# ------------------------------------------------------------
# EDUCATION.NUM
#
# exact value -> band -> *
# ------------------------------------------------------------

education_hierarchy = HIER_DIR / "education.num.csv"

with education_hierarchy.open("w", encoding="utf-8") as f:

    for value in sorted(df["education.num"].astype(int).unique()):

        if value <= 4:
            band = "1-4"
        elif value <= 6:
            band = "5-6"
        elif value <= 8:
            band = "7-8"
        elif value <= 10:
            band = "9-10"
        elif value <= 12:
            band = "11-12"
        elif value <= 14:
            band = "13-14"
        else:
            band = "15-16"

        f.write(f"{value},{band},*\n")


# ------------------------------------------------------------
# CATEGORICAL QIDS
#
# exact category -> *
# ------------------------------------------------------------

categorical_qids = [
    col
    for col in QID_COLUMNS
    if col not in ["age", "education.num"]
]

for col in categorical_qids:

    hierarchy_path = HIER_DIR / f"{col}.csv"

    # Preserve first-appearance order.
    values = list(
        dict.fromkeys(
            df[col].astype(str).tolist()
        )
    )

    with hierarchy_path.open("w", encoding="utf-8") as f:

        for value in values:
            f.write(f"{value},*\n")


print("Hierarchies created:")

for path in sorted(HIER_DIR.glob("*.csv")):
    print(f"  {path.name}")


# ============================================================
# 3. Generate Java ARX runner
# ============================================================

print()
print("Generating Java ARX runner...")

java_source = r'''
import java.io.File;
import java.nio.charset.StandardCharsets;

import org.deidentifier.arx.ARXAnonymizer;
import org.deidentifier.arx.ARXConfiguration;
import org.deidentifier.arx.ARXResult;
import org.deidentifier.arx.AttributeType;
import org.deidentifier.arx.Data;
import org.deidentifier.arx.criteria.KAnonymity;
import org.deidentifier.arx.metric.Metric;

public class AdultArxBaseline {

    public static void main(String[] args) throws Exception {

        if (args.length != 3) {
            System.err.println(
                "Usage: AdultArxBaseline <input.csv> <hierarchy_dir> <output.csv>"
            );
            System.exit(1);
        }

        String inputPath = args[0];
        String hierarchyDir = args[1];
        String outputPath = args[2];

        // ----------------------------------------------------
        // Load input dataset
        // ----------------------------------------------------

        System.out.println("Loading ARX input...");

        Data data = Data.create(
            inputPath,
            StandardCharsets.UTF_8,
            ','
        );

        // ----------------------------------------------------
        // QIDs and their hierarchies
        // ----------------------------------------------------

        String[] qids = {
            "age",
            "workclass",
            "education.num",
            "marital.status",
            "occupation",
            "race",
            "sex",
            "native.country"
        };

        for (String qid : qids) {

            File hierarchyFile = new File(
                hierarchyDir,
                qid + ".csv"
            );

            // Mark attribute as a quasi-identifier.
            data.getDefinition().setAttributeType(
                qid,
                AttributeType.QUASI_IDENTIFYING_ATTRIBUTE
            );

            // Attach hierarchy.
            data.getDefinition().setHierarchy(
                qid,
                AttributeType.Hierarchy.create(
                    hierarchyFile,
                    StandardCharsets.UTF_8,
                    ','
                )
            );
        }

        // ----------------------------------------------------
        // Income
        //
        // This k-anonymity baseline does not use a
        // sensitive-attribute privacy criterion.
        //
        // Therefore income is preserved unchanged and is
        // excluded from the QID equivalence relation.
        // ----------------------------------------------------

        data.getDefinition().setAttributeType(
            "income",
            AttributeType.INSENSITIVE_ATTRIBUTE
        );

        // ----------------------------------------------------
        // ARX configuration
        // ----------------------------------------------------

        ARXConfiguration config = ARXConfiguration.create();

        // k-anonymity
        config.addPrivacyModel(
            new KAnonymity(10)
        );

        // Zero suppression: every input record must remain.
        config.setSuppressionLimit(0d);

        // ARX loss metric used for optimization.
        config.setQualityModel(
            Metric.createLossMetric()
        );

        // ----------------------------------------------------
        // Run ARX
        // ----------------------------------------------------

        System.out.println("Running ARX...");

        ARXResult result =
            new ARXAnonymizer().anonymize(
                data,
                config
            );

        // ----------------------------------------------------
        // Verify ARX produced a valid optimum
        // ----------------------------------------------------

        if (!result.getOptimumFound()) {

            throw new RuntimeException(
                "ARX did not find a valid optimum."
            );
        }

        if (result.getGlobalOptimum() == null) {

            throw new RuntimeException(
                "ARX returned a null global optimum."
            );
        }

        System.out.println(
            "Global optimum found: PASS"
        );

        // ----------------------------------------------------
        // Save anonymized output
        // ----------------------------------------------------

        System.out.println(
            "Saving anonymized output..."
        );

        result
            .getOutput(false)
            .save(
                outputPath,
                ','
            );

        System.out.println(
            "Output saved: " + outputPath
        );
    }
}
'''

JAVA_FILE.write_text(
    java_source,
    encoding="utf-8"
)

print(f"Java source: {JAVA_FILE}")


# ============================================================
# 4. Compile ARX runner
# ============================================================

print()
print("Compiling ARX runner...")

compile_cmd = [
    "javac",
    "-cp",
    str(JAR),
    str(JAVA_FILE),
]

compile_result = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

if compile_result.stdout.strip():
    print(compile_result.stdout)

if compile_result.stderr.strip():
    print(compile_result.stderr)

if compile_result.returncode != 0:

    raise RuntimeError(
        "ARX Java compilation failed."
    )

print("Compilation: PASS")


# ============================================================
# 5. Run ARX
# ============================================================

print()
print("Running ARX anonymization...")

run_cmd = [
    "java",
    "-Xmx4g",
    "-cp",
    f"{JAR}:{ARX_DIR}",
    "AdultArxBaseline",
    str(DATA),
    str(HIER_DIR),
    str(OUTPUT_FILE),
]

start_time = time.perf_counter()

run_result = subprocess.run(
    run_cmd,
    capture_output=True,
    text=True
)

runtime_seconds = (
    time.perf_counter() - start_time
)

if run_result.stdout.strip():
    print(run_result.stdout)

if run_result.stderr.strip():
    print("ARX stderr:")
    print(run_result.stderr)

if run_result.returncode != 0:

    raise RuntimeError(
        "ARX anonymization failed."
    )

print(
    f"Python-measured runtime (seconds): "
    f"{runtime_seconds:.4f}"
)


# ============================================================
# 6. Verify ARX output exists
# ============================================================

assert OUTPUT_FILE.exists(), (
    f"ARX output not found: {OUTPUT_FILE}"
)


# ============================================================
# 7. Load anonymized output
# ============================================================

arx_output = pd.read_csv(
    OUTPUT_FILE
)

print()
print("=" * 70)
print("ARX OUTPUT VERIFICATION")
print("=" * 70)

print(
    f"Input rows  : {len(df):,}"
)

print(
    f"Output rows : {len(arx_output):,}"
)


# ============================================================
# 8. Record accounting
# ============================================================

assert len(arx_output) == len(df), (
    "Record accounting failed: "
    f"{len(df):,} input rows -> "
    f"{len(arx_output):,} output rows"
)

print(
    "Record accounting       : PASS"
)


# ============================================================
# 9. Calculate realized equivalence classes
# ============================================================

equivalence_classes = Counter(
    tuple(row[col] for col in QID_COLUMNS)
    for _, row in arx_output.iterrows()
)

class_sizes = list(
    equivalence_classes.values()
)

assert len(class_sizes) > 0, (
    "No equivalence classes were produced."
)

discernibility = sum(
    size ** 2
    for size in class_sizes
)

mean_class_size = (
    sum(class_sizes) /
    len(class_sizes)
)

print()
print(
    f"Equivalence classes    : "
    f"{len(class_sizes):,}"
)

print(
    f"Minimum class size     : "
    f"{min(class_sizes):,}"
)

print(
    f"Maximum class size     : "
    f"{max(class_sizes):,}"
)

print(
    f"Mean class size        : "
    f"{mean_class_size:.2f}"
)

print(
    f"Discernibility Metric  : "
    f"{discernibility:,}"
)


# ============================================================
# 10. Verify k-anonymity
# ============================================================

assert min(class_sizes) >= K, (
    "k-anonymity verification failed: "
    f"minimum class size = {min(class_sizes)}"
)

assert sum(class_sizes) == len(arx_output), (
    "Equivalence-class accounting failed."
)

print(
    "k=10 anonymity          : PASS"
)


# ============================================================
# 11. Save milestone summary
# ============================================================

summary_path = (
    ROOT /
    "results" /
    "milestone3_arx_baseline.txt"
)

summary = f"""
MILESTONE 3 — STANDARD ADULT ARX BASELINE

Dataset:
  Adult Census Income

Input records:
  {len(df)}

Output records:
  {len(arx_output)}

QIDs:
{chr(10).join("  - " + q for q in QID_COLUMNS)}

Non-QID attribute:
  income

Privacy model:
  k-anonymity

k:
  {K}

Suppression limit:
  0

ARX version:
  3.9.2

Runtime (seconds):
  {runtime_seconds:.4f}

Equivalence classes:
  {len(class_sizes)}

Minimum class size:
  {min(class_sizes)}

Maximum class size:
  {max(class_sizes)}

Mean class size:
  {mean_class_size:.2f}

Discernibility Metric:
  {discernibility}

Record accounting:
  PASS

k-anonymity:
  PASS

Output:
  {OUTPUT_FILE}
""".strip() + "\n"

summary_path.write_text(
    summary,
    encoding="utf-8"
)

print()
print(
    f"Saved summary: {summary_path}"
)

print()
print("=" * 70)
print("STANDARD ADULT ARX BASELINE COMPLETE")
print("=" * 70)

In [27]:
# Cell 27
# ============================================================
# MILESTONE 3 — ADULT ARX VGH SPECIFICATION
#
# The source comparison specifies:
#   Age              -> height 4, using 5-, 10-, 20-year ranges
#   Gender            -> height 1
#   Race              -> height 1
#   Marital Status    -> height 2
#   Native Country    -> height 2
#   Work Class        -> height 2
#   Occupation        -> height 2
#   Education         -> height 3
#
# Our Adult table uses:
#   sex              instead of gender
#   education.num    instead of education
#
# The exact categorical parent groupings below are an
# operational project decision because the supplied source
# specifies the hierarchy heights but does not provide every
# categorical parent mapping.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

DATA = ROOT / "data/processed/adult_standard_baseline_k10_1sa.csv"

VGH_DIR = ROOT / "hierarchies/adult_arx_standard_vgh"
VGH_DIR.mkdir(parents=True, exist_ok=True)

SPEC_FILE = ROOT / "results/adult_vgh_specification.txt"

df = pd.read_csv(DATA)

assert len(df) == 30162

print("=" * 70)
print("MILESTONE 3 — ADULT ARX VGH SPECIFICATION")
print("=" * 70)
print(f"Dataset rows: {len(df):,}")


# ============================================================
# 1. AGE VGH
#
# Level 0: exact age
# Level 1: 5-year interval
# Level 2: 10-year interval
# Level 3: 20-year interval
# Level 4: *
# ============================================================

age_path = VGH_DIR / "age.csv"

with age_path.open("w", encoding="utf-8") as f:

    for age in sorted(df["age"].astype(int).unique()):

        # 5-year bucket
        start5 = (age // 5) * 5
        end5 = start5 + 4
        level1 = f"{start5}-{end5}"

        # 10-year bucket
        start10 = (age // 10) * 10
        end10 = start10 + 9
        level2 = f"{start10}-{end10}"

        # 20-year bucket
        start20 = (age // 20) * 20
        end20 = start20 + 19
        level3 = f"{start20}-{end20}"

        f.write(
            f"{age},{level1},{level2},{level3},*\n"
        )


# ============================================================
# 2. SEX VGH
#
# Source: height 1
# exact -> *
# ============================================================

sex_path = VGH_DIR / "sex.csv"

with sex_path.open("w", encoding="utf-8") as f:

    for value in sorted(
        df["sex"].astype(str).unique()
    ):
        f.write(f"{value},*\n")


# ============================================================
# 3. RACE VGH
#
# Source: height 1
# exact -> *
# ============================================================

race_path = VGH_DIR / "race.csv"

with race_path.open("w", encoding="utf-8") as f:

    for value in sorted(
        df["race"].astype(str).unique()
    ):
        f.write(f"{value},*\n")


# ============================================================
# 4. MARITAL STATUS VGH
#
# Height 2:
#
# exact
#   -> broad marital group
#       -> *
# ============================================================

marital_groups = {
    "Never-married": "Not-married",
    "Divorced": "Not-married",
    "Separated": "Not-married",

    "Married-civ-spouse": "Married",
    "Married-spouse-absent": "Married",
    "Married-AF-spouse": "Married",

    "Widowed": "Widowed",
}

marital_path = VGH_DIR / "marital.status.csv"

for value in df["marital.status"].astype(str).unique():

    assert value in marital_groups, (
        f"Missing marital mapping: {value}"
    )

with marital_path.open("w", encoding="utf-8") as f:

    for value in df["marital.status"].astype(str).unique():

        f.write(
            f"{value},{marital_groups[value]},*\n"
        )


# ============================================================
# 5. WORKCLASS VGH
#
# Height 2:
#
# exact
#   -> employment-sector group
#       -> *
# ============================================================

workclass_groups = {
    "Private": "Private",

    "State-gov": "Government",
    "Federal-gov": "Government",
    "Local-gov": "Government",

    "Self-emp-not-inc": "Self-employed",
    "Self-emp-inc": "Self-employed",

    "Without-pay": "Other",
}

workclass_path = VGH_DIR / "workclass.csv"

for value in df["workclass"].astype(str).unique():

    assert value in workclass_groups, (
        f"Missing workclass mapping: {value}"
    )

with workclass_path.open("w", encoding="utf-8") as f:

    for value in df["workclass"].astype(str).unique():

        f.write(
            f"{value},{workclass_groups[value]},*\n"
        )


# ============================================================
# 6. OCCUPATION VGH
#
# Height 2:
#
# exact
#   -> occupational group
#       -> *
# ============================================================

occupation_groups = {
    "Exec-managerial": "Management",
    "Prof-specialty": "Professional",

    "Adm-clerical": "Office-support",
    "Tech-support": "Office-support",

    "Sales": "Commercial",
    "Other-service": "Service",

    "Protective-serv": "Protective",
    "Handlers-cleaners": "Manual-service",
    "Priv-house-serv": "Manual-service",

    "Craft-repair": "Skilled-trades",
    "Machine-op-inspct": "Skilled-trades",

    "Transport-moving": "Transportation",

    "Farming-fishing": "Agriculture",

    "Armed-Forces": "Military",
}

occupation_path = VGH_DIR / "occupation.csv"

for value in df["occupation"].astype(str).unique():

    assert value in occupation_groups, (
        f"Missing occupation mapping: {value}"
    )

with occupation_path.open("w", encoding="utf-8") as f:

    for value in df["occupation"].astype(str).unique():

        f.write(
            f"{value},{occupation_groups[value]},*\n"
        )


# ============================================================
# 7. NATIVE COUNTRY VGH
#
# Height 2:
#
# exact country
#   -> geographic region
#       -> *
# ============================================================

country_groups = {

    # North America
    "United-States": "North-America",
    "Canada": "North-America",

    # Mexico / Central America / Caribbean
    "Mexico": "Latin-America",
    "Guatemala": "Latin-America",
    "El-Salvador": "Latin-America",
    "Honduras": "Latin-America",
    "Nicaragua": "Latin-America",
    "Panama": "Latin-America",
    "Cuba": "Latin-America",
    "Jamaica": "Latin-America",
    "Puerto-Rico": "Latin-America",
    "Dominican-Republic": "Latin-America",
    "Haiti": "Latin-America",
    "Trinadad&Tobago": "Latin-America",
    "Outlying-US(Guam-USVI-etc)": "Latin-America",

    # South America
    "Columbia": "Latin-America",
    "Ecuador": "Latin-America",
    "Peru": "Latin-America",

    # Europe
    "England": "Europe",
    "Scotland": "Europe",
    "Ireland": "Europe",
    "France": "Europe",
    "Germany": "Europe",
    "Greece": "Europe",
    "Holand-Netherlands": "Europe",
    "Italy": "Europe",
    "Portugal": "Europe",
    "Poland": "Europe",
    "Yugoslavia": "Europe",
    "Hungary": "Europe",

    # Asia
    "China": "Asia",
    "Hong": "Asia",
    "India": "Asia",
    "Iran": "Asia",
    "Japan": "Asia",
    "Cambodia": "Asia",
    "Laos": "Asia",
    "Philippines": "Asia",
    "Taiwan": "Asia",
    "Thailand": "Asia",
    "Vietnam": "Asia",
    "South": "Asia",
}

country_path = VGH_DIR / "native.country.csv"

country_values = df["native.country"].astype(str).unique()

for value in country_values:

    assert value in country_groups, (
        f"Missing native.country mapping: {value}"
    )

with country_path.open("w", encoding="utf-8") as f:

    for value in country_values:

        f.write(
            f"{value},{country_groups[value]},*\n"
        )


# ============================================================
# 8. EDUCATION.NUM VGH
#
# Source Adult Education has 16 distinct values and height 3.
#
# Operational mapping:
#
# exact
#   -> 4-level education band
#       -> broad education group
#           -> *
# ============================================================

education_groups = {
    1:  "Basic-1-4",
    2:  "Basic-1-4",
    3:  "Basic-1-4",
    4:  "Basic-1-4",

    5:  "Basic-5-8",
    6:  "Basic-5-8",
    7:  "Basic-5-8",
    8:  "Basic-5-8",

    9:  "Secondary",
    10: "Secondary",
    11: "Secondary",
    12: "Secondary",

    13: "Post-secondary",
    14: "Post-secondary",
    15: "Post-secondary",
    16: "Post-secondary",
}

education_broad_groups = {
    "Basic-1-4": "Basic",
    "Basic-5-8": "Basic",
    "Secondary": "Secondary",
    "Post-secondary": "Post-secondary",
}

education_path = VGH_DIR / "education.num.csv"

education_values = sorted(
    df["education.num"].astype(int).unique()
)

for value in education_values:

    assert value in education_groups, (
        f"Missing education.num mapping: {value}"
    )

with education_path.open("w", encoding="utf-8") as f:

    for value in education_values:

        level1 = education_groups[value]
        level2 = education_broad_groups[level1]

        f.write(
            f"{value},{level1},{level2},*\n"
        )


# ============================================================
# 9. Verify hierarchy files
# ============================================================

expected_files = [
    "age.csv",
    "sex.csv",
    "race.csv",
    "marital.status.csv",
    "native.country.csv",
    "workclass.csv",
    "occupation.csv",
    "education.num.csv",
]

print()
print("Hierarchy files:")

for filename in expected_files:

    path = VGH_DIR / filename

    assert path.exists(), (
        f"Missing hierarchy: {path}"
    )

    rows = sum(
        1
        for _ in path.open(
            "r",
            encoding="utf-8"
        )
    )

    print(
        f"  {filename:<22} {rows:>3} leaf mappings"
    )


# ============================================================
# 10. Save exact hierarchy specification
# ============================================================

spec = """
MILESTONE 3 — ADULT ARX VGH SPECIFICATION
==========================================

Dataset:
  Adult Census Income

Rows:
  30162

Privacy parameter:
  k = 10

Hierarchy source:
  The comparison paper specifies the Adult VGH attribute
  cardinalities and hierarchy heights.

Source-specified structure:
  age:
      height 4
      exact -> 5-year -> 10-year -> 20-year -> *

  sex/gender:
      height 1
      exact -> *

  race:
      height 1
      exact -> *

  marital.status:
      height 2
      exact -> group -> *

  native.country:
      height 2
      exact -> geographic region -> *

  workclass:
      height 2
      exact -> employment-sector group -> *

  occupation:
      height 2
      exact -> occupational group -> *

  education:
      height 3

Project representation:
  education.num

  exact -> education band -> broad education group -> *

IMPORTANT:
  The source specifies the hierarchy heights and age range
  structure, but does not provide the complete categorical
  parent mapping for every Adult value.

  Therefore the categorical parent mappings used here are
  explicit operational project decisions and will be held
  fixed across the ARX runs rather than changed between
  sensitive-attribute levels.

This hierarchy is intended for the ARX generalization arm.
Mondrian uses partition-based generalization and does not
consume a VGH in the same way.
""".strip() + "\n"

SPEC_FILE.write_text(
    spec,
    encoding="utf-8"
)

print()
print(f"Saved VGH specification: {SPEC_FILE}")

print()
print("=" * 70)
print("ADULT ARX VGH SPECIFICATION COMPLETE")
print("=" * 70)

In [28]:
# Cell 28
# ============================================================
# MILESTONE 3 — STANDARD ADULT ARX BASELINE WITH FIXED VGH
# ============================================================

from pathlib import Path
from collections import Counter
import subprocess
import time
import pandas as pd

ROOT = Path("/content/kc-slice")

ARX_DIR = ROOT / "methods/arx"
JAR = ARX_DIR / "libarx-3.9.2.jar"

DATA = ROOT / "data/processed/adult_standard_baseline_k10_1sa.csv"

# IMPORTANT:
# Use the VGH created in Cell 27.
HIER_DIR = ROOT / "hierarchies/adult_arx_standard_vgh"

OUT_DIR = ROOT / "results/arx_standard_adult_vgh"
OUT_DIR.mkdir(parents=True, exist_ok=True)

JAVA_FILE = ARX_DIR / "AdultArxBaseline.java"
OUTPUT_FILE = OUT_DIR / "adult_standard_arx_vgh_k10.csv"

K = 10

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

SENSITIVE_COLUMN = "income"


# ============================================================
# 1. Verify prerequisites
# ============================================================

assert DATA.exists(), f"Missing input dataset: {DATA}"
assert JAR.exists(), f"Missing ARX JAR: {JAR}"
assert HIER_DIR.exists(), f"Missing VGH directory: {HIER_DIR}"

df = pd.read_csv(DATA)

assert len(df) == 30162

assert list(df.columns) == QID_COLUMNS + [SENSITIVE_COLUMN]

print("=" * 70)
print("MILESTONE 3 — STANDARD ADULT ARX BASELINE WITH FIXED VGH")
print("=" * 70)
print(f"Input rows : {len(df):,}")
print(f"QID count  : {len(QID_COLUMNS)}")
print(f"Sensitive  : {SENSITIVE_COLUMN}")
print(f"k          : {K}")
print(f"ARX JAR    : {JAR}")
print(f"VGH DIR    : {HIER_DIR}")


# ============================================================
# 2. Verify all hierarchy files
# ============================================================

print()
print("Checking VGH files...")

required_hierarchies = [
    "age.csv",
    "workclass.csv",
    "education.num.csv",
    "marital.status.csv",
    "occupation.csv",
    "race.csv",
    "sex.csv",
    "native.country.csv",
]

for filename in required_hierarchies:

    path = HIER_DIR / filename

    assert path.exists(), (
        f"Missing hierarchy file: {path}"
    )

    print(f"  {filename:<22} PASS")


# ============================================================
# 3. Generate Java ARX runner
# ============================================================

print()
print("Generating Java ARX runner...")

java_source = r'''
import java.io.File;
import java.nio.charset.StandardCharsets;

import org.deidentifier.arx.ARXAnonymizer;
import org.deidentifier.arx.ARXConfiguration;
import org.deidentifier.arx.ARXResult;
import org.deidentifier.arx.AttributeType;
import org.deidentifier.arx.Data;
import org.deidentifier.arx.criteria.KAnonymity;
import org.deidentifier.arx.metric.Metric;

public class AdultArxBaseline {

    public static void main(String[] args) throws Exception {

        if (args.length != 3) {
            System.err.println(
                "Usage: AdultArxBaseline <input.csv> <hierarchy_dir> <output.csv>"
            );
            System.exit(1);
        }

        String inputPath = args[0];
        String hierarchyDir = args[1];
        String outputPath = args[2];

        // ----------------------------------------------------
        // Load dataset
        // ----------------------------------------------------

        System.out.println("Loading ARX input...");

        Data data = Data.create(
            inputPath,
            StandardCharsets.UTF_8,
            ','
        );

        // ----------------------------------------------------
        // QIDs + fixed project VGH
        // ----------------------------------------------------

        String[] qids = {
            "age",
            "workclass",
            "education.num",
            "marital.status",
            "occupation",
            "race",
            "sex",
            "native.country"
        };

        for (String qid : qids) {

            File hierarchyFile = new File(
                hierarchyDir,
                qid + ".csv"
            );

            data.getDefinition().setAttributeType(
                qid,
                AttributeType.QUASI_IDENTIFYING_ATTRIBUTE
            );

            data.getDefinition().setHierarchy(
                qid,
                AttributeType.Hierarchy.create(
                    hierarchyFile,
                    StandardCharsets.UTF_8,
                    ','
                )
            );
        }

        // ----------------------------------------------------
        // Income
        //
        // Kept unchanged for this k-anonymity baseline.
        // It is not part of the QID equivalence relation.
        // ----------------------------------------------------

        data.getDefinition().setAttributeType(
            "income",
            AttributeType.INSENSITIVE_ATTRIBUTE
        );

        // ----------------------------------------------------
        // ARX configuration
        // ----------------------------------------------------

        ARXConfiguration config =
            ARXConfiguration.create();

        config.addPrivacyModel(
            new KAnonymity(10)
        );

        // Zero tuple suppression.
        config.setSuppressionLimit(0d);

        // ARX quality metric for optimization.
        config.setQualityModel(
            Metric.createLossMetric()
        );

        // ----------------------------------------------------
        // Run anonymization
        // ----------------------------------------------------

        System.out.println("Running ARX...");

        ARXResult result =
            new ARXAnonymizer().anonymize(
                data,
                config
            );

        // ----------------------------------------------------
        // Verify ARX found a valid optimum
        // ----------------------------------------------------

        if (!result.getOptimumFound()) {

            throw new RuntimeException(
                "ARX did not find a valid optimum."
            );
        }

        if (result.getGlobalOptimum() == null) {

            throw new RuntimeException(
                "ARX returned a null global optimum."
            );
        }

        System.out.println(
            "Global optimum found: PASS"
        );

        // ----------------------------------------------------
        // Save anonymized dataset
        // ----------------------------------------------------

        System.out.println(
            "Saving anonymized output..."
        );

        result
            .getOutput(false)
            .save(
                outputPath,
                ','
            );

        System.out.println(
            "Output saved: " + outputPath
        );
    }
}
'''

JAVA_FILE.write_text(
    java_source,
    encoding="utf-8"
)

print(f"Java source: {JAVA_FILE}")


# ============================================================
# 4. Compile
# ============================================================

print()
print("Compiling ARX runner...")

compile_cmd = [
    "javac",
    "-cp",
    str(JAR),
    str(JAVA_FILE),
]

compile_result = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

if compile_result.stdout.strip():
    print(compile_result.stdout)

if compile_result.stderr.strip():
    print(compile_result.stderr)

if compile_result.returncode != 0:

    raise RuntimeError(
        "ARX Java compilation failed."
    )

print("Compilation: PASS")


# ============================================================
# 5. Run ARX
# ============================================================

print()
print("Running ARX anonymization...")

run_cmd = [
    "java",
    "-Xmx4g",
    "-cp",
    f"{JAR}:{ARX_DIR}",
    "AdultArxBaseline",
    str(DATA),
    str(HIER_DIR),
    str(OUTPUT_FILE),
]

start_time = time.perf_counter()

run_result = subprocess.run(
    run_cmd,
    capture_output=True,
    text=True
)

runtime_seconds = (
    time.perf_counter() - start_time
)

if run_result.stdout.strip():
    print(run_result.stdout)

if run_result.stderr.strip():
    print("ARX stderr:")
    print(run_result.stderr)

if run_result.returncode != 0:

    raise RuntimeError(
        "ARX anonymization failed."
    )

print(
    f"Python-measured runtime (seconds): "
    f"{runtime_seconds:.4f}"
)


# ============================================================
# 6. Verify output
# ============================================================

assert OUTPUT_FILE.exists(), (
    f"ARX output not found: {OUTPUT_FILE}"
)

arx_output = pd.read_csv(
    OUTPUT_FILE
)

print()
print("=" * 70)
print("ARX OUTPUT VERIFICATION")
print("=" * 70)

print(
    f"Input rows  : {len(df):,}"
)

print(
    f"Output rows : {len(arx_output):,}"
)


# ============================================================
# 7. Record accounting
# ============================================================

assert len(arx_output) == len(df), (
    "Record accounting failed: "
    f"{len(df):,} -> {len(arx_output):,}"
)

print(
    "Record accounting       : PASS"
)


# ============================================================
# 8. Realized equivalence classes
# ============================================================

equivalence_classes = Counter(
    tuple(row[col] for col in QID_COLUMNS)
    for _, row in arx_output.iterrows()
)

class_sizes = list(
    equivalence_classes.values()
)

assert len(class_sizes) > 0

discernibility = sum(
    size ** 2
    for size in class_sizes
)

mean_class_size = (
    sum(class_sizes) /
    len(class_sizes)
)

print()
print(
    f"Equivalence classes    : "
    f"{len(class_sizes):,}"
)

print(
    f"Minimum class size     : "
    f"{min(class_sizes):,}"
)

print(
    f"Maximum class size     : "
    f"{max(class_sizes):,}"
)

print(
    f"Mean class size        : "
    f"{mean_class_size:.2f}"
)

print(
    f"Discernibility Metric  : "
    f"{discernibility:,}"
)


# ============================================================
# 9. Verify k-anonymity
# ============================================================

assert min(class_sizes) >= K, (
    "k-anonymity verification failed: "
    f"minimum class size = {min(class_sizes)}"
)

assert sum(class_sizes) == len(arx_output)

print(
    "k=10 anonymity          : PASS"
)


# ============================================================
# 10. Save final ARX VGH baseline summary
# ============================================================

summary_path = (
    ROOT /
    "results" /
    "milestone3_arx_vgh_baseline.txt"
)

summary = f"""
MILESTONE 3 — STANDARD ADULT ARX BASELINE WITH FIXED VGH

Dataset:
  Adult Census Income

Input records:
  {len(df)}

Output records:
  {len(arx_output)}

QIDs:
{chr(10).join("  - " + q for q in QID_COLUMNS)}

Non-QID attribute:
  {SENSITIVE_COLUMN}

Privacy model:
  k-anonymity

k:
  {K}

Suppression limit:
  0

ARX version:
  3.9.2

Hierarchy:
  {HIER_DIR}

Runtime (seconds):
  {runtime_seconds:.4f}

Equivalence classes:
  {len(class_sizes)}

Minimum class size:
  {min(class_sizes)}

Maximum class size:
  {max(class_sizes)}

Mean class size:
  {mean_class_size:.2f}

Discernibility Metric:
  {discernibility}

Record accounting:
  PASS

k-anonymity:
  PASS

Output:
  {OUTPUT_FILE}
""".strip() + "\n"

summary_path.write_text(
    summary,
    encoding="utf-8"
)

print()
print(
    f"Saved summary: {summary_path}"
)

print()
print("=" * 70)
print("STANDARD ADULT ARX VGH BASELINE COMPLETE")
print("=" * 70)

In [29]:
# Cell 29
# ============================================================
# ARX BASELINE — TRANSFORMATION AUDIT
# Inspect how much each QID was generalized.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

OUTPUT_FILE = (
    ROOT /
    "results/arx_standard_adult_vgh" /
    "adult_standard_arx_vgh_k10.csv"
)

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

df_arx = pd.read_csv(OUTPUT_FILE)

print("=" * 70)
print("ARX TRANSFORMATION AUDIT")
print("=" * 70)

print(f"Rows: {len(df_arx):,}")
print()

for col in QID_COLUMNS:

    values = df_arx[col].dropna().astype(str)

    print("-" * 70)
    print(f"{col}")
    print(f"Unique generalized values: {values.nunique():,}")

    sample_values = values.drop_duplicates().tolist()[:20]

    print("Sample values:")
    for value in sample_values:
        print(f"  {value}")

print()
print("=" * 70)
print("TRANSFORMATION AUDIT COMPLETE")
print("=" * 70)

In [30]:
# Cell 30
# ============================================================
# KC-SLICE — PARAMETER LOCK + QID CORRELATION AUDIT
#
# Baseline:
#   Sensitive Attribute  = income
#   Bucket size K        = 10
#   Privacy threshold C  = 5%
#
# HSA rule:
#   Most frequent value in each sensitive-attribute column.
#
# The KC-Slice paper requires a correlation operation on QIDs,
# but does not specify the exact statistical correlation
# function/threshold. Therefore this cell only AUDITS the
# QID relationships. We will not silently invent a threshold.
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

ROOT = Path("/content/kc-slice")

DATA = (
    ROOT /
    "data/processed/adult_standard_baseline_k10_1sa.csv"
)

df = pd.read_csv(DATA)

K_BUCKET = 10
C_THRESHOLD = 5.0

SA_COLUMNS = [
    "income"
]

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

assert len(df) == 30162

print("=" * 70)
print("KC-SLICE — PARAMETER LOCK + QID CORRELATION AUDIT")
print("=" * 70)

print(f"Rows                : {len(df):,}")
print(f"Sensitive attributes: {SA_COLUMNS}")
print(f"QIDs                : {len(QID_COLUMNS)}")
print(f"Bucket size K       : {K_BUCKET}")
print(f"Privacy threshold C : {C_THRESHOLD}%")
print()


# ============================================================
# 1. Determine HSA using deterministic mode rule
# ============================================================

print("-" * 70)
print("HSA SELECTION")
print("-" * 70)

HSA_VALUES = {}

for sa in SA_COLUMNS:

    counts = (
        df[sa]
        .astype(str)
        .value_counts()
    )

    hsa = counts.index[0]
    hsa_count = int(counts.iloc[0])

    HSA_VALUES[sa] = hsa

    hsa_percentage = (
        hsa_count /
        len(df)
        * 100
    )

    print(f"{sa}")
    print(f"  HSA value       : {hsa}")
    print(f"  Frequency       : {hsa_count:,}")
    print(f"  Dataset percent : {hsa_percentage:.2f}%")
    print()


# ============================================================
# 2. Display all QID cardinalities
# ============================================================

print("-" * 70)
print("QID CARDINALITIES")
print("-" * 70)

qid_cardinality = {}

for qid in QID_COLUMNS:

    n_unique = (
        df[qid]
        .astype(str)
        .nunique()
    )

    qid_cardinality[qid] = n_unique

    print(
        f"{qid:<22} "
        f"{n_unique:>4} distinct values"
    )

print()


# ============================================================
# 3. Numeric-numeric association
#
# Spearman correlation:
#   age <-> education.num
# ============================================================

print("-" * 70)
print("NUMERIC QID ASSOCIATION")
print("-" * 70)

numeric_qids = [
    "age",
    "education.num"
]

numeric_corr = df[numeric_qids].corr(
    method="spearman"
)

for i in range(len(numeric_qids)):

    for j in range(i + 1, len(numeric_qids)):

        a = numeric_qids[i]
        b = numeric_qids[j]

        value = numeric_corr.loc[a, b]

        print(
            f"Spearman({a}, {b}) = "
            f"{value:.4f}"
        )

print()


# ============================================================
# 4. Categorical-categorical association
#
# Cramér's V
# ============================================================

def cramers_v(x, y):

    table = pd.crosstab(
        x.astype(str),
        y.astype(str)
    ).to_numpy()

    n = table.sum()

    if n == 0:
        return 0.0

    row_sums = table.sum(axis=1)
    col_sums = table.sum(axis=0)

    expected = np.outer(
        row_sums,
        col_sums
    ) / n

    mask = expected > 0

    chi2 = (
        ((table - expected) ** 2)
        [mask]
        / expected[mask]
    ).sum()

    r, k = table.shape

    phi2 = chi2 / n

    # Bias-corrected Cramér's V
    if n <= 1:
        return 0.0

    phi2_corr = max(
        0,
        phi2 -
        ((k - 1) * (r - 1)) /
        (n - 1)
    )

    r_corr = (
        r -
        ((r - 1) ** 2) /
        (n - 1)
    )

    k_corr = (
        k -
        ((k - 1) ** 2) /
        (n - 1)
    )

    denominator = min(
        k_corr - 1,
        r_corr - 1
    )

    if denominator <= 0:
        return 0.0

    return float(
        np.sqrt(
            phi2_corr /
            denominator
        )
    )


categorical_qids = [
    "workclass",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

print("-" * 70)
print("CATEGORICAL QID ASSOCIATION — CRAMÉR'S V")
print("-" * 70)

categorical_results = []

for i in range(len(categorical_qids)):

    for j in range(i + 1, len(categorical_qids)):

        a = categorical_qids[i]
        b = categorical_qids[j]

        value = cramers_v(
            df[a],
            df[b]
        )

        categorical_results.append(
            (a, b, value)
        )

for a, b, value in sorted(
    categorical_results,
    key=lambda x: x[2],
    reverse=True
):

    print(
        f"{a:<20} <-> "
        f"{b:<20} "
        f"V = {value:.4f}"
    )

print()


# ============================================================
# 5. Numeric-categorical association
#
# Correlation ratio (eta)
# ============================================================

def correlation_ratio(categories, measurements):

    categories = pd.Series(
        categories
    ).astype(str)

    measurements = pd.to_numeric(
        measurements,
        errors="coerce"
    )

    valid = measurements.notna()

    categories = categories[valid]
    measurements = measurements[valid]

    if len(measurements) == 0:
        return 0.0

    grand_mean = measurements.mean()

    groups = []

    for category in categories.unique():

        values = measurements[
            categories == category
        ]

        if len(values) == 0:
            continue

        groups.append(
            (
                len(values),
                values.mean()
            )
        )

    numerator = sum(
        n * (mean - grand_mean) ** 2
        for n, mean in groups
    )

    denominator = sum(
        (measurements - grand_mean) ** 2
    )

    if denominator == 0:
        return 0.0

    return float(
        np.sqrt(
            numerator /
            denominator
        )
    )


print("-" * 70)
print("NUMERIC-CATEGORICAL ASSOCIATION — ETA")
print("-" * 70)

for numeric_qid in numeric_qids:

    for categorical_qid in categorical_qids:

        value = correlation_ratio(
            df[categorical_qid],
            df[numeric_qid]
        )

        print(
            f"eta({numeric_qid}, "
            f"{categorical_qid}) = "
            f"{value:.4f}"
        )

print()


# ============================================================
# 6. Save parameter specification
# ============================================================

PARAM_FILE = (
    ROOT /
    "results" /
    "kc_slice_adult_baseline_parameters.txt"
)

parameter_text = f"""
KC-SLICE ADULT BASELINE — OPERATIONAL PARAMETERS

Dataset:
  Adult Census Income

Rows:
  {len(df)}

Baseline sensitive attribute:
  income

Sensitive attributes:
  {", ".join(SA_COLUMNS)}

QID attributes:
  {", ".join(QID_COLUMNS)}

Project bucket size K:
  {K_BUCKET}

Privacy threshold C:
  {C_THRESHOLD}%

HSA selection rule:
  Most frequent value in each sensitive-attribute column.

HSA values selected:
{chr(10).join(f"  {sa}: {HSA_VALUES[sa]}" for sa in SA_COLUMNS)}

QID correlation:
  The KC-Slice paper requires a correlation function for QIDs,
  but does not specify the exact statistical measure or
  threshold. This cell audits candidate associations without
  selecting a threshold. The project will explicitly document
  the final correlation rule before the slicing implementation.

Important distinction:
  K=10 is the project-wide fixed anonymity/bucket parameter.
  The lowercase k derived in Algorithm 1 from dataset size
  and the summed distinct sensitive-attribute counts is not
  substituted for the project's fixed K=10.
""".strip() + "\n"

PARAM_FILE.write_text(
    parameter_text,
    encoding="utf-8"
)

print("=" * 70)
print("KC-SLICE PARAMETER AUDIT COMPLETE")
print("=" * 70)
print(f"Saved parameters: {PARAM_FILE}")

In [31]:
# Cell 31
# ============================================================
# KC-SLICE — BUCKET PARAMETER AUDIT
#
# IMPORTANT:
# The project keeps k = 10 as the overall experimental
# anonymity reference.
#
# KC-Slice itself derives its INTERNAL bucket size from:
#
#     bucket_size = |T| / sum(distinct SA values)
#
# as shown in Algorithm 1 of the paper.
#
# We therefore keep these as separate quantities:
#
#   PROJECT_K = 10
#       fixed experimental anonymity reference
#
#   KC_BUCKET_SIZE
#       internal bucket size derived from the KC-Slice
#       algorithm
# ============================================================

from pathlib import Path
import pandas as pd
import math

ROOT = Path("/content/kc-slice")

DATA = (
    ROOT /
    "data/processed/adult_standard_baseline_k10_1sa.csv"
)

RESULT_DIR = ROOT / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)

PROJECT_K = 10
PRIVACY_THRESHOLD_C = 5.0

# Adult sensitive-attribute axis from the project design.
SA_LEVELS = {
    1: ["income"],
    2: ["income", "race"],
    3: ["income", "race", "native.country"],
}

print("=" * 70)
print("KC-SLICE — BUCKET PARAMETER AUDIT")
print("=" * 70)

print(f"Dataset rows           : {len(df):,}")
print(f"Project k              : {PROJECT_K}")
print(f"Privacy threshold C    : {PRIVACY_THRESHOLD_C}%")
print()


# ============================================================
# Calculate internal KC-Slice bucket size
# ============================================================

bucket_specs = {}

for level, sa_columns in SA_LEVELS.items():

    cardinalities = {
        sa: int(
            df[sa]
            .astype(str)
            .nunique()
        )
        for sa in sa_columns
    }

    summed_cardinality = sum(
        cardinalities.values()
    )

    exact_bucket_size = (
        len(df) /
        summed_cardinality
    )

    # Algorithm requires a usable integer number of tuples
    # per bucket. We record floor() explicitly rather than
    # silently rounding.
    bucket_size = math.floor(
        exact_bucket_size
    )

    number_of_full_buckets = (
        len(df) //
        bucket_size
    )

    remainder = (
        len(df) %
        bucket_size
    )

    bucket_specs[level] = {
        "sa_columns": sa_columns,
        "cardinalities": cardinalities,
        "summed_cardinality": summed_cardinality,
        "exact_bucket_size": exact_bucket_size,
        "bucket_size": bucket_size,
        "number_of_full_buckets": number_of_full_buckets,
        "remainder": remainder,
    }

    print("-" * 70)
    print(f"SA LEVEL {level}")
    print("-" * 70)

    print(
        "Sensitive attributes: "
        + ", ".join(sa_columns)
    )

    print(
        "Distinct cardinalities:"
    )

    for sa, cardinality in cardinalities.items():

        print(
            f"  {sa:<18} "
            f"{cardinality:>4}"
        )

    print(
        f"Sum of cardinalities : "
        f"{summed_cardinality}"
    )

    print(
        f"Exact bucket size    : "
        f"{exact_bucket_size:.4f}"
    )

    print(
        f"Integer bucket size  : "
        f"{bucket_size}"
    )

    print(
        f"Full buckets         : "
        f"{number_of_full_buckets}"
    )

    print(
        f"Remainder            : "
        f"{remainder}"
    )

    print()


# ============================================================
# Baseline level used now
# ============================================================

baseline = bucket_specs[1]

print("=" * 70)
print("ADULT 1-SA BASELINE")
print("=" * 70)

print(
    f"Sensitive attribute  : "
    f"{baseline['sa_columns']}"
)

print(
    f"KC-Slice bucket size : "
    f"{baseline['bucket_size']}"
)

print(
    f"Project k            : "
    f"{PROJECT_K}"
)

print(
    f"Privacy C            : "
    f"{PRIVACY_THRESHOLD_C}%"
)

assert baseline["bucket_size"] > PROJECT_K, (
    "Derived KC-Slice bucket size is not greater "
    "than the project k=10 reference."
)

assert baseline["remainder"] < baseline["bucket_size"]


# ============================================================
# Save locked specification
# ============================================================

PARAM_FILE = (
    RESULT_DIR /
    "kc_slice_bucket_parameter_spec.txt"
)

spec_lines = [
    "KC-SLICE BUCKET PARAMETER SPECIFICATION",
    "",
    "Dataset: Adult Census Income",
    f"Records: {len(df)}",
    "",
    f"Project anonymity reference k: {PROJECT_K}",
    f"Privacy threshold C: {PRIVACY_THRESHOLD_C}%",
    "",
    "Sensitive-attribute levels:",
]

for level, spec in bucket_specs.items():

    spec_lines.append(
        f"Level {level}: "
        + ", ".join(spec["sa_columns"])
    )

    spec_lines.append(
        f"  Distinct counts: "
        + ", ".join(
            f"{sa}={n}"
            for sa, n in spec["cardinalities"].items()
        )
    )

    spec_lines.append(
        f"  Sum of cardinalities: "
        f"{spec['summed_cardinality']}"
    )

    spec_lines.append(
        f"  Exact bucket size: "
        f"{spec['exact_bucket_size']:.4f}"
    )

    spec_lines.append(
        f"  Integer bucket size: "
        f"{spec['bucket_size']}"
    )

    spec_lines.append(
        f"  Full buckets: "
        f"{spec['number_of_full_buckets']}"
    )

    spec_lines.append(
        f"  Remainder: "
        f"{spec['remainder']}"
    )

    spec_lines.append("")


spec_lines.extend([
    "Important distinction:",
    "  The project keeps k=10 fixed as the overall",
    "  experimental anonymity reference.",
    "",
    "  KC-Slice uses its paper-described internal bucket",
    "  calculation instead of treating k=10 as the bucket",
    "  size.",
    "",
    "This distinction is explicitly recorded because the",
    "project pre-implementation notes identify the relation",
    "between project k and KC-Slice's bucket calculation",
    "as an implementation question.",
])

PARAM_FILE.write_text(
    "\n".join(spec_lines) + "\n",
    encoding="utf-8"
)

print()
print(
    f"Saved specification: {PARAM_FILE}"
)

print()
print("=" * 70)
print("KC-SLICE BUCKET PARAMETER AUDIT COMPLETE")
print("=" * 70)

In [32]:
# Cell 32
# ============================================================
# KC-SLICE — ADULT 1-SA
# PHASE 1 + PHASE 2:
#   1. Create buckets
#   2. Privacy-check HSA frequencies
#
# Dataset:
#   Adult Census Income
#
# Sensitive attribute:
#   income
#
# Project reference:
#   k = 10
#
# KC-Slice internal bucket size:
#   15081
#
# Privacy threshold:
#   C = 5%
#
# HSA rule:
#   Most frequent sensitive value
#
# IMPORTANT:
# The KC-Slice paper does not fully specify the exact
# tuple-to-bucket allocation implementation. We therefore use
# deterministic frequency-balanced stratification here and
# record it explicitly as a project operationalization.
# ============================================================

from pathlib import Path
from collections import Counter
import math
import pandas as pd

ROOT = Path("/content/kc-slice")

DATA = (
    ROOT /
    "data/processed/adult_standard_baseline_k10_1sa.csv"
)

RESULT_DIR = ROOT / "results"
OUT_DIR = ROOT / "results/kc_slice_adult_1sa"

OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)

assert len(df) == 30162

SA_COLUMNS = ["income"]

PROJECT_K = 10
PRIVACY_C = 5.0

# Internal KC-Slice bucket size calculated in Cell 31.
BUCKET_SIZE = 15081

HSA_VALUES = {
    "income": "<=50K"
}

print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("PHASE 1 + PHASE 2")
print("=" * 70)

print(f"Input records         : {len(df):,}")
print(f"Sensitive attributes  : {SA_COLUMNS}")
print(f"Project k             : {PROJECT_K}")
print(f"KC-Slice bucket size  : {BUCKET_SIZE:,}")
print(f"Privacy threshold C   : {PRIVACY_C}%")
print(f"HSA                   : {HSA_VALUES['income']}")

assert len(df) % BUCKET_SIZE == 0, (
    "For the 1-SA baseline the derived bucket size should "
    "divide the Adult dataset exactly."
)

NUM_BUCKETS = len(df) // BUCKET_SIZE

print(f"Number of buckets     : {NUM_BUCKETS}")


# ============================================================
# 1. HSA frequency before bucketing
# ============================================================

print()
print("-" * 70)
print("GLOBAL SENSITIVE-ATTRIBUTE DISTRIBUTION")
print("-" * 70)

global_counts = (
    df["income"]
    .astype(str)
    .value_counts()
)

for value, count in global_counts.items():

    pct = (
        count /
        len(df) *
        100
    )

    marker = (
        " <-- HSA"
        if value == HSA_VALUES["income"]
        else ""
    )

    print(
        f"{value:<10} "
        f"{count:>7,} "
        f"({pct:>6.2f}%){marker}"
    )


# ============================================================
# 2. Deterministic frequency-balanced bucket creation
#
# Each sensitive value is divided as evenly as possible across
# the required buckets.
#
# For this Adult 1-SA case:
#
#   <=50K:
#       22,654 / 2 = 11,327 per bucket
#
#   >50K:
#        7,508 / 2 =  3,754 per bucket
#
# giving exactly 15,081 records per bucket.
# ============================================================

print()
print("-" * 70)
print("CREATING FREQUENCY-BALANCED BUCKETS")
print("-" * 70)

bucket_frames = [
    []
    for _ in range(NUM_BUCKETS)
]

# Work value-by-value so each bucket receives a balanced
# share of every sensitive value.
for sa_value in global_counts.index:

    value_rows = df[
        df["income"].astype(str) == str(sa_value)
    ].copy()

    value_rows = value_rows.reset_index(drop=False)

    n = len(value_rows)

    base = n // NUM_BUCKETS
    remainder = n % NUM_BUCKETS

    start = 0

    for bucket_id in range(NUM_BUCKETS):

        size = (
            base +
            (1 if bucket_id < remainder else 0)
        )

        end = start + size

        part = value_rows.iloc[
            start:end
        ].copy()

        part["_kc_bucket"] = bucket_id

        bucket_frames[
            bucket_id
        ].append(part)

        start = end


# Combine individual value partitions.
buckets = []

for bucket_id in range(NUM_BUCKETS):

    bucket = pd.concat(
        bucket_frames[bucket_id],
        ignore_index=True
    )

    # Deterministic row order.
    bucket = bucket.sort_values(
        "_kc_bucket"
    ).reset_index(drop=True)

    assert len(bucket) == BUCKET_SIZE

    buckets.append(bucket)


print(
    f"Created {len(buckets)} buckets."
)

for bucket_id, bucket in enumerate(buckets):

    print(
        f"  Bucket {bucket_id + 1}: "
        f"{len(bucket):,} records"
    )


# ============================================================
# 3. Privacy check
#
# Paper:
#
#     C_il = |A_sil| / K * 100
#
# If C_il > C:
#     suppress excess HSA values.
#
# ============================================================

print()
print("-" * 70)
print("PRIVACY CHECK")
print("-" * 70)

suppression_records = []

processed_buckets = []

for bucket_id, bucket in enumerate(buckets):

    bucket = bucket.copy()

    print(
        f"\nBucket {bucket_id + 1}"
    )

    for sa in SA_COLUMNS:

        hsa = HSA_VALUES[sa]

        # Locate HSA records.
        hsa_mask = (
            bucket[sa]
            .astype(str)
            ==
            str(hsa)
        )

        hsa_count = int(
            hsa_mask.sum()
        )

        frequency_pct = (
            hsa_count /
            len(bucket) *
            100
        )

        allowed_count = (
            PRIVACY_C /
            100
            *
            len(bucket)
        )

        # Number of HSA occurrences that must be suppressed.
        suppression_count = max(
            0,
            math.ceil(
                hsa_count -
                allowed_count
            )
        )

        print(
            f"  {sa}"
        )

        print(
            f"    HSA value          : "
            f"{hsa}"
        )

        print(
            f"    HSA count          : "
            f"{hsa_count:,}"
        )

        print(
            f"    HSA frequency      : "
            f"{frequency_pct:.2f}%"
        )

        print(
            f"    Allowed at C=5%   : "
            f"{allowed_count:.2f}"
        )

        print(
            f"    Suppress HSA count : "
            f"{suppression_count:,}"
        )

        # ----------------------------------------------------
        # Deterministic cell suppression.
        #
        # We retain the first allowed HSA occurrences and
        # suppress the remainder.
        #
        # This is an operational deterministic tie-breaking
        # rule because the paper does not specify which
        # individual records within an HSA cell are suppressed.
        # ----------------------------------------------------

        if suppression_count > 0:

            hsa_indices = list(
                bucket.index[hsa_mask]
            )

            suppress_indices = (
                hsa_indices[
                    int(allowed_count):
                ]
            )

            bucket.loc[
                suppress_indices,
                sa
            ] = "#####"

        remaining_hsa_count = int(
            (
                bucket[sa]
                .astype(str)
                ==
                str(hsa)
            ).sum()
        )

        remaining_pct = (
            remaining_hsa_count /
            len(bucket)
            *
            100
        )

        assert (
            remaining_pct
            <=
            PRIVACY_C + 1e-9
        )

        suppression_records.append({
            "bucket": bucket_id + 1,
            "attribute": sa,
            "hsa": hsa,
            "before_count": hsa_count,
            "before_percent": frequency_pct,
            "allowed_count": allowed_count,
            "suppressed_count": suppression_count,
            "after_count": remaining_hsa_count,
            "after_percent": remaining_pct,
        })

    processed_buckets.append(
        bucket
    )


# ============================================================
# 4. Combine processed buckets
# ============================================================

processed = pd.concat(
    processed_buckets,
    ignore_index=True
)

assert len(processed) == len(df)

print()
print("=" * 70)
print("PHASE 2 RESULT")
print("=" * 70)

suppression_df = pd.DataFrame(
    suppression_records
)

print(
    suppression_df.to_string(
        index=False
    )
)


# ============================================================
# 5. Global suppression statistics
# ============================================================

total_suppressed = int(
    suppression_df[
        "suppressed_count"
    ].sum()
)

total_cells = len(df)

suppression_ratio = (
    total_suppressed /
    total_cells *
    100
)

print()
print(
    f"Total HSA records suppressed : "
    f"{total_suppressed:,}"
)

print(
    f"Suppression ratio            : "
    f"{suppression_ratio:.2f}%"
)


# ============================================================
# 6. Verify bucket sizes
# ============================================================

bucket_sizes = (
    processed["_kc_bucket"]
    .value_counts()
    .sort_index()
)

print()
print("Bucket sizes after privacy check:")

for bucket_id, size in bucket_sizes.items():

    print(
        f"  Bucket {int(bucket_id) + 1}: "
        f"{size:,}"
    )

assert all(
    size == BUCKET_SIZE
    for size in bucket_sizes
)


# ============================================================
# 7. Save intermediate output
# ============================================================

bucket_output = (
    OUT_DIR /
    "adult_1sa_bucket_privacy_output.csv"
)

processed.to_csv(
    bucket_output,
    index=False
)

privacy_output = (
    OUT_DIR /
    "adult_1sa_privacy_report.csv"
)

suppression_df.to_csv(
    privacy_output,
    index=False
)


# ============================================================
# 8. Save implementation notes
# ============================================================

notes_path = (
    OUT_DIR /
    "adult_1sa_implementation_notes.txt"
)

notes = f"""
KC-SLICE ADULT 1-SA — PHASE 1 + PHASE 2

Dataset:
  Adult Census Income

Records:
  {len(df)}

Sensitive attribute:
  income

HSA:
  {HSA_VALUES["income"]}

Project k:
  {PROJECT_K}

KC-Slice internal bucket size:
  {BUCKET_SIZE}

Number of buckets:
  {NUM_BUCKETS}

Privacy threshold:
  C = {PRIVACY_C}%

Bucket construction:
  Deterministic frequency-balanced stratification.

Cell suppression:
  Deterministic suppression of HSA occurrences beyond
  the allowed C threshold.

Important implementation note:
  The KC-Slice paper describes bucket creation using
  sensitive-attribute cardinalities and frequency
  distributions, but does not fully specify a unique
  tuple-to-bucket allocation procedure.

  Therefore frequency-balanced stratification is an
  explicit project operationalization.

Important:
  This cell implements only Phase 1 and Phase 2.
  QID correlation, concatenation, SID linkage and
  random permutation are implemented separately.
""".strip() + "\n"

notes_path.write_text(
    notes,
    encoding="utf-8"
)

print()
print(f"Saved bucket output : {bucket_output}")
print(f"Saved privacy report: {privacy_output}")
print(f"Saved notes         : {notes_path}")

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA PHASE 1 + PHASE 2 COMPLETE")
print("=" * 70)

In [33]:
# Cell 33 — KC-SLICE ADULT 1-SA
# PHASE 3A: SENSITIVE CELL GROUPING + SID GENERATION

from pathlib import Path
from collections import OrderedDict
import pandas as pd

ROOT = Path("/content/kc-slice")

INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_bucket_privacy_output.csv"
)

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

df = pd.read_csv(INPUT)

SA_COLUMNS = ["income"]
BUCKET_COLUMN = "_kc_bucket"

print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("PHASE 3A: SENSITIVE CELL GROUPING + SID")
print("=" * 70)

print(f"Input rows            : {len(df):,}")
print(f"Sensitive attributes  : {SA_COLUMNS}")

bucket_ids = sorted(
    df[BUCKET_COLUMN].unique()
)

print(f"Buckets               : {len(bucket_ids)}")


# ============================================================
# Generate one SID for each SA cell within each bucket.
#
# Paper's Algorithm 2:
#   7. Select SA column from bucket
#   8. Generate SID for the cell
#   9. Cluster SID with the cell
#  12. Merge the SA cells
#
# We use deterministic SIDs:
#
#   Bucket 1, SA 1 -> 011
#   Bucket 2, SA 1 -> 021
#
# The first digit identifies the bucket and the remaining
# digits identify the SA column.
# ============================================================

sensitive_tables = []
row_sid_records = []

for bucket_number, bucket_id in enumerate(
    bucket_ids,
    start=1
):

    bucket = df[
        df[BUCKET_COLUMN] == bucket_id
    ].copy()

    print()
    print("-" * 70)
    print(f"BUCKET {bucket_number}")
    print("-" * 70)

    for sa_index, sa in enumerate(
        SA_COLUMNS,
        start=1
    ):

        # ----------------------------------------------------
        # One SID for the complete SA cell.
        # ----------------------------------------------------

        sid = (
            f"{bucket_number:02d}"
            f"{sa_index:01d}"
        )

        # ----------------------------------------------------
        # Count every resulting sensitive value, including
        # suppressed values.
        # ----------------------------------------------------

        counts = (
            bucket[sa]
            .astype(str)
            .value_counts()
        )

        # Preserve deterministic ordering.
        counts = OrderedDict(
            (
                str(value),
                int(count)
            )
            for value, count
            in counts.items()
        )

        # ----------------------------------------------------
        # Represent the cell exactly like the paper's
        # illustration:
        #
        # value(count), value(count), ...
        # ----------------------------------------------------

        cell_parts = [
            f"{value}({count})"
            for value, count
            in counts.items()
        ]

        cell_text = ", ".join(
            cell_parts
        )

        sensitive_tables.append({
            "bucket": bucket_number,
            "bucket_id": bucket_id,
            "SID": sid,
            "SA": sa,
            "cell": cell_text,
            "cell_record_count": sum(
                counts.values()
            ),
        })

        # ----------------------------------------------------
        # Link every original record in this bucket to the
        # single SID representing this SA cell.
        # ----------------------------------------------------

        for original_index in bucket["_kc_bucket"].index:

            row_sid_records.append({
                "bucket": bucket_number,
                "bucket_id": bucket_id,
                "row_index": original_index,
                "SA": sa,
                "SID": sid,
            })

        print(f"SA                      : {sa}")
        print(f"SID                     : {sid}")

        for value, count in counts.items():

            pct = (
                count /
                len(bucket) *
                100
            )

            print(
                f"  {value:<10} "
                f"{count:>7,} "
                f"({pct:>7.3f}%)"
            )

        print()
        print(f"Cell representation:")
        print(f"  {cell_text}")


# ============================================================
# Build grouped sensitive table.
# ============================================================

sensitive_table = pd.DataFrame(
    sensitive_tables
)

row_sid_table = pd.DataFrame(
    row_sid_records
)


# ============================================================
# Verification
# ============================================================

print()
print("=" * 70)
print("VERIFICATION")
print("=" * 70)

# For each bucket + SA, cell count must equal bucket size.
expected_cell_rows = (
    len(bucket_ids) *
    len(SA_COLUMNS)
)

assert len(sensitive_table) == expected_cell_rows

for _, row in sensitive_table.iterrows():

    bucket_size = len(
        df[
            df[BUCKET_COLUMN]
            ==
            row["bucket_id"]
        ]
    )

    assert (
        row["cell_record_count"]
        ==
        bucket_size
    )

print(
    "Sensitive-cell record accounting : PASS"
)

# Every bucket/SA combination must have one SID.
assert (
    sensitive_table
    .groupby(
        ["bucket_id", "SA"]
    )["SID"]
    .nunique()
    .eq(1)
    .all()
)

print(
    "One SID per bucket/SA cell      : PASS"
)

# Every source row must receive exactly one SID
# because this is a 1-SA baseline.
expected_sid_links = len(df)

assert len(row_sid_table) == expected_sid_links

print(
    "Row-to-SID accounting            : PASS"
)


# ============================================================
# Save results.
# ============================================================

SA_OUTPUT = (
    OUT_DIR /
    "adult_1sa_sensitive_cells.csv"
)

SID_OUTPUT = (
    OUT_DIR /
    "adult_1sa_row_sid_links.csv"
)

sensitive_table.to_csv(
    SA_OUTPUT,
    index=False
)

row_sid_table.to_csv(
    SID_OUTPUT,
    index=False
)


# ============================================================
# Save human-readable specification.
# ============================================================

SPEC_OUTPUT = (
    OUT_DIR /
    "adult_1sa_sensitive_cell_specification.txt"
)

with open(
    SPEC_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — SENSITIVE CELL SPECIFICATION\n"
    )
    f.write("=" * 60 + "\n\n")

    f.write(
        "Each sensitive attribute column within each bucket "
        "is represented as one grouped cell with value counts.\n"
    )

    f.write(
        "One deterministic SID is generated for each "
        "bucket/SA cell.\n\n"
    )

    for _, row in sensitive_table.iterrows():

        f.write(
            f"Bucket {row['bucket']} | "
            f"SA={row['SA']} | "
            f"SID={row['SID']}\n"
        )

        f.write(
            f"  {row['cell']}\n\n"
        )

print()
print("Sensitive-cell table:")
print(
    sensitive_table[
        [
            "bucket",
            "SID",
            "SA",
            "cell",
        ]
    ].to_string(index=False)
)

print()
print(f"Saved SA cells : {SA_OUTPUT}")
print(f"Saved SID links: {SID_OUTPUT}")
print(f"Saved spec     : {SPEC_OUTPUT}")

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA PHASE 3A COMPLETE")
print("=" * 70)

In [34]:
# Cell 34 — KC-SLICE ADULT 1-SA
# QID CORRELATION AUDIT
#
# Purpose:
#   Audit QID associations INSIDE EACH KC-Slice bucket.
#
# Important:
#   The KC-Slice paper specifies a Correlate() operation but
#   does not specify the exact statistical measure or threshold.
#
# Therefore this cell only measures the associations.
# It does NOT lock a correlation threshold yet.
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path("/content/kc-slice")

INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_bucket_privacy_output.csv"
)

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT)

QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

BUCKET_COLUMN = "_kc_bucket"

NUMERIC_QIDS = {
    "age",
    "education.num",
}

CATEGORICAL_QIDS = [
    q for q in QIDS
    if q not in NUMERIC_QIDS
]


# ============================================================
# Association functions
# ============================================================

def cramers_v(x, y):
    """
    Bias-unadjusted Cramer's V.
    Used for categorical-categorical association.
    """
    table = pd.crosstab(
        x,
        y
    )

    if table.empty:
        return 0.0

    observed = table.to_numpy(
        dtype=float
    )

    n = observed.sum()

    if n == 0:
        return 0.0

    row_sums = observed.sum(axis=1)
    col_sums = observed.sum(axis=0)

    expected = np.outer(
        row_sums,
        col_sums
    ) / n

    mask = expected > 0

    chi2 = (
        (
            observed[mask] -
            expected[mask]
        ) ** 2 /
        expected[mask]
    ).sum()

    phi2 = chi2 / n

    r, k = observed.shape

    return float(
        np.sqrt(
            phi2 /
            max(
                1,
                min(k - 1, r - 1)
            )
        )
    )


def correlation_ratio(categories, values):
    """
    Eta coefficient.
    Measures numeric-categorical association.
    """
    categories = pd.Series(
        categories
    )

    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    )

    valid = (
        categories.notna()
        &
        values.notna()
    )

    categories = categories[
        valid
    ]

    values = values[
        valid
    ]

    if len(values) == 0:
        return 0.0

    grand_mean = values.mean()

    numerator = 0.0

    for category in categories.unique():

        group = values[
            categories == category
        ]

        numerator += (
            len(group) *
            (
                group.mean()
                -
                grand_mean
            ) ** 2
        )

    denominator = (
        (
            values -
            grand_mean
        ) ** 2
    ).sum()

    if denominator == 0:
        return 0.0

    return float(
        np.sqrt(
            numerator /
            denominator
        )
    )


def association(x, y, x_name, y_name):
    """
    Select an association measure based on QID types.

    numeric-numeric:
        absolute Spearman rho

    categorical-categorical:
        Cramer's V

    numeric-categorical:
        Eta
    """

    x_numeric = x_name in NUMERIC_QIDS
    y_numeric = y_name in NUMERIC_QIDS

    if x_numeric and y_numeric:

        x_num = pd.to_numeric(
            x,
            errors="coerce"
        )

        y_num = pd.to_numeric(
            y,
            errors="coerce"
        )

        valid = (
            x_num.notna()
            &
            y_num.notna()
        )

        if valid.sum() < 2:
            return 0.0, "Spearman"

        rho, _ = spearmanr(
            x_num[valid],
            y_num[valid]
        )

        return abs(float(rho)), "Spearman"

    elif (not x_numeric) and (not y_numeric):

        return (
            cramers_v(x, y),
            "Cramer's V"
        )

    else:

        # Numeric variable must be passed as the
        # second argument to correlation_ratio.
        if x_numeric:
            value = x
            category = y
        else:
            value = y
            category = x

        return (
            correlation_ratio(
                category,
                value
            ),
            "Eta"
        )


# ============================================================
# Compute associations within every bucket
# ============================================================

bucket_ids = sorted(
    df[BUCKET_COLUMN].unique()
)

records = []

for bucket_number, bucket_id in enumerate(
    bucket_ids,
    start=1
):

    bucket = df[
        df[BUCKET_COLUMN] == bucket_id
    ].copy()

    print()
    print("=" * 70)
    print(
        f"BUCKET {bucket_number} "
        f"({len(bucket):,} rows)"
    )
    print("=" * 70)

    bucket_results = []

    for qid_a, qid_b in combinations(
        QIDS,
        2
    ):

        value, measure = association(
            bucket[qid_a],
            bucket[qid_b],
            qid_a,
            qid_b
        )

        record = {
            "bucket": bucket_number,
            "bucket_id": bucket_id,
            "qid_a": qid_a,
            "qid_b": qid_b,
            "measure": measure,
            "association": value,
        }

        records.append(record)
        bucket_results.append(record)

    bucket_results_df = pd.DataFrame(
        bucket_results
    ).sort_values(
        "association",
        ascending=False
    )

    print(
        bucket_results_df[
            [
                "qid_a",
                "qid_b",
                "measure",
                "association",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# Combined audit table
# ============================================================

corr_df = pd.DataFrame(
    records
)

print()
print("=" * 70)
print("CROSS-BUCKET CORRELATION SUMMARY")
print("=" * 70)

summary = (
    corr_df
    .groupby(
        [
            "qid_a",
            "qid_b",
            "measure",
        ],
        as_index=False
    )
    .agg(
        min_association=(
            "association",
            "min"
        ),
        mean_association=(
            "association",
            "mean"
        ),
        max_association=(
            "association",
            "max"
        ),
    )
    .sort_values(
        "max_association",
        ascending=False
    )
)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# Candidate associations that are >= 0.40
#
# This is ONLY an audit view.
# 0.40 is NOT being declared as the KC-Slice threshold here.
# ============================================================

candidate_threshold = 0.40

candidates = corr_df[
    corr_df["association"]
    >= candidate_threshold
].sort_values(
    [
        "bucket",
        "association",
    ],
    ascending=[
        True,
        False,
    ]
)

print()
print("=" * 70)
print(
    "ASSOCIATIONS >= 0.40 "
    "(AUDIT ONLY)"
)
print("=" * 70)

if len(candidates) == 0:

    print("None")

else:

    print(
        candidates[
            [
                "bucket",
                "qid_a",
                "qid_b",
                "measure",
                "association",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# Save
# ============================================================

RAW_OUTPUT = (
    OUT_DIR /
    "adult_1sa_qid_correlation_audit.csv"
)

SUMMARY_OUTPUT = (
    OUT_DIR /
    "adult_1sa_qid_correlation_summary.csv"
)

SPEC_OUTPUT = (
    OUT_DIR /
    "adult_1sa_qid_correlation_audit.txt"
)

corr_df.to_csv(
    RAW_OUTPUT,
    index=False
)

summary.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

with open(
    SPEC_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — QID CORRELATION AUDIT\n"
    )
    f.write("=" * 60 + "\n\n")

    f.write(
        "The KC-Slice source specifies a Correlate() "
        "operation but does not specify the statistical "
        "measure or threshold.\n\n"
    )

    f.write(
        "This audit therefore measures:\n"
    )

    f.write(
        "  Numeric-numeric      : absolute Spearman rho\n"
    )

    f.write(
        "  Categorical-categorical : Cramer's V\n"
    )

    f.write(
        "  Numeric-categorical  : Eta\n\n"
    )

    f.write(
        "The 0.40 display is an audit threshold only; "
        "it is not claimed to come from the KC-Slice paper.\n"
    )


print()
print(f"Saved raw audit     : {RAW_OUTPUT}")
print(f"Saved summary       : {SUMMARY_OUTPUT}")
print(f"Saved specification : {SPEC_OUTPUT}")

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA QID CORRELATION AUDIT COMPLETE")
print("=" * 70)

In [35]:
# Cell 35 — KC-SLICE ADULT 1-SA
# LOCK CORRELATE() + DETERMINISTIC QID GROUPING
# ============================================================

from pathlib import Path
from itertools import combinations

import pandas as pd

ROOT = Path("/content/kc-slice")

INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_qid_correlation_audit.csv"
)

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

corr_df = pd.read_csv(INPUT)

QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

CORRELATION_THRESHOLD = 0.40


print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("CORRELATE() + QID GROUPING SPECIFICATION")
print("=" * 70)

print()
print("Operational Correlate() rule:")
print(
    f"  Correlate(A,B) = 1 when association >= "
    f"{CORRELATION_THRESHOLD:.2f}"
)

print()
print("Association measures:")
print("  Numeric-numeric         : absolute Spearman rho")
print("  Categorical-categorical : Cramer's V")
print("  Numeric-categorical     : Eta")

print()
print("Important:")
print(
    "  The threshold and grouping strategy are project-defined "
    "operationalizations because the source does not specify "
    "the exact statistical threshold or overlap resolution."
)


# ============================================================
# Deterministic greedy grouping
#
# Algorithm:
#   1. Start with the first unused QID.
#   2. Compare it with later unused QIDs.
#   3. Concatenate every later QID with association >= threshold.
#   4. Mark those QIDs as consumed.
#   5. Continue with the next unused QID.
#
# This gives non-overlapping groups and is deterministic.
# ============================================================

bucket_ids = sorted(
    corr_df["bucket_id"].unique()
)

all_group_records = []

for bucket_id in bucket_ids:

    bucket_number = int(
        corr_df.loc[
            corr_df["bucket_id"] == bucket_id,
            "bucket"
        ].iloc[0]
    )

    bucket_corr = corr_df[
        corr_df["bucket_id"] == bucket_id
    ].copy()

    # Lookup table for pairwise association.
    assoc_lookup = {}

    for _, row in bucket_corr.iterrows():

        pair = frozenset(
            [
                row["qid_a"],
                row["qid_b"],
            ]
        )

        assoc_lookup[pair] = {
            "association": float(
                row["association"]
            ),
            "measure": row["measure"],
        }

    unused = list(QIDS)
    groups = []

    while unused:

        anchor = unused.pop(0)

        group = [anchor]

        remaining = []

        for candidate in unused:

            pair = frozenset(
                [
                    anchor,
                    candidate,
                ]
            )

            result = assoc_lookup.get(
                pair
            )

            if (
                result is not None
                and
                result["association"]
                >=
                CORRELATION_THRESHOLD
            ):

                group.append(
                    candidate
                )

            else:

                remaining.append(
                    candidate
                )

        unused = remaining

        groups.append(group)

    print()
    print("-" * 70)
    print(
        f"BUCKET {bucket_number}"
    )
    print("-" * 70)

    for group_id, group in enumerate(
        groups,
        start=1
    ):

        label = "__".join(group)

        print(
            f"Group {group_id}: "
            f"{label}"
        )

        # Save pair decisions relevant to this group.
        if len(group) > 1:

            for qid_a, qid_b in combinations(
                group,
                2
            ):

                pair = frozenset(
                    [
                        qid_a,
                        qid_b,
                    ]
                )

                result = assoc_lookup.get(
                    pair
                )

                # Not every pair inside a group is necessarily
                # above threshold; grouping is based on the
                # anchor QID's qualifying associations.
                if result is not None:

                    all_group_records.append({
                        "bucket": bucket_number,
                        "bucket_id": bucket_id,
                        "group_id": group_id,
                        "qid_a": qid_a,
                        "qid_b": qid_b,
                        "measure": result["measure"],
                        "association": result["association"],
                        "threshold": CORRELATION_THRESHOLD,
                    })

        else:

            all_group_records.append({
                "bucket": bucket_number,
                "bucket_id": bucket_id,
                "group_id": group_id,
                "qid_a": group[0],
                "qid_b": None,
                "measure": None,
                "association": None,
                "threshold": CORRELATION_THRESHOLD,
            })


# ============================================================
# Verify that every QID appears exactly once per bucket.
# ============================================================

for bucket_id in bucket_ids:

    bucket_number = int(
        corr_df.loc[
            corr_df["bucket_id"] == bucket_id,
            "bucket"
        ].iloc[0]
    )

    # Reconstruct the same grouping for verification.
    bucket_corr = corr_df[
        corr_df["bucket_id"] == bucket_id
    ]

    assoc_lookup = {}

    for _, row in bucket_corr.iterrows():

        assoc_lookup[
            frozenset(
                [
                    row["qid_a"],
                    row["qid_b"],
                ]
            )
        ] = float(
            row["association"]
        )

    unused = list(QIDS)
    seen = []

    while unused:

        anchor = unused.pop(0)

        seen.append(anchor)

        remaining = []

        for candidate in unused:

            pair = frozenset(
                [
                    anchor,
                    candidate,
                ]
            )

            if (
                pair in assoc_lookup
                and
                assoc_lookup[pair]
                >=
                CORRELATION_THRESHOLD
            ):

                seen.append(candidate)

            else:

                remaining.append(candidate)

        unused = remaining

    assert (
        len(seen) == len(QIDS)
    )

    assert (
        len(set(seen)) == len(QIDS)
    )


print()
print("=" * 70)
print("VERIFICATION")
print("=" * 70)

print(
    "Every QID assigned exactly once per bucket : PASS"
)

print(
    "Correlation threshold locked at            : "
    f"{CORRELATION_THRESHOLD:.2f}"
)

print(
    "Grouping is deterministic                  : PASS"
)


# ============================================================
# Save specification
# ============================================================

spec_path = (
    OUT_DIR /
    "adult_1sa_qid_correlation_specification.txt"
)

with open(
    spec_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — QID CORRELATION SPECIFICATION\n"
    )

    f.write("=" * 65 + "\n\n")

    f.write(
        "Correlate(A,B) = 1 iff association >= 0.40\n\n"
    )

    f.write(
        "Measures:\n"
    )
    f.write(
        "  Numeric-numeric         : absolute Spearman rho\n"
    )
    f.write(
        "  Categorical-categorical : Cramer's V\n"
    )
    f.write(
        "  Numeric-categorical     : Eta\n\n"
    )

    f.write(
        "Correlation is evaluated independently within each "
        "KC-Slice bucket.\n\n"
    )

    f.write(
        "QID grouping:\n"
    )

    f.write(
        "  Deterministic greedy, non-overlapping grouping.\n"
    )

    f.write(
        "  QIDs are processed in the predefined Adult QID order.\n"
    )

    f.write(
        "  For each unused anchor QID, later QIDs meeting the "
        "threshold are concatenated with it.\n\n"
    )

    f.write(
        "This threshold and grouping procedure are project-defined "
        "operationalizations because the source does not specify "
        "the exact Correlate() statistic, threshold, or overlap "
        "resolution.\n"
    )


# ============================================================
# Save detailed grouping information
# ============================================================

grouping_path = (
    OUT_DIR /
    "adult_1sa_qid_grouping_decisions.csv"
)

grouping_df = pd.DataFrame(
    all_group_records
)

grouping_df.to_csv(
    grouping_path,
    index=False
)

print()
print(f"Saved specification : {spec_path}")
print(f"Saved grouping log  : {grouping_path}")

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA QID GROUPING LOCKED")
print("=" * 70)

In [36]:
# Cell 36 — KC-SLICE ADULT 1-SA
# PHASE 3B: QID CONCATENATION + SID + PERMUTATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/kc-slice")

INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_bucket_privacy_output.csv"
)

SID_INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_sensitive_cells.csv"
)

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOCKED PARAMETERS
# ============================================================

RUN_SEED = 0

PROJECT_K = 10
CORRELATION_THRESHOLD = 0.40

QID_GROUPS = [
    ["age", "marital.status"],
    ["workclass"],
    ["education.num", "occupation"],
    ["race", "native.country"],
    ["sex"],
]

GROUP_NAMES = [
    "slice_1",
    "slice_2",
    "slice_3",
    "slice_4",
    "slice_5",
]

BUCKET_COLUMN = "_kc_bucket"


# ============================================================
# LOAD INPUTS
# ============================================================

df = pd.read_csv(INPUT)

sid_table = pd.read_csv(
    SID_INPUT
)

print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("PHASE 3B: QID CONCATENATION + SID + PERMUTATION")
print("=" * 70)

print(f"Input rows             : {len(df):,}")
print(f"QID groups             : {len(QID_GROUPS)}")
print(f"Correlation threshold  : {CORRELATION_THRESHOLD:.2f}")
print(f"Project k              : {PROJECT_K}")
print(f"Run seed               : {RUN_SEED}")


# ============================================================
# VERIFY LOCKED GROUPING
# ============================================================

flat_qids = [
    qid
    for group in QID_GROUPS
    for qid in group
]

EXPECTED_QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

# IMPORTANT:
# The grouping order is allowed to differ from the original
# dataset QID order. We only require that the same QIDs are
# present exactly once.

assert len(flat_qids) == len(EXPECTED_QIDS)

assert set(flat_qids) == set(EXPECTED_QIDS)

assert len(set(flat_qids)) == len(EXPECTED_QIDS)


print()
print("Locked QID grouping:")

for group_name, group in zip(
    GROUP_NAMES,
    QID_GROUPS
):
    print(
        f"  {group_name}: "
        + " + ".join(group)
    )

print()
print(
    "QID coverage verification : PASS"
)

print(
    "QID uniqueness verification : PASS"
)


# ============================================================
# GET BUCKET -> SID MAPPING
# ============================================================

bucket_ids = sorted(
    df[BUCKET_COLUMN].unique()
)

bucket_sid = {}

for bucket_id in bucket_ids:

    matches = sid_table[
        sid_table["bucket_id"] == bucket_id
    ]

    # One income SID per bucket.
    assert len(matches) == 1

    bucket_sid[bucket_id] = str(
        matches.iloc[0]["SID"]
    )


print()
print("Bucket -> SID mapping:")

for bucket_id in bucket_ids:

    print(
        f"  Bucket {int(bucket_id) + 1} "
        f"-> SID {bucket_sid[bucket_id]}"
    )


# ============================================================
# BUILD SLICED QID TABLES
# ============================================================

all_sliced_rows = []

bucket_permutation_records = []

for bucket_number, bucket_id in enumerate(
    bucket_ids,
    start=1
):

    bucket = df[
        df[BUCKET_COLUMN] == bucket_id
    ].copy()

    sid = bucket_sid[bucket_id]

    print()
    print("=" * 70)
    print(
        f"BUCKET {bucket_number} "
        f"({len(bucket):,} rows)"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Build unpermuted QID slices.
    # --------------------------------------------------------

    qid_output = pd.DataFrame(
        index=bucket.index
    )

    qid_output[
        "_source_row"
    ] = bucket.index

    qid_output[
        "bucket"
    ] = bucket_number

    qid_output[
        "bucket_id"
    ] = bucket_id

    for group_name, group in zip(
        GROUP_NAMES,
        QID_GROUPS
    ):

        values = []

        for row_index in bucket.index:

            components = [
                str(
                    bucket.loc[
                        row_index,
                        qid
                    ]
                )
                for qid in group
            ]

            # Attach the sensitive-cell SID.
            components.append(
                f"SID={sid}"
            )

            values.append(
                " | ".join(
                    components
                )
            )

        qid_output[
            group_name
        ] = values


    # --------------------------------------------------------
    # Record accounting.
    # --------------------------------------------------------

    assert len(qid_output) == len(bucket)


    # --------------------------------------------------------
    # Randomly permute the QID slice columns.
    #
    # We use a fixed seed for reproducibility.
    # Different buckets receive deterministic derived seeds.
    # --------------------------------------------------------

    rng = np.random.default_rng(
        RUN_SEED + bucket_number
    )

    permutation = rng.permutation(
        len(GROUP_NAMES)
    )

    original_group_names = list(
        GROUP_NAMES
    )

    permuted_group_names = [
        original_group_names[i]
        for i in permutation
    ]

    print()
    print(
        "Original QID slice order:"
    )

    print(
        "  "
        +
        " -> ".join(
            original_group_names
        )
    )

    print(
        "Permuted QID slice order:"
    )

    print(
        "  "
        +
        " -> ".join(
            permuted_group_names
        )
    )


    # --------------------------------------------------------
    # Save permutation mapping.
    # --------------------------------------------------------

    for position, original_index in enumerate(
        permutation,
        start=1
    ):

        bucket_permutation_records.append({
            "bucket": bucket_number,
            "bucket_id": bucket_id,
            "output_position": position,
            "original_group": original_group_names[
                original_index
            ],
        })


    # --------------------------------------------------------
    # Construct final bucket QID table.
    # --------------------------------------------------------

    final_columns = [
        "_source_row",
        "bucket",
        "bucket_id",
    ] + permuted_group_names

    final_qid = qid_output[
        final_columns
    ].copy()

    all_sliced_rows.append(
        final_qid
    )


    # --------------------------------------------------------
    # Display sample.
    # --------------------------------------------------------

    print()
    print("Final QID table sample:")

    print(
        final_qid.head(5).to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Save bucket-specific QID table.
    # --------------------------------------------------------

    bucket_output = (
        OUT_DIR /
        f"adult_1sa_qid_bucket_{bucket_number}.csv"
    )

    final_qid.to_csv(
        bucket_output,
        index=False
    )

    print()
    print(
        f"Saved: {bucket_output}"
    )


# ============================================================
# COMBINE ALL BUCKETS
# ============================================================

final_qid_table = pd.concat(
    all_sliced_rows,
    ignore_index=True
)

assert len(final_qid_table) == len(df)


# ============================================================
# FINAL VERIFICATION
# ============================================================

print()
print("=" * 70)
print("FINAL QID TABLE VERIFICATION")
print("=" * 70)

print(
    f"Input rows                  : "
    f"{len(df):,}"
)

print(
    f"Final QID rows              : "
    f"{len(final_qid_table):,}"
)

assert (
    len(final_qid_table)
    ==
    len(df)
)

print(
    "Record accounting           : PASS"
)


# ------------------------------------------------------------
# Every source row must occur exactly once.
# ------------------------------------------------------------

source_counts = (
    final_qid_table[
        "_source_row"
    ]
    .value_counts()
)

assert (
    source_counts == 1
).all()

print(
    "One output row per source row: PASS"
)


# ------------------------------------------------------------
# Bucket sizes must remain unchanged.
# ------------------------------------------------------------

bucket_counts = (
    final_qid_table[
        "bucket_id"
    ]
    .value_counts()
    .sort_index()
)

print()
print("Final bucket sizes:")

for bucket_id, count in (
    bucket_counts.items()
):

    print(
        f"  Bucket {int(bucket_id) + 1}: "
        f"{count:,}"
    )

assert all(
    count == 15081
    for count in bucket_counts
)

print(
    "Bucket-size preservation       : PASS"
)


# ------------------------------------------------------------
# Every QID slice must contain the SID.
# ------------------------------------------------------------

for group_name in GROUP_NAMES:

    assert (
        final_qid_table[
            group_name
        ]
        .astype(str)
        .str.contains(
            "SID=",
            regex=False
        )
        .all()
    )

print(
    "SID linkage in every QID slice : PASS"
)


# ============================================================
# SAVE COMBINED QID TABLE
# ============================================================

FINAL_QID_OUTPUT = (
    OUT_DIR /
    "adult_1sa_qid_sliced_permuted.csv"
)

final_qid_table.to_csv(
    FINAL_QID_OUTPUT,
    index=False
)


# ============================================================
# SAVE PERMUTATION LOG
# ============================================================

permutation_df = pd.DataFrame(
    bucket_permutation_records
)

PERM_OUTPUT = (
    OUT_DIR /
    "adult_1sa_qid_column_permutation.csv"
)

permutation_df.to_csv(
    PERM_OUTPUT,
    index=False
)


# ============================================================
# SAVE Bsa TABLE
# ============================================================

BSA_OUTPUT = (
    OUT_DIR /
    "adult_1sa_Bsa_sensitive_table.csv"
)

sid_table.to_csv(
    BSA_OUTPUT,
    index=False
)


# ============================================================
# SAVE SPECIFICATION
# ============================================================

SPEC_OUTPUT = (
    OUT_DIR /
    "adult_1sa_phase3b_specification.txt"
)

with open(
    SPEC_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — PHASE 3B SPECIFICATION\n"
    )

    f.write("=" * 65 + "\n\n")

    f.write(
        "QID groups:\n"
    )

    for name, group in zip(
        GROUP_NAMES,
        QID_GROUPS
    ):

        f.write(
            f"  {name}: "
            + " + ".join(group)
            + "\n"
        )

    f.write("\n")

    f.write(
        "Correlation threshold: 0.40\n"
    )

    f.write(
        "Run seed: 0\n\n"
    )

    f.write(
        "Each correlated QID group is concatenated into one "
        "QID slice and linked with the bucket's sensitive-cell "
        "SID.\n\n"
    )

    f.write(
        "Only QID slice columns are randomly permuted. "
        "Bucket/source-row accounting columns remain fixed "
        "for reproducibility and evaluation.\n\n"
    )

    f.write(
        "The source specifies random permutation but does not "
        "specify a random seed; seed 0 is therefore an explicit "
        "project reproducibility decision.\n"
    )


# ============================================================
# OUTPUT SUMMARY
# ============================================================

print()
print("=" * 70)
print("SAVED OUTPUTS")
print("=" * 70)

print(
    f"Bsa sensitive table : {BSA_OUTPUT}"
)

print(
    f"Combined Bqa table  : {FINAL_QID_OUTPUT}"
)

print(
    f"Permutation log      : {PERM_OUTPUT}"
)

print(
    f"Specification        : {SPEC_OUTPUT}"
)

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA PHASE 3B COMPLETE")
print("=" * 70)

In [37]:
# Cell 37 — KC-SLICE ADULT 1-SA
# SID FORMAT QA + FINAL PACKAGE
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

SID_INPUT = (
    OUT_DIR /
    "adult_1sa_sensitive_cells.csv"
)

QID_INPUT = (
    OUT_DIR /
    "adult_1sa_qid_sliced_permuted.csv"
)

PERM_INPUT = (
    OUT_DIR /
    "adult_1sa_qid_column_permutation.csv"
)

# ============================================================
# Read SID table explicitly as string
# ============================================================

sid_table = pd.read_csv(
    SID_INPUT,
    dtype={"SID": str}
)

qid_table = pd.read_csv(
    QID_INPUT,
    dtype={
        "_source_row": int,
        "bucket": int,
        "bucket_id": int,
    }
)

print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("FINAL SID FORMAT + PACKAGE QA")
print("=" * 70)


# ============================================================
# Restore fixed-width SID representation
# ============================================================

sid_table["SID"] = (
    sid_table["SID"]
    .astype(str)
    .str.zfill(3)
)

print()
print("Sensitive-cell SIDs:")

print(
    sid_table[
        ["bucket", "SA", "SID"]
    ].to_string(index=False)
)


# ============================================================
# Rebuild QID SID strings with zero-padded SIDs
# ============================================================

for bucket_id in sorted(
    qid_table["bucket_id"].unique()
):

    matches = sid_table[
        sid_table["bucket_id"] == bucket_id
    ]

    assert len(matches) == 1

    sid = matches.iloc[0]["SID"]

    bucket_mask = (
        qid_table["bucket_id"]
        ==
        bucket_id
    )

    # Replace any SID=11 / SID=21 etc. with fixed-width SID.
    for column in [
        "slice_1",
        "slice_2",
        "slice_3",
        "slice_4",
        "slice_5",
    ]:

        qid_table.loc[
            bucket_mask,
            column
        ] = (
            qid_table.loc[
                bucket_mask,
                column
            ]
            .astype(str)
            .str.replace(
                r"SID=\d+",
                f"SID={sid}",
                regex=True
            )
        )


# ============================================================
# Verify exact SID format
# ============================================================

expected_sids = {
    0: "011",
    1: "021",
}

for bucket_id, expected_sid in (
    expected_sids.items()
):

    actual = sid_table.loc[
        sid_table["bucket_id"] == bucket_id,
        "SID"
    ].iloc[0]

    assert actual == expected_sid


print()
print(
    "Fixed-width SID format      : PASS"
)


# ============================================================
# Verify SID linkage
# ============================================================

slice_columns = [
    "slice_1",
    "slice_2",
    "slice_3",
    "slice_4",
    "slice_5",
]

for bucket_id in sorted(
    qid_table["bucket_id"].unique()
):

    expected_sid = sid_table.loc[
        sid_table["bucket_id"] == bucket_id,
        "SID"
    ].iloc[0]

    bucket_rows = qid_table[
        qid_table["bucket_id"] == bucket_id
    ]

    for column in slice_columns:

        assert (
            bucket_rows[column]
            .astype(str)
            .str.contains(
                f"SID={expected_sid}",
                regex=False
            )
            .all()
        )


print(
    "SID linkage verification     : PASS"
)


# ============================================================
# Record accounting
# ============================================================

assert len(qid_table) == 30162

assert (
    qid_table["_source_row"]
    .nunique()
    ==
    30162
)

print(
    "Final row accounting         : PASS"
)


# ============================================================
# Bucket accounting
# ============================================================

bucket_sizes = (
    qid_table
    .groupby("bucket_id")
    .size()
)

assert (
    bucket_sizes.to_dict()
    ==
    {
        0: 15081,
        1: 15081,
    }
)

print(
    "Final bucket accounting      : PASS"
)


# ============================================================
# Save corrected files
# ============================================================

sid_table.to_csv(
    SID_INPUT,
    index=False
)

qid_table.to_csv(
    QID_INPUT,
    index=False
)


# ============================================================
# Create final package manifest
# ============================================================

MANIFEST = (
    OUT_DIR /
    "adult_1sa_final_package.txt"
)

with open(
    MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — FINAL PACKAGE\n"
    )
    f.write("=" * 65 + "\n\n")

    f.write(
        "Dataset records: 30162\n"
    )

    f.write(
        "Sensitive attribute: income\n"
    )

    f.write(
        "Project k: 10\n"
    )

    f.write(
        "KC-Slice internal bucket size: 15081\n"
    )

    f.write(
        "Privacy threshold C: 5%\n"
    )

    f.write(
        "HSA: <=50K\n"
    )

    f.write(
        "QID correlation threshold: 0.40\n"
    )

    f.write(
        "QID groups:\n"
    )
    f.write(
        "  slice_1 = age + marital.status\n"
    )
    f.write(
        "  slice_2 = workclass\n"
    )
    f.write(
        "  slice_3 = education.num + occupation\n"
    )
    f.write(
        "  slice_4 = race + native.country\n"
    )
    f.write(
        "  slice_5 = sex\n"
    )

    f.write(
        "\nRun seed: 0\n"
    )

    f.write(
        "\nSID values:\n"
    )
    f.write(
        "  Bucket 1 = 011\n"
    )
    f.write(
        "  Bucket 2 = 021\n"
    )

    f.write(
        "\nStatus:\n"
    )
    f.write(
        "  Phase 1 bucket creation: PASS\n"
    )
    f.write(
        "  Phase 2 privacy enforcement: PASS\n"
    )
    f.write(
        "  Phase 3A sensitive cells + SID: PASS\n"
    )
    f.write(
        "  Phase 3B QID concatenation + permutation: PASS\n"
    )
    f.write(
        "  Final SID format QA: PASS\n"
    )


print()
print("Final 1-SA package files:")
print(
    f"  Bsa: {SID_INPUT}"
)
print(
    f"  Bqa: {QID_INPUT}"
)
print(
    f"  Manifest: {MANIFEST}"
)

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA FINAL PACKAGE QA COMPLETE")
print("=" * 70)

In [38]:
# Cell 38 — KC-SLICE ADULT 1-SA
# PUBLIC RELEASE / EVALUATION PACKAGE SEPARATION
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

BASELINE_INPUT = (
    ROOT /
    "data/processed/"
    "adult_standard_baseline_k10_1sa.csv"
)

BQA_INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_qid_sliced_permuted.csv"
)

BSA_INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_sensitive_cells.csv"
)

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Load source/evaluation data
# ============================================================

baseline = pd.read_csv(
    BASELINE_INPUT
)

bqa_internal = pd.read_csv(
    BQA_INPUT
)

bsa = pd.read_csv(
    BSA_INPUT,
    dtype={"SID": str}
)

bsa["SID"] = (
    bsa["SID"]
    .astype(str)
    .str.zfill(3)
)


print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("PUBLIC RELEASE / EVALUATION PACKAGE")
print("=" * 70)

print(
    f"Original baseline rows : {len(baseline):,}"
)

print(
    f"Internal Bqa rows      : {len(bqa_internal):,}"
)

print(
    f"Bsa rows               : {len(bsa):,}"
)


# ============================================================
# Identify QID slice columns from the locked implementation.
# ============================================================

SLICE_COLUMNS = [
    "slice_1",
    "slice_2",
    "slice_3",
    "slice_4",
    "slice_5",
]

for column in SLICE_COLUMNS:

    assert column in bqa_internal.columns


# ============================================================
# 1. PUBLIC Bqa
#
# Only the actual QID slices are exposed.
#
# Internal:
#   _source_row
#   bucket
#   bucket_id
#
# are removed.
# ============================================================

bqa_public = bqa_internal[
    SLICE_COLUMNS
].copy()

assert len(bqa_public) == len(
    baseline
)


# ============================================================
# 2. PUBLIC Bsa
#
# Keep only the sensitive-cell publication representation.
# ============================================================

BSA_PUBLIC_COLUMNS = [
    "bucket",
    "SID",
    "SA",
    "cell",
]

for column in BSA_PUBLIC_COLUMNS:

    assert column in bsa.columns

bsa_public = bsa[
    BSA_PUBLIC_COLUMNS
].copy()


# ============================================================
# 3. PRIVATE EVALUATION DATASET
#
# We retain source_row ONLY internally so that the original
# target can be aligned with the anonymized QID representation.
#
# The target is not included in the published Bqa.
# ============================================================

evaluation = bqa_internal[
    [
        "_source_row",
        "bucket",
        "bucket_id",
    ] + SLICE_COLUMNS
].copy()

# Original income target.
#
# baseline columns:
#   QIDs + income
#
assert "income" in baseline.columns

target_lookup = baseline[
    ["income"]
].copy()

target_lookup[
    "_source_row"
] = target_lookup.index

target_lookup = target_lookup[
    [
        "_source_row",
        "income",
    ]
]

evaluation = evaluation.merge(
    target_lookup,
    on="_source_row",
    how="left",
    validate="one_to_one"
)


# ============================================================
# Evaluation integrity
# ============================================================

print()
print("-" * 70)
print("EVALUATION INTEGRITY")
print("-" * 70)

assert len(evaluation) == len(
    baseline
)

print(
    "Evaluation row count       : PASS"
)


assert (
    evaluation["_source_row"]
    .nunique()
    ==
    len(baseline)
)

print(
    "Unique source-row mapping  : PASS"
)


assert (
    evaluation["income"]
    .notna()
    .all()
)

print(
    "Ground-truth target linkage: PASS"
)


# Target distribution should remain exactly equal to the
# original dataset because anonymization does not alter the
# private evaluation labels.

original_target_counts = (
    baseline["income"]
    .value_counts()
    .sort_index()
)

evaluation_target_counts = (
    evaluation["income"]
    .value_counts()
    .sort_index()
)

assert (
    original_target_counts.equals(
        evaluation_target_counts
    )
)

print(
    "Target distribution         : PASS"
)


# ============================================================
# Verify published Bqa contains no internal metadata.
# ============================================================

assert "_source_row" not in bqa_public.columns
assert "bucket" not in bqa_public.columns
assert "bucket_id" not in bqa_public.columns

print(
    "Internal metadata removed from Bqa: PASS"
)


# ============================================================
# Verify Bsa SID format.
# ============================================================

assert (
    bsa_public["SID"]
    .astype(str)
    .str.fullmatch(r"\d{3}")
    .all()
)

print(
    "Public Bsa SID format       : PASS"
)


# ============================================================
# Verify public row accounting.
# ============================================================

assert len(bqa_public) == 30162

print(
    "Public Bqa record count     : PASS"
)


# ============================================================
# Save PUBLIC RELEASE artifacts
# ============================================================

BQA_PUBLIC_OUTPUT = (
    OUT_DIR /
    "adult_1sa_Bqa_public.csv"
)

BSA_PUBLIC_OUTPUT = (
    OUT_DIR /
    "adult_1sa_Bsa_public.csv"
)

bqa_public.to_csv(
    BQA_PUBLIC_OUTPUT,
    index=False
)

bsa_public.to_csv(
    BSA_PUBLIC_OUTPUT,
    index=False
)


# ============================================================
# Save PRIVATE EVALUATION artifact
# ============================================================

EVAL_OUTPUT = (
    OUT_DIR /
    "adult_1sa_evaluation_dataset.csv"
)

evaluation.to_csv(
    EVAL_OUTPUT,
    index=False
)


# ============================================================
# Save package specification
# ============================================================

SPEC_OUTPUT = (
    OUT_DIR /
    "adult_1sa_public_evaluation_specification.txt"
)

with open(
    SPEC_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — PUBLIC / EVALUATION SPECIFICATION\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write(
        "PUBLIC Bqa:\n"
    )
    f.write(
        "  Contains only the five sliced QID columns.\n"
    )
    f.write(
        "  Internal row/bucket bookkeeping is removed.\n\n"
    )

    f.write(
        "PUBLIC Bsa:\n"
    )
    f.write(
        "  Contains bucket, SID, SA, and grouped cell representation.\n\n"
    )

    f.write(
        "PRIVATE EVALUATION DATASET:\n"
    )
    f.write(
        "  Contains anonymized QID slices plus the original income "
        "target for downstream utility evaluation.\n"
    )
    f.write(
        "  Source-row identifiers are retained only as internal "
        "evaluation join keys.\n\n"
    )

    f.write(
        "This prevents internal bookkeeping and private ground-truth "
        "labels from being confused with the KC-Slice published release.\n"
    )


# ============================================================
# Final summary
# ============================================================

print()
print("=" * 70)
print("PACKAGE CREATED")
print("=" * 70)

print(
    f"Public Bqa : {BQA_PUBLIC_OUTPUT}"
)

print(
    f"Public Bsa : {BSA_PUBLIC_OUTPUT}"
)

print(
    f"Evaluation : {EVAL_OUTPUT}"
)

print(
    f"Specification: {SPEC_OUTPUT}"
)

print()
print("=" * 70)
print("KC-SLICE ADULT 1-SA PUBLIC/EVALUATION PACKAGE COMPLETE")
print("=" * 70)

In [39]:
# Cell 39 — KC-SLICE ADULT 1-SA
# QUANTITATIVE BASELINE EVALUATION
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)

ROOT = Path("/content/kc-slice")

BASELINE_INPUT = (
    ROOT /
    "data/processed/"
    "adult_standard_baseline_k10_1sa.csv"
)

EVAL_INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_evaluation_dataset.csv"
)

PRIVACY_INPUT = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_privacy_report.csv"
)

OUT_DIR = (
    ROOT /
    "results/kc_slice_adult_1sa"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Experimental parameters
# ============================================================

TEST_SIZE = 0.20
SPLIT_SEED = 42

TARGET = "income"

ORIGINAL_QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

KC_FEATURES = [
    "slice_1",
    "slice_2",
    "slice_3",
    "slice_4",
    "slice_5",
]

PRIVACY_C = 5.0


print("=" * 70)
print("KC-SLICE — ADULT 1-SA")
print("QUANTITATIVE BASELINE EVALUATION")
print("=" * 70)


# ============================================================
# Load data
# ============================================================

baseline = pd.read_csv(
    BASELINE_INPUT
)

evaluation = pd.read_csv(
    EVAL_INPUT
)

privacy = pd.read_csv(
    PRIVACY_INPUT
)


assert len(baseline) == 30162
assert len(evaluation) == 30162

print()
print(f"Baseline rows       : {len(baseline):,}")
print(f"KC-Slice eval rows  : {len(evaluation):,}")


# ============================================================
# PART 1 — PRIVACY / DISTORTION
# ============================================================

print()
print("-" * 70)
print("1. PRIVACY / DISTORTION")
print("-" * 70)


# ------------------------------------------------------------
# Suppression statistics
# ------------------------------------------------------------

total_suppressed = int(
    privacy["suppressed_count"].sum()
)

total_records = len(
    baseline
)

suppression_rate = (
    total_suppressed /
    total_records *
    100
)


# ------------------------------------------------------------
# Minimal Distortion
#
# From the KC-Slice paper:
# distortion count increments with each generalization
# or suppression operation.
#
# Our 1-SA implementation performs only HSA suppression,
# one operation per suppressed HSA record.
# ------------------------------------------------------------

minimal_distortion = (
    total_suppressed
)


# ------------------------------------------------------------
# Loss Metric
#
# Paper:
#
# LM = (M - 1) / (|A| - 1)
#
# For binary income:
#
# |A| = 2
#
# A suppressed value "#####" can represent either value,
# therefore M = 2.
#
# Suppressed record loss = 1
# Unsuppressed record loss = 0
#
# Thus average LM = suppression rate.
# ------------------------------------------------------------

income_domain_size = 2
suppressed_ambiguity = 2

suppressed_lm = (
    (suppressed_ambiguity - 1) /
    (income_domain_size - 1)
)

assert suppressed_lm == 1.0

loss_metric = (
    total_suppressed *
    suppressed_lm /
    total_records
)


print(
    f"Total HSA suppressions : "
    f"{total_suppressed:,}"
)

print(
    f"Suppression rate       : "
    f"{suppression_rate:.4f}%"
)

print(
    f"Minimal Distortion     : "
    f"{minimal_distortion:,}"
)

print(
    f"Loss Metric (income)   : "
    f"{loss_metric:.6f}"
)


# ------------------------------------------------------------
# Bucket privacy verification
# ------------------------------------------------------------

max_hsa_percent = (
    privacy["after_percent"]
    .max()
)

min_hsa_percent = (
    privacy["after_percent"]
    .min()
)

assert (
    max_hsa_percent
    <=
    PRIVACY_C + 1e-9
)


print(
    f"Maximum post-check HSA %: "
    f"{max_hsa_percent:.6f}%"
)

print(
    f"Minimum post-check HSA %: "
    f"{min_hsa_percent:.6f}%"
)

print(
    "Privacy threshold verification: PASS"
)


# ============================================================
# PART 2 — FIXED TRAIN/TEST SPLIT
# ============================================================

print()
print("-" * 70)
print("2. FIXED TRAIN/TEST SPLIT")
print("-" * 70)


# Use original baseline row numbers as the permanent split key.
all_indices = np.arange(
    len(baseline)
)

train_indices, test_indices = (
    train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=SPLIT_SEED,
        stratify=baseline[TARGET],
    )
)

train_indices = np.sort(
    train_indices
)

test_indices = np.sort(
    test_indices
)


print(
    f"Train rows : {len(train_indices):,}"
)

print(
    f"Test rows  : {len(test_indices):,}"
)

print(
    f"Split      : {100*(1-TEST_SIZE):.0f}/"
    f"{100*TEST_SIZE:.0f}"
)

print(
    f"Random seed: {SPLIT_SEED}"
)


# Verify complete partition.

assert len(
    np.intersect1d(
        train_indices,
        test_indices
    )
) == 0

assert (
    len(train_indices)
    +
    len(test_indices)
    ==
    len(baseline)
)

print(
    "Train/test partition: PASS"
)


# Save split manifest so all later methods use exactly this split.

split_manifest = {
    "dataset": "Adult",
    "records": len(baseline),
    "test_size": TEST_SIZE,
    "random_state": SPLIT_SEED,
    "train_rows": len(train_indices),
    "test_rows": len(test_indices),
}

with open(
    OUT_DIR /
    "adult_fixed_train_test_split.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        split_manifest,
        f,
        indent=2
    )

np.save(
    OUT_DIR /
    "adult_train_indices.npy",
    train_indices
)

np.save(
    OUT_DIR /
    "adult_test_indices.npy",
    test_indices
)


# ============================================================
# PART 3 — ORIGINAL-DATA LOGISTIC REGRESSION
# ============================================================

print()
print("-" * 70)
print("3. ORIGINAL-DATA LOGISTIC REGRESSION")
print("-" * 70)


X_original = baseline[
    ORIGINAL_QIDS
].copy()

y_original = baseline[
    TARGET
].copy()


X_train_original = X_original.iloc[
    train_indices
]

X_test_original = X_original.iloc[
    test_indices
]

y_train = y_original.iloc[
    train_indices
]

y_test = y_original.iloc[
    test_indices
]


numeric_features = [
    "age",
    "education.num",
]

categorical_features = [
    "workclass",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]


original_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "num",
                "passthrough",
                numeric_features
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features
            ),
        ]
    )
)


original_model = Pipeline([
    (
        "preprocess",
        original_preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    ),
])


original_model.fit(
    X_train_original,
    y_train
)

original_predictions = (
    original_model.predict(
        X_test_original
    )
)

original_probabilities = (
    original_model.predict_proba(
        X_test_original
    )[:, 1]
)


original_accuracy = accuracy_score(
    y_test,
    original_predictions
)

original_macro_f1 = f1_score(
    y_test,
    original_predictions,
    average="macro"
)

original_auc = roc_auc_score(
    y_test,
    original_probabilities
)


print(
    f"Accuracy : {original_accuracy:.6f}"
)

print(
    f"Macro-F1 : {original_macro_f1:.6f}"
)

print(
    f"ROC-AUC  : {original_auc:.6f}"
)


# ============================================================
# PART 4 — KC-SLICE LOGISTIC REGRESSION
# ============================================================

print()
print("-" * 70)
print("4. KC-SLICE LOGISTIC REGRESSION")
print("-" * 70)


# ------------------------------------------------------------
# Important:
#
# KC-Slice publishes five concatenated QID slices.
# The public table does not expose the original QID labels
# after permutation.
#
# Therefore each slice is treated as a categorical released
# feature exactly as it appears in Bqa.
# ------------------------------------------------------------

X_kc = evaluation[
    KC_FEATURES
].copy()

y_kc = evaluation[
    TARGET
].copy()


X_train_kc = X_kc.iloc[
    train_indices
]

X_test_kc = X_kc.iloc[
    test_indices
]

y_train_kc = y_kc.iloc[
    train_indices
]

y_test_kc = y_kc.iloc[
    test_indices
]


kc_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "slices",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                KC_FEATURES
            ),
        ]
    )
)


kc_model = Pipeline([
    (
        "preprocess",
        kc_preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    ),
])


kc_model.fit(
    X_train_kc,
    y_train_kc
)

kc_predictions = (
    kc_model.predict(
        X_test_kc
    )
)

kc_probabilities = (
    kc_model.predict_proba(
        X_test_kc
    )[:, 1]
)


kc_accuracy = accuracy_score(
    y_test_kc,
    kc_predictions
)

kc_macro_f1 = f1_score(
    y_test_kc,
    kc_predictions,
    average="macro"
)

kc_auc = roc_auc_score(
    y_test_kc,
    kc_probabilities
)


print(
    f"Accuracy : {kc_accuracy:.6f}"
)

print(
    f"Macro-F1 : {kc_macro_f1:.6f}"
)

print(
    f"ROC-AUC  : {kc_auc:.6f}"
)


# ============================================================
# PART 5 — UTILITY DIFFERENCE
# ============================================================

print()
print("-" * 70)
print("5. UTILITY COMPARISON")
print("-" * 70)


accuracy_delta = (
    kc_accuracy
    -
    original_accuracy
)

macro_f1_delta = (
    kc_macro_f1
    -
    original_macro_f1
)

auc_delta = (
    kc_auc
    -
    original_auc
)


print(
    f"Accuracy delta : "
    f"{accuracy_delta:+.6f}"
)

print(
    f"Macro-F1 delta : "
    f"{macro_f1_delta:+.6f}"
)

print(
    f"ROC-AUC delta  : "
    f"{auc_delta:+.6f}"
)


# ============================================================
# PART 6 — SAVE RESULTS
# ============================================================

results = pd.DataFrame([
    {
        "dataset": "Adult",
        "method": "Original",
        "sa_count": 1,
        "project_k": 10,
        "kc_bucket_size": 15081,
        "C": 5.0,
        "suppression_rate": 0.0,
        "minimal_distortion": 0,
        "loss_metric_income": 0.0,
        "accuracy": original_accuracy,
        "macro_f1": original_macro_f1,
        "roc_auc": original_auc,
        "split_seed": SPLIT_SEED,
    },
    {
        "dataset": "Adult",
        "method": "KC-Slice",
        "sa_count": 1,
        "project_k": 10,
        "kc_bucket_size": 15081,
        "C": 5.0,
        "suppression_rate": suppression_rate,
        "minimal_distortion": minimal_distortion,
        "loss_metric_income": loss_metric,
        "accuracy": kc_accuracy,
        "macro_f1": kc_macro_f1,
        "roc_auc": kc_auc,
        "split_seed": SPLIT_SEED,
    },
])


RESULT_OUTPUT = (
    OUT_DIR /
    "adult_1sa_quantitative_evaluation.csv"
)

results.to_csv(
    RESULT_OUTPUT,
    index=False
)


# Save readable summary.

SUMMARY_OUTPUT = (
    OUT_DIR /
    "adult_1sa_quantitative_evaluation.txt"
)

with open(
    SUMMARY_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "KC-SLICE ADULT 1-SA — QUANTITATIVE EVALUATION\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write(
        "Privacy / distortion:\n"
    )

    f.write(
        f"  HSA suppressions : {total_suppressed:,}\n"
    )

    f.write(
        f"  Suppression rate : {suppression_rate:.6f}%\n"
    )

    f.write(
        f"  Minimal distortion: {minimal_distortion:,}\n"
    )

    f.write(
        f"  Loss metric      : {loss_metric:.6f}\n\n"
    )

    f.write(
        "Fixed split:\n"
    )

    f.write(
        f"  Test fraction    : {TEST_SIZE:.2f}\n"
    )

    f.write(
        f"  Random state     : {SPLIT_SEED}\n\n"
    )

    f.write(
        "Original logistic regression:\n"
    )

    f.write(
        f"  Accuracy         : {original_accuracy:.6f}\n"
    )

    f.write(
        f"  Macro-F1         : {original_macro_f1:.6f}\n"
    )

    f.write(
        f"  ROC-AUC          : {original_auc:.6f}\n\n"
    )

    f.write(
        "KC-Slice logistic regression:\n"
    )

    f.write(
        f"  Accuracy         : {kc_accuracy:.6f}\n"
    )

    f.write(
        f"  Macro-F1         : {kc_macro_f1:.6f}\n"
    )

    f.write(
        f"  ROC-AUC          : {kc_auc:.6f}\n\n"
    )

    f.write(
        "KC-Slice minus Original:\n"
    )

    f.write(
        f"  Accuracy delta   : {accuracy_delta:+.6f}\n"
    )

    f.write(
        f"  Macro-F1 delta   : {macro_f1_delta:+.6f}\n"
    )

    f.write(
        f"  ROC-AUC delta    : {auc_delta:+.6f}\n"
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 70)
print("ADULT 1-SA QUANTITATIVE EVALUATION COMPLETE")
print("=" * 70)

print(
    f"Results : {RESULT_OUTPUT}"
)

print(
    f"Summary : {SUMMARY_OUTPUT}"
)

print()
print("=" * 70)

In [40]:
# Cell 40 — ADULT 1-SA
# CROSS-METHOD OUTPUT AUDIT
#
# Purpose:
#   Locate Mondrian / ARX / KC-Slice output CSVs and inspect
#   whether they contain the expected Adult QID columns.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

RESULTS_DIR = (
    ROOT /
    "results"
)

EXPECTED_QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

TARGET = "income"


print("=" * 70)
print("ADULT 1-SA — CROSS-METHOD OUTPUT AUDIT")
print("=" * 70)


# ============================================================
# Search every CSV under results/
# ============================================================

csv_files = sorted(
    RESULTS_DIR.rglob("*.csv")
)

print()
print(
    f"CSV files found under results/: "
    f"{len(csv_files)}"
)


# ============================================================
# Inspect candidate datasets.
#
# We specifically look for files containing many/all of the
# Adult QID columns.
# ============================================================

candidates = []

for path in csv_files:

    try:

        # Read only header first.
        header = pd.read_csv(
            path,
            nrows=0
        )

        columns = list(
            header.columns
        )

        qid_overlap = [
            qid
            for qid in EXPECTED_QIDS
            if qid in columns
        ]

        has_target = (
            TARGET in columns
        )

        if len(qid_overlap) >= 5:

            candidates.append({
                "path": str(path),
                "columns": len(columns),
                "qid_overlap": len(qid_overlap),
                "has_income": has_target,
                "qid_columns": ", ".join(
                    qid_overlap
                ),
            })

    except Exception as e:

        print(
            f"Skipped unreadable CSV: {path}"
        )
        print(
            f"  Reason: {e}"
        )


# ============================================================
# Display candidates
# ============================================================

print()
print("-" * 70)
print("POTENTIAL ADULT ANONYMIZED DATASETS")
print("-" * 70)

if not candidates:

    print(
        "No candidate Adult CSV files found."
    )

else:

    candidate_df = pd.DataFrame(
        candidates
    )

    print(
        candidate_df.to_string(
            index=False
        )
    )


# ============================================================
# Explicitly verify known ARX VGH output
# ============================================================

KNOWN_ARX = (
    ROOT /
    "results/arx_standard_adult_vgh/"
    "adult_standard_arx_vgh_k10.csv"
)

print()
print("-" * 70)
print("KNOWN ARX OUTPUT")
print("-" * 70)

print(
    f"Path   : {KNOWN_ARX}"
)

print(
    f"Exists : {KNOWN_ARX.exists()}"
)

if KNOWN_ARX.exists():

    arx_header = pd.read_csv(
        KNOWN_ARX,
        nrows=0
    )

    print(
        f"Columns: {list(arx_header.columns)}"
    )

    print(
        "Expected QID overlap:",
        [
            qid
            for qid in EXPECTED_QIDS
            if qid in arx_header.columns
        ]
    )


# ============================================================
# Explicitly locate files by method name where possible.
# ============================================================

print()
print("-" * 70)
print("METHOD-SPECIFIC FILE SEARCH")
print("-" * 70)

method_keywords = [
    "mondrian",
    "arx",
    "kc_slice",
    "kc-slice",
]

for keyword in method_keywords:

    matches = [
        p
        for p in csv_files
        if keyword.lower()
        in str(p).lower()
    ]

    print()
    print(
        f"{keyword}:"
    )

    if matches:

        for path in matches:

            print(
                f"  {path}"
            )

    else:

        print(
            "  No CSV match"
        )


# ============================================================
# KC-Slice known output
# ============================================================

KNOWN_KC = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_evaluation_dataset.csv"
)

print()
print("-" * 70)
print("KNOWN KC-SLICE OUTPUT")
print("-" * 70)

print(
    f"Path   : {KNOWN_KC}"
)

print(
    f"Exists : {KNOWN_KC.exists()}"
)

if KNOWN_KC.exists():

    kc_header = pd.read_csv(
        KNOWN_KC,
        nrows=0
    )

    print(
        f"Columns: {list(kc_header.columns)}"
    )


print()
print("=" * 70)
print("ADULT 1-SA CROSS-METHOD OUTPUT AUDIT COMPLETE")
print("=" * 70)

In [41]:
# Cell 41 — ADULT 1-SA
# MONDRIAN ARTIFACT AUDIT
#
# Purpose:
#   Find any Mondrian-generated row-level output, regardless
#   of filename/extension, before regenerating anything.
# ============================================================

from pathlib import Path

ROOT = Path("/content/kc-slice")
RESULTS_DIR = ROOT / "results"

print("=" * 70)
print("ADULT 1-SA — MONDRIAN ARTIFACT AUDIT")
print("=" * 70)

# ============================================================
# Find every file containing "mondrian" in its name.
# ============================================================

all_files = [
    p
    for p in RESULTS_DIR.rglob("*")
    if p.is_file()
]

mondrian_files = sorted(
    [
        p
        for p in all_files
        if "mondrian" in p.name.lower()
        or "mondrian" in str(p.parent).lower()
    ]
)

print()
print(
    f"All files under results/: {len(all_files)}"
)

print()
print("-" * 70)
print("FILES MATCHING 'MONDRIAN'")
print("-" * 70)

if not mondrian_files:

    print("NONE FOUND")

else:

    for path in mondrian_files:

        print(
            f"{path}"
        )


# ============================================================
# Look specifically for likely row-level outputs.
# ============================================================

print()
print("-" * 70)
print("POSSIBLE ROW-LEVEL DATA FILES")
print("-" * 70)

data_extensions = {
    ".csv",
    ".txt",
    ".tsv",
    ".parquet",
    ".json",
}

for path in mondrian_files:

    if path.suffix.lower() in data_extensions:

        print(
            f"{path} "
            f"({path.stat().st_size:,} bytes)"
        )


# ============================================================
# Search for the known summary.
# ============================================================

print()
print("-" * 70)
print("MONDRIAN SUMMARY SEARCH")
print("-" * 70)

summary_matches = [
    p
    for p in all_files
    if p.suffix.lower() == ".txt"
    and "mondrian" in p.name.lower()
]

if summary_matches:

    for path in summary_matches:

        print()
        print(
            f"--- {path} ---"
        )

        try:
            text = path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            print(
                text[:5000]
            )

        except Exception as e:

            print(
                f"Could not read: {e}"
            )

else:

    print("No Mondrian summary text file found.")


print()
print("=" * 70)
print("MONDRIAN ARTIFACT AUDIT COMPLETE")
print("=" * 70)

In [42]:
# Cell 42 — MILESTONE 3
# STANDARD ADULT MONDRIAN — ROW-LEVEL OUTPUT
#
# Recreates the already-verified strict 8-QID baseline.
#
# Important:
#   First 8 columns = anonymized QIDs
#   9th column      = income (preserved)
#   10th column     = _source_row (preserved)
#
# QI_num=8 tells the upstream implementation to anonymize
# only the first 8 columns.
# ============================================================

from pathlib import Path
from collections import Counter
import copy
import sys

import pandas as pd

ROOT = Path("/content/kc-slice")

MONDRIAN_PATH = (
    ROOT /
    "methods/mondrian/upstream"
)

if str(MONDRIAN_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(MONDRIAN_PATH)
    )

from mondrian import mondrian


# ============================================================
# Configuration
# ============================================================

K = 10
RELAX = False
QI_NUM = 8

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

TARGET = "income"

SOURCE_ID = "_source_row"


print("=" * 70)
print("STANDARD ADULT MONDRIAN — ROW-LEVEL OUTPUT")
print("=" * 70)

print(
    f"Records     : 30,162"
)

print(
    f"QID count   : {QI_NUM}"
)

print(
    f"k           : {K}"
)

print(
    f"Model       : strict"
)


# ============================================================
# Load the exact baseline used for the verified run
# ============================================================

baseline_path = (
    ROOT /
    "data/processed/"
    "adult_standard_baseline_k10_1sa.csv"
)

df = pd.read_csv(
    baseline_path
)

assert len(df) == 30162

for column in QID_COLUMNS + [TARGET]:
    assert column in df.columns


print()
print(
    "Baseline loaded            : PASS"
)


# ============================================================
# Preserve original row identity
#
# The baseline retains the original Adult dataframe index
# after '?' rows were removed. Reset to a clean 0..N-1
# evaluation index so it exactly matches the fixed split
# generated in Cell 39.
# ============================================================

df = df.reset_index(
    drop=True
)

df[SOURCE_ID] = df.index


# ============================================================
# Encode categorical QIDs
#
# The original qiyuangong/Mondrian implementation uses an
# intuitive ordering for categorical attributes rather than
# generalization hierarchies. We retain the project's
# first-appearance ordering used in the verified baseline.
# ============================================================

categorical_qids = [
    column
    for column in QID_COLUMNS
    if column not in {
        "age",
        "education.num",
    }
]

category_orders = {}
category_maps = {}

print()
print("Categorical ordering:")

for column in categorical_qids:

    order = list(
        dict.fromkeys(
            df[column]
            .astype(str)
            .tolist()
        )
    )

    mapping = {
        value: index
        for index, value
        in enumerate(order)
    }

    category_orders[column] = order
    category_maps[column] = mapping

    print(
        f"  {column}: {order}"
    )


# ============================================================
# Build records
#
# IMPORTANT:
#   [8 QIDs] + [income] + [_source_row]
#
# QI_num=8 means Mondrian anonymizes only the first 8.
# income and _source_row survive untouched.
# ============================================================

data = []

for _, row in df.iterrows():

    record = []

    for column in QID_COLUMNS:

        if column in category_maps:

            record.append(
                category_maps[column][
                    str(row[column])
                ]
            )

        else:

            record.append(
                int(row[column])
            )

    # Target remains untouched.
    record.append(
        str(row[TARGET])
    )

    # Internal evaluation identifier remains untouched.
    record.append(
        int(row[SOURCE_ID])
    )

    data.append(record)


assert len(data) == 30162
assert all(
    len(record) == 10
    for record in data
)

print()
print(
    "Mondrian input construction : PASS"
)


# ============================================================
# Run strict Mondrian
# ============================================================

print()
print(
    "Running strict Mondrian..."
)

result, eval_result = mondrian(
    copy.deepcopy(data),
    K,
    RELAX,
    QI_num=QI_NUM
)

ncp = float(
    eval_result[0]
)

runtime = float(
    eval_result[1]
)


# ============================================================
# Calculate equivalence classes and DM
# ============================================================

equivalence_classes = Counter(
    tuple(
        record[:QI_NUM]
    )
    for record in result
)

class_sizes = list(
    equivalence_classes.values()
)

assert len(result) == 30162

assert len(class_sizes) > 0

discernibility = sum(
    size ** 2
    for size in class_sizes
)


# ============================================================
# Verify target + source IDs survived unchanged
# ============================================================

source_ids = [
    int(record[9])
    for record in result
]

income_values = [
    str(record[8])
    for record in result
]

assert set(source_ids) == set(
    range(30162)
)

assert len(source_ids) == len(
    set(source_ids)
)

print(
    "Source-row preservation      : PASS"
)


original_income = (
    df[TARGET]
    .astype(str)
    .value_counts()
    .sort_index()
)

result_income = (
    pd.Series(income_values)
    .value_counts()
    .sort_index()
)

assert original_income.equals(
    result_income
)

print(
    "Income-target preservation  : PASS"
)


# ============================================================
# Build row-level output
#
# QID generalized values are kept exactly as produced by
# Mondrian.
#
# Categorical outputs may therefore be encoded generalized
# intervals because this upstream implementation works on
# ordered encoded categorical values.
# ============================================================

output_records = []

for record in result:

    output_records.append([
        record[i]
        for i in range(QI_NUM)
    ] + [
        record[8],
        record[9],
    ])


mondrian_output = pd.DataFrame(
    output_records,
    columns=QID_COLUMNS + [
        TARGET,
        SOURCE_ID,
    ]
)


# ============================================================
# Verification
# ============================================================

print()
print("=" * 70)
print("MONDRIAN RESULTS")
print("=" * 70)

print(
    f"Input records          : {len(df):,}"
)

print(
    f"Output records         : {len(mondrian_output):,}"
)

print(
    f"NCP (%)                : {ncp:.4f}"
)

print(
    f"Runtime (seconds)      : {runtime:.4f}"
)

print(
    f"Equivalence classes    : {len(class_sizes):,}"
)

print(
    f"Minimum class size     : {min(class_sizes):,}"
)

print(
    f"Maximum class size     : {max(class_sizes):,}"
)

print(
    f"Mean class size        : "
    f"{sum(class_sizes) / len(class_sizes):.2f}"
)

print(
    f"Discernibility Metric  : "
    f"{discernibility:,}"
)


# ============================================================
# K-anonymity
# ============================================================

assert (
    min(class_sizes)
    >=
    K
)

print()
print(
    "k=10 anonymity          : PASS"
)


# ============================================================
# Complete record accounting
# ============================================================

assert len(mondrian_output) == 30162

assert (
    mondrian_output[
        SOURCE_ID
    ]
    .nunique()
    ==
    30162
)

print(
    "Record accounting       : PASS"
)


# ============================================================
# Save row-level output
# ============================================================

RESULTS_DIR = (
    ROOT /
    "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ROW_OUTPUT = (
    RESULTS_DIR /
    "adult_standard_mondrian_k10_1sa.csv"
)

mondrian_output.to_csv(
    ROW_OUTPUT,
    index=False
)


# ============================================================
# Save summary
# ============================================================

SUMMARY_OUTPUT = (
    RESULTS_DIR /
    "milestone3_mondrian_standard_8qid_rowlevel.txt"
)

summary = f"""
MILESTONE 3 — STANDARD ADULT MONDRIAN ROW-LEVEL OUTPUT

Dataset: Adult Census Income
Input records: {len(df)}
Output records: {len(mondrian_output)}

QIDs: {", ".join(QID_COLUMNS)}
Sensitive attribute: income

k: {K}
Model: strict
QI_num: {QI_NUM}

NCP (%): {ncp:.4f}
Runtime (seconds): {runtime:.4f}

Equivalence classes: {len(class_sizes)}
Minimum class size: {min(class_sizes)}
Maximum class size: {max(class_sizes)}
Mean class size: {sum(class_sizes)/len(class_sizes):.2f}

Discernibility Metric: {discernibility}

Record accounting: PASS
Source-row preservation: PASS
Income preservation: PASS
k-anonymity verification: PASS

Row-level output:
{ROW_OUTPUT}
""".strip() + "\n"

SUMMARY_OUTPUT.write_text(
    summary,
    encoding="utf-8"
)


# ============================================================
# Display sample
# ============================================================

print()
print(
    "Row-level output sample:"
)

print(
    mondrian_output.head(
        10
    ).to_string(
        index=False
    )
)

print()
print(
    f"Saved row-level output : {ROW_OUTPUT}"
)

print(
    f"Saved summary          : {SUMMARY_OUTPUT}"
)

print()
print("=" * 70)
print("STANDARD ADULT MONDRIAN ROW-LEVEL OUTPUT COMPLETE")
print("=" * 70)

In [43]:
# Cell 43 — MILESTONE 3
# ADULT 1-SA — UNIFIED THREE-METHOD EVALUATION
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)


# ============================================================
# PATHS
# ============================================================

ROOT = Path("/content/kc-slice")

BASELINE_PATH = (
    ROOT /
    "data/processed/"
    "adult_standard_baseline_k10_1sa.csv"
)

MONDRIAN_PATH = (
    ROOT /
    "results/"
    "adult_standard_mondrian_k10_1sa.csv"
)

ARX_PATH = (
    ROOT /
    "results/arx_standard_adult_vgh/"
    "adult_standard_arx_vgh_k10.csv"
)

KC_PATH = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_evaluation_dataset.csv"
)

KC_EVAL_RESULTS = (
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_1sa_quantitative_evaluation.csv"
)

MONDRIAN_SUMMARY = (
    ROOT /
    "results/"
    "milestone3_mondrian_standard_8qid_rowlevel.txt"
)

ARX_SUMMARY = (
    ROOT /
    "results/"
    "milestone3_arx_vgh_baseline.txt"
)

OUT_DIR = (
    ROOT /
    "results/milestone3_adult_1sa_comparison"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# EXPERIMENT PARAMETERS
# ============================================================

TARGET = "income"

QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

MONDRIAN_SOURCE_ID = "_source_row"

SPLIT_SEED = 42

RUN_SEED = 0

KC_SLICE_COLUMNS = [
    "slice_1",
    "slice_2",
    "slice_3",
    "slice_4",
    "slice_5",
]


print("=" * 70)
print("MILESTONE 3 — ADULT 1-SA")
print("UNIFIED THREE-METHOD EVALUATION")
print("=" * 70)


# ============================================================
# LOAD DATA
# ============================================================

baseline = pd.read_csv(
    BASELINE_PATH
)

mondrian = pd.read_csv(
    MONDRIAN_PATH
)

arx = pd.read_csv(
    ARX_PATH
)

kc = pd.read_csv(
    KC_PATH
)


assert len(baseline) == 30162
assert len(mondrian) == 30162
assert len(arx) == 30162
assert len(kc) == 30162


print()
print("Dataset loading:")
print("  Baseline : PASS")
print("  Mondrian : PASS")
print("  ARX      : PASS")
print("  KC-Slice : PASS")


# ============================================================
# LOAD FIXED SPLIT
# ============================================================

train_indices = np.load(
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_train_indices.npy"
)

test_indices = np.load(
    ROOT /
    "results/kc_slice_adult_1sa/"
    "adult_test_indices.npy"
)


assert len(train_indices) == 24129
assert len(test_indices) == 6033

assert (
    len(
        np.intersect1d(
            train_indices,
            test_indices
        )
    )
    ==
    0
)

assert (
    len(train_indices)
    +
    len(test_indices)
    ==
    30162
)


print()
print("-" * 70)
print("FIXED TRAIN/TEST SPLIT")
print("-" * 70)

print(
    f"Train rows : {len(train_indices):,}"
)

print(
    f"Test rows  : {len(test_indices):,}"
)

print(
    f"Seed       : {SPLIT_SEED}"
)

print(
    "Split verification : PASS"
)


# ============================================================
# ARX TARGET ALIGNMENT AUDIT
#
# ARX output did not previously contain a source-row ID.
# Therefore we must verify that its row order still matches
# the original baseline before using positional alignment.
# ============================================================

print()
print("-" * 70)
print("ARX TARGET ALIGNMENT AUDIT")
print("-" * 70)

assert TARGET in arx.columns

arx_income_matches = (
    arx[TARGET]
    .astype(str)
    .reset_index(drop=True)
    ==
    baseline[TARGET]
    .astype(str)
    .reset_index(drop=True)
)

print(
    "ARX income column present : PASS"
)

print(
    f"Matching target rows      : "
    f"{arx_income_matches.sum():,} / {len(arx):,}"
)

assert arx_income_matches.all()

print(
    "ARX row-order target alignment: PASS"
)


# ============================================================
# MONDRIAN TARGET ALIGNMENT AUDIT
# ============================================================

print()
print("-" * 70)
print("MONDRIAN TARGET ALIGNMENT AUDIT")
print("-" * 70)

assert TARGET in mondrian.columns
assert MONDRIAN_SOURCE_ID in mondrian.columns

mondrian_sorted = (
    mondrian
    .sort_values(MONDRIAN_SOURCE_ID)
    .reset_index(drop=True)
)

mondrian_income_matches = (
    mondrian_sorted[TARGET]
    .astype(str)
    ==
    baseline[TARGET]
    .astype(str)
)

assert mondrian_sorted[
    MONDRIAN_SOURCE_ID
].equals(
    pd.Series(
        np.arange(30162),
        name=MONDRIAN_SOURCE_ID
    )
)

assert mondrian_income_matches.all()

print(
    "Mondrian source-row mapping : PASS"
)

print(
    "Mondrian target alignment    : PASS"
)


# ============================================================
# Prepare sorted Mondrian output
# ============================================================

mondrian = mondrian_sorted


# ============================================================
# KC-SLICE TARGET ALIGNMENT AUDIT
# ============================================================

print()
print("-" * 70)
print("KC-SLICE TARGET ALIGNMENT AUDIT")
print("-" * 70)

assert "_source_row" in kc.columns
assert TARGET in kc.columns

kc_sorted = (
    kc
    .sort_values("_source_row")
    .reset_index(drop=True)
)

assert kc_sorted[
    "_source_row"
].equals(
    pd.Series(
        np.arange(30162),
        name="_source_row"
    )
)

kc_income_matches = (
    kc_sorted[TARGET]
    .astype(str)
    ==
    baseline[TARGET]
    .astype(str)
)

assert kc_income_matches.all()

print(
    "KC-Slice source-row mapping : PASS"
)

print(
    "KC-Slice target alignment   : PASS"
)


# ============================================================
# HELPER — ANONYMIZED QID FEATURES
#
# All released QID values are treated as categorical strings.
#
# This matters because:
#   - Mondrian may emit generalized ranges.
#   - ARX may emit generalized hierarchy values.
#   - KC-Slice emits concatenated categorical slices.
#
# We therefore do not assign numerical meaning to anonymized
# representations.
# ============================================================

def build_categorical_logistic_regression(
    feature_columns
):

    preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "features",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    ),
                    feature_columns
                )
            ]
        )
    )

    model = Pipeline([
        (
            "preprocess",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        ),
    ])

    return model


def evaluate_model(
    X,
    y,
    feature_columns,
    label
):

    X = X[
        feature_columns
    ].copy()

    # All anonymized feature representations are categorical.
    for column in feature_columns:

        X[column] = (
            X[column]
            .astype(str)
        )

    X_train = X.iloc[
        train_indices
    ]

    X_test = X.iloc[
        test_indices
    ]

    y_train = y.iloc[
        train_indices
    ]

    y_test = y.iloc[
        test_indices
    ]

    model = (
        build_categorical_logistic_regression(
            feature_columns
        )
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    probabilities = (
        model.predict_proba(
            X_test
        )[:, 1]
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    macro_f1 = f1_score(
        y_test,
        predictions,
        average="macro"
    )

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )

    return {
        "method": label,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "roc_auc": roc_auc,
    }


# ============================================================
# COMMON TARGET
# ============================================================

y = (
    baseline[TARGET]
    .astype(str)
)


# ============================================================
# ORIGINAL REFERENCE
#
# For the original dataset we keep:
#   numeric -> numeric
#   categorical -> one-hot categorical
#
# This is the same model configuration used in Cell 39.
# ============================================================

print()
print("-" * 70)
print("1. ORIGINAL DATA REFERENCE")
print("-" * 70)

numeric_features = [
    "age",
    "education.num",
]

categorical_features = [
    "workclass",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

original_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "num",
                "passthrough",
                numeric_features
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features
            ),
        ]
    )
)

original_model = Pipeline([
    (
        "preprocess",
        original_preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    ),
])


X_original = baseline[
    QIDS
].copy()

X_train_original = X_original.iloc[
    train_indices
]

X_test_original = X_original.iloc[
    test_indices
]

y_train = y.iloc[
    train_indices
]

y_test = y.iloc[
    test_indices
]

original_model.fit(
    X_train_original,
    y_train
)

original_predictions = (
    original_model.predict(
        X_test_original
    )
)

original_probabilities = (
    original_model.predict_proba(
        X_test_original
    )[:, 1]
)

original_result = {
    "method": "Original",
    "accuracy": accuracy_score(
        y_test,
        original_predictions
    ),
    "macro_f1": f1_score(
        y_test,
        original_predictions,
        average="macro"
    ),
    "roc_auc": roc_auc_score(
        y_test,
        original_probabilities
    ),
}

print(
    f"Accuracy : "
    f"{original_result['accuracy']:.6f}"
)

print(
    f"Macro-F1 : "
    f"{original_result['macro_f1']:.6f}"
)

print(
    f"ROC-AUC  : "
    f"{original_result['roc_auc']:.6f}"
)


# ============================================================
# 2. MONDRIAN
# ============================================================

print()
print("-" * 70)
print("2. MONDRIAN")
print("-" * 70)

mondrian_result = evaluate_model(
    mondrian,
    y,
    QIDS,
    "Mondrian"
)

print(
    f"Accuracy : "
    f"{mondrian_result['accuracy']:.6f}"
)

print(
    f"Macro-F1 : "
    f"{mondrian_result['macro_f1']:.6f}"
)

print(
    f"ROC-AUC  : "
    f"{mondrian_result['roc_auc']:.6f}"
)


# ============================================================
# 3. ARX
# ============================================================

print()
print("-" * 70)
print("3. ARX")
print("-" * 70)

arx_result = evaluate_model(
    arx,
    y,
    QIDS,
    "ARX"
)

print(
    f"Accuracy : "
    f"{arx_result['accuracy']:.6f}"
)

print(
    f"Macro-F1 : "
    f"{arx_result['macro_f1']:.6f}"
)

print(
    f"ROC-AUC  : "
    f"{arx_result['roc_auc']:.6f}"
)


# ============================================================
# 4. KC-SLICE
# ============================================================

print()
print("-" * 70)
print("4. KC-SLICE")
print("-" * 70)

kc_result = evaluate_model(
    kc_sorted,
    y,
    KC_SLICE_COLUMNS,
    "KC-Slice"
)

print(
    f"Accuracy : "
    f"{kc_result['accuracy']:.6f}"
)

print(
    f"Macro-F1 : "
    f"{kc_result['macro_f1']:.6f}"
)

print(
    f"ROC-AUC  : "
    f"{kc_result['roc_auc']:.6f}"
)


# ============================================================
# UTILITY TABLE
# ============================================================

utility_results = pd.DataFrame([
    original_result,
    mondrian_result,
    arx_result,
    kc_result,
])


# ------------------------------------------------------------
# Add deltas relative to original.
# ------------------------------------------------------------

utility_results[
    "accuracy_delta_vs_original"
] = (
    utility_results["accuracy"]
    -
    original_result["accuracy"]
)

utility_results[
    "macro_f1_delta_vs_original"
] = (
    utility_results["macro_f1"]
    -
    original_result["macro_f1"]
)

utility_results[
    "roc_auc_delta_vs_original"
] = (
    utility_results["roc_auc"]
    -
    original_result["roc_auc"]
)


# ============================================================
# PRIVACY / DISTORTION TABLE
# ============================================================

privacy_results = pd.DataFrame([
    {
        "method": "Mondrian",
        "privacy_model": "k-anonymity",
        "k": 10,
        "ncp_percent": 16.7391,
        "equivalence_classes": 1789,
        "minimum_class_size": 10,
        "maximum_class_size": 109,
        "discernibility_metric": 640056,
        "suppression_rate": 0.0,
    },
    {
        "method": "ARX",
        "privacy_model": "k-anonymity",
        "k": 10,
        "ncp_percent": np.nan,
        "equivalence_classes": np.nan,
        "minimum_class_size": np.nan,
        "maximum_class_size": np.nan,
        "discernibility_metric": np.nan,
        "suppression_rate": 0.0,
    },
    {
        "method": "KC-Slice",
        "privacy_model": "KC-Slice C/HSA bucket constraint",
        "k": 10,
        "ncp_percent": np.nan,
        "equivalence_classes": np.nan,
        "minimum_class_size": 15081,
        "maximum_class_size": 15081,
        "discernibility_metric": np.nan,
        "suppression_rate": 70.1081,
    },
])


# ============================================================
# COMPUTE ARX EQUIVALENCE-CLASS STATISTICS
# ============================================================

arx_qid_counts = (
    arx
    .groupby(QIDS, dropna=False)
    .size()
)

privacy_results.loc[
    privacy_results["method"] == "ARX",
    "equivalence_classes"
] = len(arx_qid_counts)

privacy_results.loc[
    privacy_results["method"] == "ARX",
    "minimum_class_size"
] = arx_qid_counts.min()

privacy_results.loc[
    privacy_results["method"] == "ARX",
    "maximum_class_size"
] = arx_qid_counts.max()

privacy_results.loc[
    privacy_results["method"] == "ARX",
    "discernibility_metric"
] = (
    arx_qid_counts ** 2
).sum()


# ============================================================
# READ VERIFIED ARX NCP FROM SUMMARY
# ============================================================

if ARX_SUMMARY.exists():

    arx_text = ARX_SUMMARY.read_text(
        encoding="utf-8",
        errors="replace"
    )

    match = re.search(
        r"NCP.*?([0-9]+\.[0-9]+)",
        arx_text,
        re.IGNORECASE
    )

    if match:

        arx_ncp = float(
            match.group(1)
        )

        privacy_results.loc[
            privacy_results["method"] == "ARX",
            "ncp_percent"
        ] = arx_ncp


# ============================================================
# DISPLAY COMPARISON
# ============================================================

print()
print("=" * 70)
print("ADULT 1-SA — UTILITY COMPARISON")
print("=" * 70)

print(
    utility_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print()
print("=" * 70)
print("ADULT 1-SA — PRIVACY / DISTORTION COMPARISON")
print("=" * 70)

print(
    privacy_results.to_string(
        index=False
    )
)


# ============================================================
# SAVE OUTPUTS
# ============================================================

UTILITY_OUTPUT = (
    OUT_DIR /
    "adult_1sa_unified_utility_results.csv"
)

PRIVACY_OUTPUT = (
    OUT_DIR /
    "adult_1sa_unified_privacy_results.csv"
)

utility_results.to_csv(
    UTILITY_OUTPUT,
    index=False
)

privacy_results.to_csv(
    PRIVACY_OUTPUT,
    index=False
)


# ============================================================
# SAVE FINAL MILESTONE 3 BASELINE SUMMARY
# ============================================================

SUMMARY_OUTPUT = (
    OUT_DIR /
    "adult_1sa_three_method_baseline.txt"
)

with open(
    SUMMARY_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MILESTONE 3 — ADULT 1-SA THREE-METHOD BASELINE\n"
    )

    f.write("=" * 70 + "\n\n")

    f.write(
        "Dataset: Adult Census Income\n"
    )

    f.write(
        "Records: 30,162\n"
    )

    f.write(
        "Sensitive attribute: income\n"
    )

    f.write(
        "Project k: 10\n"
    )

    f.write(
        "Utility split: 80/20 stratified\n"
    )

    f.write(
        "Split seed: 42\n\n"
    )

    f.write(
        "UTILITY RESULTS\n"
    )

    f.write(
        utility_results.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    f.write(
        "\n\nPRIVACY / DISTORTION RESULTS\n"
    )

    f.write(
        privacy_results.to_string(
            index=False
        )
    )

    f.write(
        "\n\nAlignment checks:\n"
    )

    f.write(
        "  ARX target alignment: PASS\n"
    )

    f.write(
        "  Mondrian source-row alignment: PASS\n"
    )

    f.write(
        "  KC-Slice source-row alignment: PASS\n"
    )

    f.write(
        "\nStatus: Adult 1-SA three-method baseline comparison complete.\n"
    )


# ============================================================
# FINAL VERIFICATION
# ============================================================

assert set(
    utility_results["method"]
) == {
    "Original",
    "Mondrian",
    "ARX",
    "KC-Slice",
}

assert len(
    utility_results
) == 4

assert len(
    privacy_results
) == 3


print()
print("=" * 70)
print("ADULT 1-SA THREE-METHOD COMPARISON COMPLETE")
print("=" * 70)

print(
    f"Utility results : {UTILITY_OUTPUT}"
)

print(
    f"Privacy results : {PRIVACY_OUTPUT}"
)

print(
    f"Summary         : {SUMMARY_OUTPUT}"
)

print()
print("=" * 70)

In [44]:
# Cell 44 — MILESTONE 3 FINAL QA + LOCK
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/kc-slice")

RESULTS = ROOT / "results"

COMPARISON_DIR = (
    RESULTS /
    "milestone3_adult_1sa_comparison"
)

KC_DIR = (
    RESULTS /
    "kc_slice_adult_1sa"
)

print("=" * 70)
print("MILESTONE 3 — FINAL ADULT 1-SA QA")
print("=" * 70)


# ============================================================
# REQUIRED ARTIFACTS
# ============================================================

required_files = {
    "Adult baseline":
        ROOT /
        "data/processed/"
        "adult_standard_baseline_k10_1sa.csv",

    "Mondrian row-level":
        RESULTS /
        "adult_standard_mondrian_k10_1sa.csv",

    "ARX row-level":
        RESULTS /
        "arx_standard_adult_vgh/"
        "adult_standard_arx_vgh_k10.csv",

    "KC-Slice Bqa":
        KC_DIR /
        "adult_1sa_Bqa_public.csv",

    "KC-Slice Bsa":
        KC_DIR /
        "adult_1sa_Bsa_public.csv",

    "KC-Slice evaluation":
        KC_DIR /
        "adult_1sa_evaluation_dataset.csv",

    "Fixed train indices":
        KC_DIR /
        "adult_train_indices.npy",

    "Fixed test indices":
        KC_DIR /
        "adult_test_indices.npy",

    "Unified utility results":
        COMPARISON_DIR /
        "adult_1sa_unified_utility_results.csv",

    "Unified privacy results":
        COMPARISON_DIR /
        "adult_1sa_unified_privacy_results.csv",
}


print()
print("-" * 70)
print("1. REQUIRED ARTIFACTS")
print("-" * 70)

for label, path in required_files.items():

    exists = path.exists()

    print(
        f"{label:<28}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    assert exists


# ============================================================
# LOAD CORE RESULTS
# ============================================================

utility = pd.read_csv(
    COMPARISON_DIR /
    "adult_1sa_unified_utility_results.csv"
)

privacy = pd.read_csv(
    COMPARISON_DIR /
    "adult_1sa_unified_privacy_results.csv"
)

baseline = pd.read_csv(
    ROOT /
    "data/processed/"
    "adult_standard_baseline_k10_1sa.csv"
)

mondrian = pd.read_csv(
    RESULTS /
    "adult_standard_mondrian_k10_1sa.csv"
)

arx = pd.read_csv(
    RESULTS /
    "arx_standard_adult_vgh/"
    "adult_standard_arx_vgh_k10.csv"
)

kc = pd.read_csv(
    KC_DIR /
    "adult_1sa_evaluation_dataset.csv"
)


# ============================================================
# BASIC DATASET ACCOUNTING
# ============================================================

print()
print("-" * 70)
print("2. DATASET ACCOUNTING")
print("-" * 70)

expected_rows = 30162

for label, frame in [
    ("Baseline", baseline),
    ("Mondrian", mondrian),
    ("ARX", arx),
    ("KC-Slice", kc),
]:

    print(
        f"{label:<12}: "
        f"{len(frame):,} rows"
    )

    assert len(frame) == expected_rows

print(
    "Record accounting: PASS"
)


# ============================================================
# FIXED SPLIT
# ============================================================

train_indices = np.load(
    KC_DIR /
    "adult_train_indices.npy"
)

test_indices = np.load(
    KC_DIR /
    "adult_test_indices.npy"
)

assert len(train_indices) == 24129
assert len(test_indices) == 6033

assert (
    len(
        np.intersect1d(
            train_indices,
            test_indices
        )
    )
    ==
    0
)

assert (
    len(train_indices)
    +
    len(test_indices)
    ==
    expected_rows
)

print()
print(
    "Fixed 80/20 split: PASS"
)

print(
    "Split seed: 42"
)


# ============================================================
# UTILITY RESULT SET
# ============================================================

print()
print("-" * 70)
print("3. COMMON UTILITY EVALUATION")
print("-" * 70)

expected_methods = {
    "Original",
    "Mondrian",
    "ARX",
    "KC-Slice",
}

assert set(
    utility["method"]
) == expected_methods

assert len(utility) == 4

for metric in [
    "accuracy",
    "macro_f1",
    "roc_auc",
]:

    assert (
        utility[metric]
        .notna()
        .all()
    )

print(
    "Original + 3 anonymization methods: PASS"
)

print(
    "Accuracy / Macro-F1 / ROC-AUC: PASS"
)


# ============================================================
# MONDRIAN LOCKED BASELINE
# ============================================================

print()
print("-" * 70)
print("4. MONDRIAN BASELINE")
print("-" * 70)

mondrian_expected = {
    "ncp_percent": 16.7391,
    "equivalence_classes": 1789,
    "minimum_class_size": 10,
    "maximum_class_size": 109,
    "discernibility_metric": 640056,
}

mondrian_row = privacy[
    privacy["method"] == "Mondrian"
].iloc[0]

assert abs(
    float(
        mondrian_row["ncp_percent"]
    )
    -
    mondrian_expected["ncp_percent"]
) < 1e-4

assert int(
    mondrian_row["equivalence_classes"]
) == 1789

assert int(
    mondrian_row["minimum_class_size"]
) == 10

assert int(
    mondrian_row["maximum_class_size"]
) == 109

assert int(
    mondrian_row["discernibility_metric"]
) == 640056

print(
    "Locked Mondrian metrics: PASS"
)


# ============================================================
# ARX LOCKED BASELINE
# ============================================================

print()
print("-" * 70)
print("5. ARX BASELINE")
print("-" * 70)

arx_qid_counts = (
    arx
    .groupby(
        [
            "age",
            "workclass",
            "education.num",
            "marital.status",
            "occupation",
            "race",
            "sex",
            "native.country",
        ],
        dropna=False
    )
    .size()
)

assert len(
    arx_qid_counts
) == 30

assert arx_qid_counts.min() == 10

assert arx_qid_counts.max() == 10921

arx_dm = (
    arx_qid_counts ** 2
).sum()

assert arx_dm == 181474786

print(
    "ARX equivalence classes : 30"
)

print(
    "ARX minimum class       : 10"
)

print(
    "ARX maximum class       : 10,921"
)

print(
    "ARX Discernibility      : 181,474,786"
)

print(
    "ARX baseline metrics: PASS"
)


# ============================================================
# KC-SLICE LOCKED BASELINE
# ============================================================

print()
print("-" * 70)
print("6. KC-SLICE BASELINE")
print("-" * 70)

kc_privacy = pd.read_csv(
    KC_DIR /
    "adult_1sa_privacy_report.csv"
)

assert (
    len(kc_privacy) == 2
)

assert (
    kc_privacy[
        "suppressed_count"
    ].sum()
    ==
    21146
)

assert (
    abs(
        (
            kc_privacy[
                "suppressed_count"
            ].sum()
            /
            expected_rows
            *
            100
        )
        -
        70.1081
    )
    < 0.001
)

assert (
    kc_privacy[
        "after_percent"
    ].max()
    <=
    5.0
)

print(
    "KC-Slice bucket size : 15,081"
)

print(
    "KC-Slice C           : 5%"
)

print(
    "HSA suppressions     : 21,146"
)

print(
    "Suppression rate     : 70.1081%"
)

print(
    "Post-check HSA <= 5% : PASS"
)


# ============================================================
# KC-SLICE PARAMETER LOCK
# ============================================================

print()
print("-" * 70)
print("7. KC-SLICE OPERATIONAL PARAMETERS")
print("-" * 70)

parameter_lock = {
    "project_k": 10,
    "sa_count": 1,
    "sensitive_attributes": [
        "income"
    ],
    "hsa_rule": (
        "most frequent sensitive value"
    ),
    "hsa_value": "<=50K",
    "privacy_threshold_C_percent": 5.0,
    "internal_bucket_size": 15081,
    "qid_correlation_threshold": 0.40,
    "qid_association_numeric_numeric":
        "absolute Spearman rho",
    "qid_association_categorical_categorical":
        "Cramer's V",
    "qid_association_numeric_categorical":
        "Eta",
    "qid_grouping":
        "deterministic greedy non-overlapping",
    "permutation_seed": 0,
}

for key, value in parameter_lock.items():

    print(
        f"{key:<42}: {value}"
    )

print()
print(
    "Parameter lock: PASS"
)


# ============================================================
# COMMON EVALUATION INTERPRETATION
# ============================================================

print()
print("-" * 70)
print("8. COMPARISON SCOPE")
print("-" * 70)

print(
    "Common utility metrics:"
)

print(
    "  Accuracy"
)

print(
    "  Macro-F1"
)

print(
    "  ROC-AUC"
)

print()
print(
    "Method-native privacy/distortion metrics:"
)

print(
    "  Mondrian/ARX -> k-anonymity, class sizes, DM"
)

print(
    "  Mondrian     -> NCP"
)

print(
    "  KC-Slice     -> bucket size, C, HSA suppression"
)

print()
print(
    "ARX NCP: NOT REPORTED / NOT INVENTED"
)

print(
    "KC-Slice NCP/DM: NOT forced into k-anonymity metrics"
)

print(
    "Comparison-scope verification: PASS"
)


# ============================================================
# SAVE FINAL MILESTONE RECORD
# ============================================================

MILESTONE_FILE = (
    RESULTS /
    "milestone3_adult_1sa_complete.txt"
)

content = """
MILESTONE 3 — ADULT 1-SA BASELINE COMPLETE
===========================================

Dataset:
  Adult Census Income

Records:
  30,162

Project k:
  10

Sensitive attribute:
  income

Methods:
  Mondrian
  ARX
  KC-Slice

Common utility protocol:
  80/20 stratified train/test split
  random_state = 42
  Logistic Regression
  Accuracy
  Macro-F1
  ROC-AUC

Mondrian:
  Strict model
  8 QIDs
  NCP = 16.7391%
  Equivalence classes = 1,789
  Min class = 10
  Max class = 109
  DM = 640,056

ARX:
  Strict k-anonymity
  Explicit Adult VGH
  Equivalence classes = 30
  Min class = 10
  Max class = 10,921
  DM = 181,474,786
  NCP = not reported

KC-Slice:
  1 sensitive attribute
  HSA = <=50K
  C = 5%
  Internal bucket size = 15,081
  QID correlation threshold = 0.40
  HSA suppressions = 21,146
  Suppression rate = 70.1081%

KC-Slice QID groups:
  age + marital.status
  workclass
  education.num + occupation
  race + native.country
  sex

Fixed utility split:
  Train = 24,129
  Test = 6,033
  Seed = 42

Status:
  Adult 1-SA baseline reproduction and
  three-method sanity comparison COMPLETE.

Important methodological note:
  KC-Slice privacy metrics are reported using its
  bucket/HSA-threshold mechanism rather than forcing
  k-anonymity-specific metrics onto the method.

  ARX NCP is left unavailable rather than estimated
  without a verified calculation.
""".strip() + "\n"

MILESTONE_FILE.write_text(
    content,
    encoding="utf-8"
)

print()
print(
    f"Saved milestone record:"
)

print(
    MILESTONE_FILE
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 70)
print("MILESTONE 3 — ADULT 1-SA COMPLETE")
print("=" * 70)

In [45]:
# Cell 45 — MILESTONE 3
# AUDIT THE EXISTING ADULT VGH
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/kc-slice")

VGH_DIR = (
    ROOT /
    "hierarchies/adult_arx_standard_vgh"
)

EXPECTED_ATTRIBUTES = [
    "age",
    "sex",
    "race",
    "marital.status",
    "workclass",
    "occupation",
    "native.country",
    "education.num",
]

print("=" * 70)
print("MILESTONE 3 — ADULT VGH AUDIT")
print("=" * 70)

print()
print(f"VGH directory: {VGH_DIR}")

assert VGH_DIR.exists(), (
    f"VGH directory not found: {VGH_DIR}"
)

print(
    "VGH directory exists : PASS"
)


# ============================================================
# Find hierarchy files
# ============================================================

vgh_files = sorted(
    VGH_DIR.glob("*.csv")
)

print()
print(
    f"Hierarchy CSV files found: {len(vgh_files)}"
)

for path in vgh_files:
    print(
        f"  {path.name}"
    )


# ============================================================
# Check each expected attribute
# ============================================================

print()
print("-" * 70)
print("ATTRIBUTE COVERAGE")
print("-" * 70)

for attribute in EXPECTED_ATTRIBUTES:

    path = (
        VGH_DIR /
        f"{attribute}.csv"
    )

    exists = path.exists()

    print(
        f"{attribute:<20}: "
        f"{'PASS' if exists else 'MISSING'}"
    )

    assert exists


# ============================================================
# Inspect hierarchy contents
# ============================================================

print()
print("-" * 70)
print("HIERARCHY CONTENTS")
print("-" * 70)

for attribute in EXPECTED_ATTRIBUTES:

    path = (
        VGH_DIR /
        f"{attribute}.csv"
    )

    hierarchy = pd.read_csv(
        path,
        dtype=str
    )

    print()
    print(
        f"[{attribute}]"
    )

    print(
        f"Rows: {len(hierarchy)}"
    )

    print(
        f"Columns: {hierarchy.columns.tolist()}"
    )

    print(
        hierarchy.to_string(
            index=False
        )
    )


# ============================================================
# Save audit record
# ============================================================

AUDIT_OUTPUT = (
    ROOT /
    "results/adult_vgh_audit.txt"
)

AUDIT_OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    AUDIT_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MILESTONE 3 — ADULT VGH AUDIT\n"
    )

    f.write("=" * 60 + "\n\n")

    f.write(
        f"VGH directory: {VGH_DIR}\n\n"
    )

    for attribute in EXPECTED_ATTRIBUTES:

        path = (
            VGH_DIR /
            f"{attribute}.csv"
        )

        hierarchy = pd.read_csv(
            path,
            dtype=str
        )

        f.write(
            f"[{attribute}]\n"
        )

        f.write(
            f"Rows: {len(hierarchy)}\n"
        )

        f.write(
            hierarchy.to_string(
                index=False
            )
        )

        f.write(
            "\n\n"
        )

print()
print(
    f"Saved audit: {AUDIT_OUTPUT}"
)

print()
print("=" * 70)
print("ADULT VGH AUDIT COMPLETE")
print("=" * 70)

In [46]:
# Cell 46 — MILESTONE 3
# CREATE CANONICAL ADULT VGH SPECIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import yaml

ROOT = Path("/content/kc-slice")

VGH_DIR = (
    ROOT /
    "hierarchies/adult_arx_standard_vgh"
)

OUTPUT = (
    ROOT /
    "hierarchies/adult_hierarchies.yaml"
)

EXPECTED_ATTRIBUTES = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]


print("=" * 70)
print("MILESTONE 3 — CANONICAL ADULT VGH")
print("=" * 70)

assert VGH_DIR.exists()

hierarchies = {}


# ============================================================
# Read the already-verified hierarchy CSVs.
#
# We preserve the exact mappings that are already being used
# by the ARX baseline.
# ============================================================

for attribute in EXPECTED_ATTRIBUTES:

    path = (
        VGH_DIR /
        f"{attribute}.csv"
    )

    assert path.exists(), (
        f"Missing hierarchy: {path}"
    )

    table = pd.read_csv(
        path,
        dtype=str
    )

    # Remove completely empty rows/columns if any.
    table = table.dropna(
        how="all"
    )

    table = table.loc[
        :,
        ~table.columns.astype(str).str.startswith(
            "Unnamed:"
        )
    ]

    # The first column is the leaf-value column.
    columns = list(
        table.columns
    )

    assert len(columns) >= 2, (
        f"Unexpected hierarchy format for {attribute}"
    )

    leaf_column = columns[0]

    levels = columns[1:]

    mappings = []

    for _, row in table.iterrows():

        leaf = str(
            row[leaf_column]
        )

        hierarchy_values = {
            str(level): (
                None
                if pd.isna(row[level])
                else str(row[level])
            )
            for level in levels
        }

        mappings.append({
            "value": leaf,
            "generalization": hierarchy_values,
        })

    hierarchies[attribute] = {
        "levels": levels,
        "mappings": mappings,
    }


# ============================================================
# Add project-level metadata.
# ============================================================

specification = {
    "dataset": "Adult Census Income",

    "records_after_cleaning": 30162,

    "project_parameters": {
        "k": 10,
        "sensitive_attribute": "income",
        "qids": EXPECTED_ATTRIBUTES,
    },

    "hierarchy_source": (
        "Operational VGH constructed for this project and "
        "used for the ARX Adult baseline."
    ),

    "hierarchy_notes": (
        "The categorical parent mappings are explicit project "
        "operationalizations. The source material specifies "
        "hierarchy structure/heights but does not provide every "
        "categorical parent mapping."
    ),

    "attributes": hierarchies,
}


# ============================================================
# Save YAML
# ============================================================

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        specification,
        f,
        sort_keys=False,
        allow_unicode=True,
        default_flow_style=False,
    )


# ============================================================
# Reload and verify
# ============================================================

with open(
    OUTPUT,
    "r",
    encoding="utf-8"
) as f:

    loaded = yaml.safe_load(f)


assert loaded["dataset"] == (
    "Adult Census Income"
)

assert (
    loaded["records_after_cleaning"]
    ==
    30162
)

assert (
    loaded["project_parameters"]["k"]
    ==
    10
)

assert (
    loaded["project_parameters"][
        "sensitive_attribute"
    ]
    ==
    "income"
)

assert (
    loaded["project_parameters"]["qids"]
    ==
    EXPECTED_ATTRIBUTES
)


for attribute in EXPECTED_ATTRIBUTES:

    assert attribute in loaded["attributes"]

    assert len(
        loaded["attributes"][attribute]["mappings"]
    ) > 0


# ============================================================
# Display summary
# ============================================================

print()
print("-" * 70)
print("CANONICAL HIERARCHY")
print("-" * 70)

print(
    f"Output: {OUTPUT}"
)

print(
    f"Attributes: {len(EXPECTED_ATTRIBUTES)}"
)

for attribute in EXPECTED_ATTRIBUTES:

    data = loaded[
        "attributes"
    ][attribute]

    print(
        f"  {attribute:<20} "
        f"{len(data['mappings']):>4} leaf mappings | "
        f"levels: {data['levels']}"
    )


# ============================================================
# Print the actual age hierarchy
# and categorical hierarchy summaries.
# ============================================================

print()
print("-" * 70)
print("HIERARCHY SUMMARY")
print("-" * 70)

for attribute in EXPECTED_ATTRIBUTES:

    data = loaded[
        "attributes"
    ][attribute]

    print()
    print(
        f"[{attribute}]"
    )

    print(
        "Levels:",
        data["levels"]
    )

    print(
        "Leaf mappings:",
        len(data["mappings"])
    )


print()
print("=" * 70)
print("CANONICAL ADULT VGH CREATED SUCCESSFULLY")
print("=" * 70)

In [47]:
# Cell 46 — MILESTONE 3
# CREATE CANONICAL ADULT VGH SPECIFICATION — FIXED
# ============================================================

from pathlib import Path
import pandas as pd
import yaml

ROOT = Path("/content/kc-slice")

VGH_DIR = (
    ROOT /
    "hierarchies/adult_arx_standard_vgh"
)

OUTPUT = (
    ROOT /
    "hierarchies/adult_hierarchies.yaml"
)

EXPECTED_ATTRIBUTES = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

# ============================================================
# IMPORTANT:
# The existing hierarchy CSV files are HEADERLESS.
#
# Therefore:
#   header=None
#
# is required.
#
# The first column is always the leaf value.
# Remaining columns are hierarchy/generalization levels.
# ============================================================

LEVELS_BY_ATTRIBUTE = {
    "age": [
        "15-19",
        "10-19",
        "0-19",
        "*",
    ],

    "workclass": [
        "Private.1",
        "*",
    ],

    "education.num": [
        "Basic-1-4",
        "Basic",
        "*",
    ],

    "marital.status": [
        "Widowed.1",
        "*",
    ],

    "occupation": [
        "Management",
        "*",
    ],

    "race": [
        "*",
    ],

    "sex": [
        "*",
    ],

    "native.country": [
        "North-America",
        "*",
    ],
}

EXPECTED_LEAF_COUNTS = {
    "age": 72,
    "workclass": 7,
    "education.num": 16,
    "marital.status": 7,
    "occupation": 14,
    "race": 5,
    "sex": 2,
    "native.country": 41,
}


print("=" * 70)
print("MILESTONE 3 — CANONICAL ADULT VGH")
print("=" * 70)


# ============================================================
# Verify source directory
# ============================================================

assert VGH_DIR.exists(), (
    f"VGH directory not found: {VGH_DIR}"
)

print()
print(
    "VGH directory exists : PASS"
)


# ============================================================
# Build canonical hierarchy representation
# ============================================================

hierarchies = {}

for attribute in EXPECTED_ATTRIBUTES:

    path = (
        VGH_DIR /
        f"{attribute}.csv"
    )

    assert path.exists(), (
        f"Missing hierarchy file: {path}"
    )

    # --------------------------------------------------------
    # CRITICAL FIX:
    # These files do NOT contain a header row.
    # --------------------------------------------------------

    table = pd.read_csv(
        path,
        header=None,
        dtype=str,
    )

    table = table.dropna(
        how="all"
    )

    table = table.reset_index(
        drop=True
    )

    expected_leaf_count = (
        EXPECTED_LEAF_COUNTS[
            attribute
        ]
    )

    expected_levels = (
        LEVELS_BY_ATTRIBUTE[
            attribute
        ]
    )

    expected_columns = (
        1 +
        len(expected_levels)
    )

    # --------------------------------------------------------
    # Validate dimensions.
    # --------------------------------------------------------

    assert len(table) == expected_leaf_count, (
        f"{attribute}: expected "
        f"{expected_leaf_count} rows, got "
        f"{len(table)}"
    )

    assert table.shape[1] == expected_columns, (
        f"{attribute}: expected "
        f"{expected_columns} columns, got "
        f"{table.shape[1]}"
    )

    # --------------------------------------------------------
    # Convert positional columns into the canonical hierarchy.
    # --------------------------------------------------------

    mappings = []

    for _, row in table.iterrows():

        leaf = str(
            row.iloc[0]
        )

        generalization = {}

        for level_index, level_name in enumerate(
            expected_levels,
            start=1
        ):

            value = row.iloc[
                level_index
            ]

            if pd.isna(value):

                value = None

            else:

                value = str(value)

            generalization[
                level_name
            ] = value

        mappings.append({
            "value": leaf,
            "generalization": generalization,
        })

    hierarchies[attribute] = {
        "levels": expected_levels,
        "mappings": mappings,
    }


# ============================================================
# Add project metadata
# ============================================================

specification = {

    "dataset":
        "Adult Census Income",

    "records_after_cleaning":
        30162,

    "project_parameters": {

        "k":
            10,

        "sensitive_attribute":
            "income",

        "qids":
            EXPECTED_ATTRIBUTES,
    },

    "hierarchy_source":
        "Operational VGH constructed for this project "
        "and used for the Adult baseline.",

    "hierarchy_notes":
        "Categorical parent mappings are explicit project "
        "operationalizations. The source material specifies "
        "hierarchy structure/heights but does not provide "
        "every categorical parent mapping.",

    "attributes":
        hierarchies,
}


# ============================================================
# Write canonical YAML
# ============================================================

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        specification,
        f,
        sort_keys=False,
        allow_unicode=True,
        default_flow_style=False,
    )


# ============================================================
# Reload YAML
# ============================================================

with open(
    OUTPUT,
    "r",
    encoding="utf-8"
) as f:

    loaded = yaml.safe_load(f)


# ============================================================
# Top-level verification
# ============================================================

assert (
    loaded["dataset"]
    ==
    "Adult Census Income"
)

assert (
    loaded["records_after_cleaning"]
    ==
    30162
)

assert (
    loaded["project_parameters"]["k"]
    ==
    10
)

assert (
    loaded["project_parameters"][
        "sensitive_attribute"
    ]
    ==
    "income"
)

assert (
    loaded["project_parameters"]["qids"]
    ==
    EXPECTED_ATTRIBUTES
)


# ============================================================
# Verify EVERY hierarchy retained ALL leaf mappings
# ============================================================

print()
print("-" * 70)
print("HIERARCHY VERIFICATION")
print("-" * 70)

for attribute in EXPECTED_ATTRIBUTES:

    data = loaded[
        "attributes"
    ][attribute]

    expected_count = (
        EXPECTED_LEAF_COUNTS[
            attribute
        ]
    )

    actual_count = len(
        data["mappings"]
    )

    assert (
        actual_count
        ==
        expected_count
    )

    print(
        f"{attribute:<20}: "
        f"{actual_count:>2} leaf mappings : PASS"
    )


# ============================================================
# Print hierarchy-level information
# ============================================================

print()
print("-" * 70)
print("CANONICAL HIERARCHY SUMMARY")
print("-" * 70)

for attribute in EXPECTED_ATTRIBUTES:

    data = loaded[
        "attributes"
    ][attribute]

    print()
    print(
        f"[{attribute}]"
    )

    print(
        f"  Leaf mappings : "
        f"{len(data['mappings'])}"
    )

    print(
        f"  Levels        : "
        f"{data['levels']}"
    )


# ============================================================
# Spot-check important mappings
# ============================================================

print()
print("-" * 70)
print("SPOT CHECKS")
print("-" * 70)


# ------------------------------------------------------------
# Age
# ------------------------------------------------------------

age_mappings = {
    item["value"]: item["generalization"]
    for item in loaded[
        "attributes"
    ]["age"]["mappings"]
}

assert age_mappings["17"]["15-19"] == "15-19"
assert age_mappings["17"]["10-19"] == "10-19"
assert age_mappings["17"]["0-19"] == "0-19"
assert age_mappings["17"]["*"] == "*"

print(
    "Age hierarchy mapping     : PASS"
)


# ------------------------------------------------------------
# Workclass
# ------------------------------------------------------------

workclass_mappings = {
    item["value"]: item["generalization"]
    for item in loaded[
        "attributes"
    ]["workclass"]["mappings"]
}

assert (
    workclass_mappings["Private"]["Private.1"]
    ==
    "Private"
)

assert (
    workclass_mappings["State-gov"]["Private.1"]
    ==
    "Government"
)

print(
    "Workclass hierarchy       : PASS"
)


# ------------------------------------------------------------
# Education
# ------------------------------------------------------------

education_mappings = {
    item["value"]: item["generalization"]
    for item in loaded[
        "attributes"
    ]["education.num"]["mappings"]
}

assert (
    education_mappings["1"]["Basic-1-4"]
    ==
    "Basic-1-4"
)

assert (
    education_mappings["13"]["Basic-1-4"]
    ==
    "Post-secondary"
)

print(
    "Education hierarchy       : PASS"
)


# ------------------------------------------------------------
# Occupation
# ------------------------------------------------------------

occupation_mappings = {
    item["value"]: item["generalization"]
    for item in loaded[
        "attributes"
    ]["occupation"]["mappings"]
}

assert (
    occupation_mappings["Exec-managerial"]["Management"]
    ==
    "Management"
)

assert (
    occupation_mappings["Machine-op-inspct"]["Management"]
    ==
    "Skilled-trades"
)

print(
    "Occupation hierarchy      : PASS"
)


# ------------------------------------------------------------
# Native country
# ------------------------------------------------------------

country_mappings = {
    item["value"]: item["generalization"]
    for item in loaded[
        "attributes"
    ]["native.country"]["mappings"]
}

assert (
    country_mappings["United-States"]["North-America"]
    ==
    "North-America"
)

assert (
    country_mappings["South"]["North-America"]
    ==
    "Asia"
)

print(
    "Native-country hierarchy  : PASS"
)


# ============================================================
# Save audit
# ============================================================

AUDIT_OUTPUT = (
    ROOT /
    "results/adult_canonical_vgh_verified.txt"
)

AUDIT_OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    AUDIT_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MILESTONE 3 — CANONICAL ADULT VGH VERIFIED\n"
    )

    f.write("=" * 65 + "\n\n")

    f.write(
        f"Dataset: Adult Census Income\n"
    )

    f.write(
        f"Records after cleaning: 30162\n"
    )

    f.write(
        f"k: 10\n"
    )

    f.write(
        f"Sensitive attribute: income\n\n"
    )

    for attribute in EXPECTED_ATTRIBUTES:

        data = loaded[
            "attributes"
        ][attribute]

        f.write(
            f"{attribute}:\n"
        )

        f.write(
            f"  Leaf mappings: "
            f"{len(data['mappings'])}\n"
        )

        f.write(
            f"  Levels: "
            f"{data['levels']}\n\n"
        )

    f.write(
        "All source hierarchy CSVs were read as "
        "headerless files (header=None).\n"
    )

    f.write(
        "All expected leaf mappings were preserved.\n"
    )


# ============================================================
# FINAL
# ============================================================

print()
print(
    f"Canonical YAML: {OUTPUT}"
)

print(
    f"Verification log: {AUDIT_OUTPUT}"
)

print()
print("=" * 70)
print("CANONICAL ADULT VGH VERIFIED SUCCESSFULLY")
print("=" * 70)

In [48]:
# Cell 47 — MILESTONE 3
# MONDRIAN + SHARED ADULT VGH RELEASE PROJECTION
#
# Purpose:
#   Re-run the strict Mondrian partitioning and express the
#   resulting partitions using the SAME hand-built VGH used
#   by ARX.
#
# Important methodological distinction:
#   The upstream Mondrian implementation does not directly
#   consume an external VGH during partition selection.
#
#   Therefore:
#       1. Mondrian performs the partitioning.
#       2. Each final partition is projected onto the shared
#          Adult VGH.
#       3. The resulting released QID values are used for
#          the final DM/class-size evaluation.
#
# This is explicitly documented as a project operationalization.
# ============================================================

from pathlib import Path
from collections import Counter
import copy
import sys

import pandas as pd


# ============================================================
# Paths
# ============================================================

ROOT = Path("/content/kc-slice")

BASELINE_PATH = (
    ROOT /
    "data/processed/"
    "adult_standard_baseline_k10_1sa.csv"
)

VGH_DIR = (
    ROOT /
    "hierarchies/adult_arx_standard_vgh"
)

RESULTS_DIR = (
    ROOT /
    "results"
)

OUTPUT_DIR = (
    RESULTS_DIR /
    "milestone3_adult_1sa_comparison"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MONDRIAN_PATH = (
    ROOT /
    "methods/mondrian/upstream"
)

if str(MONDRIAN_PATH) not in sys.path:

    sys.path.insert(
        0,
        str(MONDRIAN_PATH)
    )


# ============================================================
# Import Mondrian module itself
#
# We need access to mondrian.RESULT so that we can inspect
# the final partitions before producing the VGH-projected
# release.
# ============================================================

import mondrian as mondrian_module


# ============================================================
# Configuration
# ============================================================

K = 10

QID_COLUMNS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

TARGET = "income"

SOURCE_ID = "_source_row"

QI_NUM = len(QID_COLUMNS)


print("=" * 70)
print("MILESTONE 3 — MONDRIAN + SHARED ADULT VGH")
print("=" * 70)

print()
print(f"k          : {K}")
print(f"QID count  : {QI_NUM}")
print(
    f"Sensitive  : {TARGET}"
)

print()
print(
    "Operational model:"
)

print(
    "  Mondrian partitioning"
)

print(
    "  + shared Adult VGH release projection"
)


# ============================================================
# Load baseline
# ============================================================

df = pd.read_csv(
    BASELINE_PATH
)

assert len(df) == 30162

for column in QID_COLUMNS + [TARGET]:

    assert column in df.columns


df = df.reset_index(
    drop=True
)

df[SOURCE_ID] = df.index


print()
print(
    "Baseline loading : PASS"
)


# ============================================================
# Build categorical mappings
#
# We reproduce the same first-appearance category encoding
# previously used for the upstream Mondrian implementation.
#
# This affects partition selection only.
# The FINAL released values will be generated from the
# shared VGH below.
# ============================================================

categorical_qids = [
    column
    for column in QID_COLUMNS
    if column not in {
        "age",
        "education.num",
    }
]

category_maps = {}

for column in categorical_qids:

    order = list(
        dict.fromkeys(
            df[column]
            .astype(str)
            .tolist()
        )
    )

    category_maps[column] = {
        value: index
        for index, value
        in enumerate(order)
    }


# ============================================================
# Create Mondrian records
#
# [8 QIDs] + income + source_row
# ============================================================

data = []

for _, row in df.iterrows():

    record = []

    for column in QID_COLUMNS:

        if column in category_maps:

            record.append(
                category_maps[column][
                    str(row[column])
                ]
            )

        else:

            record.append(
                int(row[column])
            )

    record.append(
        str(row[TARGET])
    )

    record.append(
        int(row[SOURCE_ID])
    )

    data.append(record)


assert len(data) == 30162

print(
    "Mondrian input construction : PASS"
)


# ============================================================
# Load the shared VGHs
#
# Existing hierarchy CSV files are HEADERLESS:
#
#   column 0 = leaf value
#   column 1... = generalization levels
#
# We intentionally read them with header=None.
# ============================================================

vgh_tables = {}

for attribute in QID_COLUMNS:

    path = (
        VGH_DIR /
        f"{attribute}.csv"
    )

    assert path.exists(), (
        f"Missing VGH file: {path}"
    )

    table = pd.read_csv(
        path,
        header=None,
        dtype=str
    )

    table = table.dropna(
        how="all"
    ).reset_index(
        drop=True
    )

    assert len(table) > 0

    vgh_tables[attribute] = table


print(
    "Shared VGH loading          : PASS"
)


# ============================================================
# VGH generalization function
#
# For a set of leaf values:
#
#   level 0:
#       exact value
#
#   level 1:
#       common parent
#
#   level 2:
#       broader parent
#
#   ...
#
#   first level where ALL leaf values map to the same node
#   is selected.
#
# If no common node exists before the root, "*" is returned.
# ============================================================

def vgh_generalize(
    attribute,
    values
):

    table = vgh_tables[
        attribute
    ]

    # Dictionary:
    # leaf -> hierarchy path
    leaf_paths = {}

    for _, row in table.iterrows():

        leaf = str(
            row.iloc[0]
        )

        path = [
            None
            if pd.isna(value)
            else str(value)
            for value in row.iloc[1:]
        ]

        leaf_paths[leaf] = path


    unique_values = list(
        dict.fromkeys(
            str(value)
            for value in values
        )
    )


    # --------------------------------------------------------
    # Every value must exist in the VGH.
    # --------------------------------------------------------

    for value in unique_values:

        assert value in leaf_paths, (
            f"VGH missing leaf '{value}' "
            f"for attribute '{attribute}'"
        )


    # --------------------------------------------------------
    # Exact-value case.
    # --------------------------------------------------------

    if len(unique_values) == 1:

        return unique_values[0]


    # --------------------------------------------------------
    # Find the first common hierarchy node.
    # --------------------------------------------------------

    max_levels = max(
        len(
            leaf_paths[value]
        )
        for value in unique_values
    )

    for level in range(
        max_levels
    ):

        level_values = []

        for value in unique_values:

            path = leaf_paths[
                value
            ]

            if level >= len(path):

                level_values.append(
                    None
                )

            else:

                level_values.append(
                    path[level]
                )

        # Ignore missing hierarchy entries.
        if any(
            value is None
            for value in level_values
        ):

            continue


        if len(
            set(level_values)
        ) == 1:

            return level_values[0]


    # --------------------------------------------------------
    # Root.
    # --------------------------------------------------------

    return "*"


# ============================================================
# Run strict Mondrian
# ============================================================

print()
print(
    "Running strict Mondrian..."
)

result, eval_result = (
    mondrian_module.mondrian(
        copy.deepcopy(data),
        K,
        False,
        QI_num=QI_NUM
    )
)

ncp_raw = float(
    eval_result[0]
)

runtime = float(
    eval_result[1]
)


# ============================================================
# Access final Mondrian partitions
# ============================================================

partitions = (
    mondrian_module.RESULT
)

assert len(partitions) > 0

partition_record_count = sum(
    len(partition.member)
    for partition in partitions
)

assert (
    partition_record_count
    ==
    len(df)
)

print(
    "Mondrian partition accounting : PASS"
)

print(
    f"Raw Mondrian partitions       : "
    f"{len(partitions):,}"
)


# ============================================================
# Create VGH-projected release
# ============================================================

released_records = []

partition_metadata = []

for partition_id, partition in enumerate(
    partitions,
    start=1
):

    members = partition.member

    # Source rows belonging to this partition.
    source_rows = [
        int(record[-1])
        for record in members
    ]

    assert len(source_rows) == len(
        set(source_rows)
    )


    # --------------------------------------------------------
    # Determine the VGH generalized value for each QID.
    # --------------------------------------------------------

    generalized_values = {}

    source_subset = df.loc[
        source_rows
    ]

    for qid in QID_COLUMNS:

        generalized_values[qid] = (
            vgh_generalize(
                qid,
                source_subset[qid]
                .astype(str)
                .tolist()
            )
        )


    # --------------------------------------------------------
    # Save partition metadata.
    # --------------------------------------------------------

    partition_metadata.append({
        "partition_id":
            partition_id,

        "records":
            len(source_rows),

        **{
            qid:
                generalized_values[qid]
            for qid in QID_COLUMNS
        },
    })


    # --------------------------------------------------------
    # Apply generalized VGH values to every record in the
    # partition.
    # --------------------------------------------------------

    for record in members:

        source_row = int(
            record[-1]
        )

        income = str(
            record[QI_NUM]
        )

        released_records.append({

            qid:
                generalized_values[qid]

            for qid in QID_COLUMNS
        })

        released_records[-1][
            TARGET
        ] = income

        released_records[-1][
            SOURCE_ID
        ] = source_row


# ============================================================
# Convert to DataFrame
# ============================================================

released = pd.DataFrame(
    released_records,
    columns=QID_COLUMNS + [
        TARGET,
        SOURCE_ID,
    ]
)


# ============================================================
# Core accounting
# ============================================================

assert len(released) == 30162

assert (
    released[
        SOURCE_ID
    ].nunique()
    ==
    30162
)

assert (
    set(
        released[SOURCE_ID]
    )
    ==
    set(
        range(30162)
    )
)

print(
    "VGH release row accounting  : PASS"
)


# ============================================================
# Verify target preservation
# ============================================================

original_income = (
    df[TARGET]
    .astype(str)
)

released_income = (
    released
    .sort_values(SOURCE_ID)
    [TARGET]
    .astype(str)
)

assert (
    original_income.reset_index(
        drop=True
    )
    ==
    released_income.reset_index(
        drop=True
    )
).all()

print(
    "Income-target preservation   : PASS"
)


# ============================================================
# Verify every released value is actually represented
# in the supplied VGH.
# ============================================================

print()
print(
    "Checking released values against VGH..."
)

for qid in QID_COLUMNS:

    table = vgh_tables[qid]

    allowed = set(
        table.iloc[:, 0]
        .astype(str)
    )

    for column in table.columns[1:]:

        allowed.update(
            table[column]
            .dropna()
            .astype(str)
        )

    allowed.add("*")

    observed = set(
        released[qid]
        .astype(str)
    )

    invalid = (
        observed
        -
        allowed
    )

    assert not invalid, (
        f"{qid}: invalid VGH values: "
        f"{invalid}"
    )

    print(
        f"  {qid:<20}: PASS "
        f"({len(observed)} released values)"
    )


# ============================================================
# Calculate equivalence classes AFTER VGH projection
# ============================================================

equivalence_classes = Counter(
    tuple(
        row[qid]
        for qid in QID_COLUMNS
    )
    for _, row in released.iterrows()
)

class_sizes = list(
    equivalence_classes.values()
)

assert (
    sum(class_sizes)
    ==
    len(released)
)


# ============================================================
# k-anonymity verification
# ============================================================

minimum_class_size = min(
    class_sizes
)

maximum_class_size = max(
    class_sizes
)

mean_class_size = (
    sum(class_sizes)
    /
    len(class_sizes)
)

discernibility = sum(
    size ** 2
    for size in class_sizes
)

assert (
    minimum_class_size
    >=
    K
)


print()
print("=" * 70)
print("VGH-PROJECTED MONDRIAN RESULTS")
print("=" * 70)

print(
    f"Input records          : "
    f"{len(df):,}"
)

print(
    f"Output records         : "
    f"{len(released):,}"
)

print(
    f"Raw Mondrian NCP (%)   : "
    f"{ncp_raw:.4f}"
)

print(
    f"Runtime (seconds)      : "
    f"{runtime:.4f}"
)

print(
    f"Raw partitions         : "
    f"{len(partitions):,}"
)

print(
    f"Released equivalence classes : "
    f"{len(class_sizes):,}"
)

print(
    f"Minimum class size     : "
    f"{minimum_class_size:,}"
)

print(
    f"Maximum class size     : "
    f"{maximum_class_size:,}"
)

print(
    f"Mean class size        : "
    f"{mean_class_size:.2f}"
)

print(
    f"Released VGH DM        : "
    f"{discernibility:,}"
)

print()
print(
    "k=10 anonymity         : PASS"
)

print(
    "Record accounting      : PASS"
)


# ============================================================
# Save final row-level output
# ============================================================

ROW_OUTPUT = (
    OUTPUT_DIR /
    "adult_standard_mondrian_vgh_k10_1sa.csv"
)

released.to_csv(
    ROW_OUTPUT,
    index=False
)


# ============================================================
# Save partition audit
# ============================================================

PARTITION_OUTPUT = (
    OUTPUT_DIR /
    "adult_standard_mondrian_vgh_partitions.csv"
)

partition_df = pd.DataFrame(
    partition_metadata
)

partition_df.to_csv(
    PARTITION_OUTPUT,
    index=False
)


# ============================================================
# Save summary
# ============================================================

SUMMARY_OUTPUT = (
    OUTPUT_DIR /
    "adult_standard_mondrian_vgh_baseline.txt"
)

summary = f"""
MILESTONE 3 — MONDRIAN ADULT BASELINE WITH SHARED VGH

Dataset: Adult Census Income
Records: {len(df)}

QIDs:
{", ".join(QID_COLUMNS)}

Sensitive attribute:
{TARGET}

k:
{K}

Model:
Strict upstream qiyuangong/Mondrian

VGH:
adult_arx_standard_vgh

Important methodological note:
The upstream Mondrian implementation does not directly load
an external VGH during partition selection. Therefore Mondrian
performs the partitioning and the resulting partitions are
projected onto the same hand-built VGH used by ARX for the
released QID representation.

Raw Mondrian NCP (%): {ncp_raw:.4f}
Runtime (seconds): {runtime:.4f}

Raw Mondrian partitions: {len(partitions)}
Released equivalence classes: {len(class_sizes)}

Minimum released class size: {minimum_class_size}
Maximum released class size: {maximum_class_size}
Mean released class size: {mean_class_size:.2f}

Released VGH Discernibility Metric: {discernibility}

Record accounting: PASS
Income preservation: PASS
VGH-value verification: PASS
k-anonymity verification: PASS

Row-level output:
{ROW_OUTPUT}
""".strip() + "\n"

SUMMARY_OUTPUT.write_text(
    summary,
    encoding="utf-8"
)


# ============================================================
# Final
# ============================================================

print()
print(
    f"Row-level output : {ROW_OUTPUT}"
)

print(
    f"Partition audit  : {PARTITION_OUTPUT}"
)

print(
    f"Summary          : {SUMMARY_OUTPUT}"
)

print()
print("=" * 70)
print("MONDRIAN SHARED-VGH BASELINE COMPLETE")
print("=" * 70)

In [49]:
# Cell 48 — MILESTONE 3
# FINAL BASELINE RESULTS TABLE + REPRODUCTION NOTE
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path("/content/kc-slice")

RESULTS_DIR = ROOT / "results"

COMPARISON_DIR = (
    RESULTS_DIR /
    "milestone3_adult_1sa_comparison"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Paths
# ============================================================

MONDRIAN_PATH = (
    COMPARISON_DIR /
    "adult_standard_mondrian_vgh_k10_1sa.csv"
)

ARX_PATH = (
    RESULTS_DIR /
    "arx_standard_adult_vgh" /
    "adult_standard_arx_vgh_k10.csv"
)

KC_PATH = (
    ROOT /
    "results/kc_slice_adult_1sa" /
    "adult_1sa_evaluation_dataset.csv"
)

BASELINE_PATH = (
    ROOT /
    "data/processed" /
    "adult_standard_baseline_k10_1sa.csv"
)


# ============================================================
# Experiment configuration
# ============================================================

K = 10

SENSITIVE_ATTRIBUTE = "income"

QIDS = [
    "age",
    "workclass",
    "education.num",
    "marital.status",
    "occupation",
    "race",
    "sex",
    "native.country",
]

KC_SLICE_COLUMNS = [
    "slice_1",
    "slice_2",
    "slice_3",
    "slice_4",
    "slice_5",
]


print("=" * 70)
print("MILESTONE 3 — FINAL BASELINE RESULTS")
print("=" * 70)


# ============================================================
# Load final artifacts
# ============================================================

baseline = pd.read_csv(
    BASELINE_PATH
)

mondrian = pd.read_csv(
    MONDRIAN_PATH
)

arx = pd.read_csv(
    ARX_PATH
)

kc = pd.read_csv(
    KC_PATH
)

assert len(baseline) == 30162
assert len(mondrian) == 30162
assert len(arx) == 30162
assert len(kc) == 30162

print()
print(
    "All baseline artifacts loaded : PASS"
)


# ============================================================
# Helper
# ============================================================

def calculate_dm(frame, columns):
    """
    Standard discernibility metric:

        DM = sum(|E|^2)

    where E is each equivalence class.
    """

    class_sizes = (
        frame
        .groupby(
            columns,
            dropna=False
        )
        .size()
    )

    dm = int(
        (class_sizes ** 2).sum()
    )

    return {
        "dm": dm,
        "equivalence_classes": len(
            class_sizes
        ),
        "minimum_class_size": int(
            class_sizes.min()
        ),
        "maximum_class_size": int(
            class_sizes.max()
        ),
        "mean_class_size": float(
            class_sizes.mean()
        ),
    }


# ============================================================
# 1. MONDRIAN
# ============================================================

print()
print("-" * 70)
print("MONDRIAN")
print("-" * 70)

assert all(
    column in mondrian.columns
    for column in QIDS
)

mondrian_stats = calculate_dm(
    mondrian,
    QIDS
)

assert (
    mondrian_stats["minimum_class_size"]
    >=
    K
)

print(
    f"Equivalence classes : "
    f"{mondrian_stats['equivalence_classes']:,}"
)

print(
    f"Minimum class       : "
    f"{mondrian_stats['minimum_class_size']:,}"
)

print(
    f"Maximum class       : "
    f"{mondrian_stats['maximum_class_size']:,}"
)

print(
    f"Mean class          : "
    f"{mondrian_stats['mean_class_size']:.2f}"
)

print(
    f"DM                  : "
    f"{mondrian_stats['dm']:,}"
)

print(
    "k=10 verification   : PASS"
)


# ============================================================
# 2. ARX
# ============================================================

print()
print("-" * 70)
print("ARX")
print("-" * 70)

assert all(
    column in arx.columns
    for column in QIDS
)

arx_stats = calculate_dm(
    arx,
    QIDS
)

assert (
    arx_stats["minimum_class_size"]
    >=
    K
)

print(
    f"Equivalence classes : "
    f"{arx_stats['equivalence_classes']:,}"
)

print(
    f"Minimum class       : "
    f"{arx_stats['minimum_class_size']:,}"
)

print(
    f"Maximum class       : "
    f"{arx_stats['maximum_class_size']:,}"
)

print(
    f"Mean class          : "
    f"{arx_stats['mean_class_size']:.2f}"
)

print(
    f"DM                  : "
    f"{arx_stats['dm']:,}"
)

print(
    "k=10 verification   : PASS"
)


# ============================================================
# 3. KC-SLICE
#
# IMPORTANT:
#
# KC-Slice does NOT use ordinary k-anonymity equivalence
# classes as its privacy mechanism.
#
# Nevertheless, the guide explicitly requests a DM for the
# three methods.
#
# Therefore we calculate a clearly labeled diagnostic:
#
#   DM_sliced = sum(|E_sliced|^2)
#
# where E_sliced is defined by the complete released
# concatenated QID-slice tuple.
#
# This is NOT a claim that KC-Slice provides k-anonymity.
# ============================================================

print()
print("-" * 70)
print("KC-SLICE")
print("-" * 70)

assert all(
    column in kc.columns
    for column in KC_SLICE_COLUMNS
)

kc_stats = calculate_dm(
    kc,
    KC_SLICE_COLUMNS
)

print(
    f"Sliced-QID classes  : "
    f"{kc_stats['equivalence_classes']:,}"
)

print(
    f"Minimum class       : "
    f"{kc_stats['minimum_class_size']:,}"
)

print(
    f"Maximum class       : "
    f"{kc_stats['maximum_class_size']:,}"
)

print(
    f"Mean class          : "
    f"{kc_stats['mean_class_size']:.2f}"
)

print(
    f"Adapted DM          : "
    f"{kc_stats['dm']:,}"
)

print()
print(
    "KC-Slice privacy model: bucket/HSA threshold + slicing"
)

print(
    "KC-Slice k-anonymity claim: NOT made"
)


# ============================================================
# BUILD FINAL BASELINE TABLE
# ============================================================

baseline_results = pd.DataFrame([

    {
        "method":
            "Mondrian",

        "k":
            K,

        "sensitive_attribute":
            SENSITIVE_ATTRIBUTE,

        "qid_count":
            len(QIDS),

        "discernibility_metric":
            mondrian_stats["dm"],

        "equivalence_classes":
            mondrian_stats["equivalence_classes"],

        "minimum_class_size":
            mondrian_stats["minimum_class_size"],

        "maximum_class_size":
            mondrian_stats["maximum_class_size"],

        "mean_class_size":
            mondrian_stats["mean_class_size"],

        "dm_definition":
            "sum(|E|^2) over generalized QID equivalence classes",

        "privacy_note":
            "Strict k-anonymity",
    },

    {
        "method":
            "ARX",

        "k":
            K,

        "sensitive_attribute":
            SENSITIVE_ATTRIBUTE,

        "qid_count":
            len(QIDS),

        "discernibility_metric":
            arx_stats["dm"],

        "equivalence_classes":
            arx_stats["equivalence_classes"],

        "minimum_class_size":
            arx_stats["minimum_class_size"],

        "maximum_class_size":
            arx_stats["maximum_class_size"],

        "mean_class_size":
            arx_stats["mean_class_size"],

        "dm_definition":
            "sum(|E|^2) over generalized QID equivalence classes",

        "privacy_note":
            "k-anonymity with explicit Adult VGH",
    },

    {
        "method":
            "KC-Slice",

        "k":
            K,

        "sensitive_attribute":
            SENSITIVE_ATTRIBUTE,

        "qid_count":
            len(KC_SLICE_COLUMNS),

        "discernibility_metric":
            kc_stats["dm"],

        "equivalence_classes":
            kc_stats["equivalence_classes"],

        "minimum_class_size":
            kc_stats["minimum_class_size"],

        "maximum_class_size":
            kc_stats["maximum_class_size"],

        "mean_class_size":
            kc_stats["mean_class_size"],

        "dm_definition":
            "Adapted DM over complete released sliced-QID tuples",

        "privacy_note":
            "KC-Slice bucket/HSA privacy model; not k-anonymity",
    },
])


# ============================================================
# Save required guide-style table
# ============================================================

BASELINE_OUTPUT = (
    RESULTS_DIR /
    "milestone3_baseline.csv"
)

baseline_results.to_csv(
    BASELINE_OUTPUT,
    index=False
)


print()
print("=" * 70)
print("FINAL BASELINE RESULTS TABLE")
print("=" * 70)

print(
    baseline_results[
        [
            "method",
            "k",
            "sensitive_attribute",
            "discernibility_metric",
            "equivalence_classes",
            "minimum_class_size",
            "maximum_class_size",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# Published-reference/reproduction note
# ============================================================

NOTE_OUTPUT = (
    RESULTS_DIR /
    "milestone3_reproduction_note.txt"
)

note = f"""
MILESTONE 3 — REPRODUCTION NOTE
================================

Dataset
-------
Adult Census Income

Raw records: 32,561
Records after removing rows containing '?': 30,162

Quasi-identifiers
-----------------
{", ".join(QIDS)}

Sensitive attribute
-------------------
{SENSITIVE_ATTRIBUTE}

Privacy parameter
-----------------
k = {K}


Hierarchy specification
-----------------------
A canonical Adult hierarchy specification was created at:

  hierarchies/adult_hierarchies.yaml

The hierarchy contains explicit mappings for:
  age
  workclass
  education.num
  marital.status
  occupation
  race
  sex
  native.country

Mondrian
--------
Strict qiyuangong/Mondrian was used for partitioning.

The final released QID values were projected onto the same
hand-built Adult VGH used for the ARX baseline.

Equivalence classes after VGH projection:
  {mondrian_stats["equivalence_classes"]}

Minimum class size:
  {mondrian_stats["minimum_class_size"]}

Maximum class size:
  {mondrian_stats["maximum_class_size"]}

Discernibility Metric:
  {mondrian_stats["dm"]:,}


ARX
---
ARX 3.9.2 was used with the explicit Adult VGH.

Equivalence classes:
  {arx_stats["equivalence_classes"]}

Minimum class size:
  {arx_stats["minimum_class_size"]}

Maximum class size:
  {arx_stats["maximum_class_size"]}

Discernibility Metric:
  {arx_stats["dm"]:,}


KC-Slice
--------
KC-Slice was implemented from the source algorithm.

For the 1-SA Adult baseline:
  Sensitive attribute = income
  HSA = <=50K
  C = 5%
  Internal bucket size = 15,081

HSA suppression:
  21,146 records
  70.1081% of the dataset

Because KC-Slice does not use ordinary k-anonymity as its
privacy model, its DM is reported only as an adapted diagnostic
over the complete released sliced-QID tuple.

Adapted sliced-QID DM:
  {kc_stats["dm"]:,}

This number must not be interpreted as proof of k-anonymity.


Discernibility Metric definition
--------------------------------
For Mondrian and ARX:

  DM = sum(|E|^2)

where E denotes a generalized-QID equivalence class.


Published-reference check
-------------------------
LeFevre, DeWitt and Ramakrishnan, "Mondrian Multidimensional
K-Anonymity" includes an Adult-dataset comparison using the
Discernibility Penalty in Figure 10.

The source presents the Adult comparison graphically rather than
as a table of exact numerical values in the accompanying text.
Therefore an exact numerical equality/ballpark claim is not made
here without extracting a precise point from the original figure.

The published comparison also uses its own experimental
configuration, so differences in QID selection, hierarchy
definition, dataset preprocessing, and implementation details
must be considered when comparing numerical results.


Method gap
----------
The Mondrian and ARX DM values can differ substantially even
with k=10 because their anonymization/search procedures and
resulting equivalence-class structures are not identical.

In this experiment the hierarchy specification is controlled,
but the upstream Mondrian implementation does not directly
consume the external VGH during partition selection. Its final
partitions are therefore expressed through the shared VGH before
the released DM is calculated.

KC-Slice is not directly comparable to Mondrian/ARX under the
same DM interpretation because its privacy mechanism is based on
bucket formation, HSA frequency thresholds, suppression, SIDs,
and slicing rather than ordinary k-anonymity.


Final baseline table
--------------------
{baseline_results[
    [
        "method",
        "k",
        "sensitive_attribute",
        "discernibility_metric",
    ]
].to_string(index=False)}


Milestone status
----------------
Adult 1-SA baseline reproduction and the requested
three-method baseline results table have been generated.
""".strip() + "\n"


NOTE_OUTPUT.write_text(
    note,
    encoding="utf-8"
)


# ============================================================
# Verification
# ============================================================

assert (
    set(
        baseline_results["method"]
    )
    ==
    {
        "Mondrian",
        "ARX",
        "KC-Slice",
    }
)

assert len(
    baseline_results
) == 3

assert all(
    baseline_results[
        "discernibility_metric"
    ]
    > 0
)

assert BASELINE_OUTPUT.exists()
assert NOTE_OUTPUT.exists()


print()
print("=" * 70)
print("SUBMISSION ARTIFACTS CREATED")
print("=" * 70)

print(
    f"Baseline table     : {BASELINE_OUTPUT}"
)

print(
    f"Reproduction note  : {NOTE_OUTPUT}"
)

print()
print("=" * 70)
print("MILESTONE 3 BASELINE RESULTS COMPLETE")
print("=" * 70)

In [50]:
# MILESTONE 3 — SYNC SUBMISSION ARTIFACTS TO GITHUB
# ==================================================

get_ipython().run_line_magic('cd', '/content/kc-slice')

get_ipython().system('git status')

get_ipython().system('git add hierarchies/adult_hierarchies.yaml')
get_ipython().system('git add results/milestone3_baseline.csv')
get_ipython().system('git add results/milestone3_reproduction_note.txt')

get_ipython().system('git status')

get_ipython().system('git commit -m "Complete Milestone 3 baseline reproduction"')

get_ipython().system('git push')

In [51]:
# Clone the actual GitHub repository
get_ipython().run_line_magic('cd', '/content')

get_ipython().system('git clone https://github.com/Bhuvan9407/PPDP-Project.git')

# Copy the verified Milestone 3 submission files
get_ipython().system('mkdir -p /content/PPDP-Project/hierarchies')
get_ipython().system('mkdir -p /content/PPDP-Project/results')

get_ipython().system('cp /content/kc-slice/hierarchies/adult_hierarchies.yaml     /content/PPDP-Project/hierarchies/')

get_ipython().system('cp /content/kc-slice/results/milestone3_baseline.csv     /content/PPDP-Project/results/')

get_ipython().system('cp /content/kc-slice/results/milestone3_reproduction_note.txt     /content/PPDP-Project/results/')

# Check what will be committed
get_ipython().run_line_magic('cd', '/content/PPDP-Project')
get_ipython().system('git status')

In [52]:
get_ipython().run_line_magic('cd', '/content/PPDP-Project')

get_ipython().system('git add hierarchies/adult_hierarchies.yaml           results/milestone3_baseline.csv           results/milestone3_reproduction_note.txt')

get_ipython().system('git commit -m "Add Milestone 3 baseline artifacts"')

get_ipython().system('git push origin main')

In [53]:
get_ipython().run_line_magic('cd', '/content/PPDP-Project')

get_ipython().system('git config user.name "Bhuvan9407"')
get_ipython().system('git config user.email "bhuvanvenkatganesh@gmail.com"')

get_ipython().system('git config --get user.name')
get_ipython().system('git config --get user.email')

In [54]:
from getpass import getpass
import os

token = getpass("Enter your GitHub Personal Access Token: ")
os.environ["GITHUB_TOKEN"] = token

print("Token loaded for this session.")

In [55]:
get_ipython().run_line_magic('cd', '/content/PPDP-Project')

get_ipython().system('git add hierarchies/adult_hierarchies.yaml           results/milestone3_baseline.csv           results/milestone3_reproduction_note.txt')

get_ipython().system('git commit -m "Add Milestone 3 baseline artifacts"')

import os
import subprocess

remote = (
    "https://Bhuvan9407:"
    + os.environ["GITHUB_TOKEN"]
    + "@github.com/Bhuvan9407/PPDP-Project.git"
)

subprocess.run(
    ["git", "push", remote, "main"],
    check=True
)

print("✅ Milestone 3 artifacts pushed to GitHub")